# 3\.3\.5 Compare Candidate Anomaly Frameworks

This notebook consolidates the retained DBSCAN, Isolation Forest, and Gaussian Mixture Model outputs into one shared comparison package built on the 3\.3\.1 anomaly backbone\. Its job is synthesis rather than recalibration: align the selected framework surfaces, compare where they agree or diverge, add directionality, and prepare a downstream\-ready handoff for later review, policy interpretation, and social\-context joins\.

A key interpretive goal is to isolate stress\-direction anomalies before we start talking about consensus or downstream relevance\. That lens becomes central later in the notebook, but it will still be evaluated alongside framework overlap and shared\-modality driver patterns rather than treated as a standalone shortcut\.

<img src="image-20260917-021909.png" width="" align="" />

In [1]:
# ---------------------------------------------------------------------
# Import libraries for framework synthesis and downstream handoff
# ---------------------------------------------------------------------

from pathlib import Path
import json

import duckdb
import geopandas as gpd
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from shapely import wkb

from project_branding import (
    BRAND_COLORS,
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_MAP_COLORS,
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = BRAND_COLOR_SEQUENCE
px.colors.qualitative.Set2 = BRAND_COLOR_SEQUENCE

# Make plotly_white behave like the project brand template so downstream
# cells that already use template="plotly_white" inherit the same look.
pio.templates["plotly_white"].layout.paper_bgcolor = "white"
pio.templates["plotly_white"].layout.plot_bgcolor = BRAND_COLORS["ice"]
pio.templates["plotly_white"].layout.font = {"color": BRAND_COLORS["dark_teal"]}
pio.templates["plotly_white"].layout.title = {"font": {"color": BRAND_COLORS["dark_teal"]}}
pio.templates["plotly_white"].layout.legend = {
    "bgcolor": "rgba(255,255,255,0.85)",
    "bordercolor": BRAND_COLORS["seafoam"],
    "borderwidth": 1,
    "font": {"color": BRAND_COLORS["dark_teal"]},
}
pio.templates["plotly_white"].layout.xaxis = {
    "gridcolor": "rgba(11, 78, 74, 0.14)",
    "zerolinecolor": "rgba(11, 78, 74, 0.20)",
    "tickfont": {"color": BRAND_COLORS["dark_teal"]},
    "title": {"font": {"color": BRAND_COLORS["dark_teal"]}},
}
pio.templates["plotly_white"].layout.yaxis = {
    "gridcolor": "rgba(11, 78, 74, 0.14)",
    "zerolinecolor": "rgba(11, 78, 74, 0.20)",
    "tickfont": {"color": BRAND_COLORS["dark_teal"]},
    "title": {"font": {"color": BRAND_COLORS["dark_teal"]}},
}

LEGACY_COLOR_MAP = {
    "#4c78a8": BRAND_COLORS["dark_teal"],
    "#e45756": BRAND_COLORS["terracotta"],
    "#72b7b2": BRAND_COLORS["seafoam"],
    "#2e8b57": BRAND_COLORS["seafoam"],
    "gray": BRAND_COLORS["seafoam"],
    "grey": BRAND_COLORS["seafoam"],
}


def remap_legacy_color(value):
    if isinstance(value, str):
        return LEGACY_COLOR_MAP.get(value.strip().lower(), value)
    if isinstance(value, (list, tuple)):
        return [remap_legacy_color(item) for item in value]
    return value


def apply_brand_plotly_layout(
    fig,
    *,
    title=None,
    width=None,
    height=None,
    legend_title=None,
    margin=None,
):
    layout_updates = {
        "template": "plotly_white",
        "paper_bgcolor": "white",
        "plot_bgcolor": BRAND_COLORS["ice"],
        "font": {"color": BRAND_COLORS["dark_teal"]},
        "width": width,
        "height": height,
        "margin": margin or {"l": 70, "r": 40, "t": 80, "b": 70},
    }
    if title is not None:
        layout_updates["title"] = {
            "text": title,
            "x": 0.5,
            "xanchor": "center",
            "y": 0.98,
            "yanchor": "top",
            "font": {"color": BRAND_COLORS["dark_teal"], "size": 20},
        }

    fig.update_layout(**layout_updates)

    fig.update_xaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont={"color": BRAND_COLORS["dark_teal"]},
        title_font={"color": BRAND_COLORS["dark_teal"]},
    )
    fig.update_yaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont={"color": BRAND_COLORS["dark_teal"]},
        title_font={"color": BRAND_COLORS["dark_teal"]},
    )

    if legend_title is not None:
        fig.update_layout(
            legend={
                "title": {"text": legend_title},
                "orientation": "h",
                "yanchor": "top",
                "y": -0.18,
                "xanchor": "center",
                "x": 0.5,
                "bgcolor": "rgba(255,255,255,0.85)",
                "bordercolor": BRAND_COLORS["seafoam"],
                "borderwidth": 1,
                "font": {"color": BRAND_COLORS["dark_teal"]},
            }
        )

    return fig


def remap_legacy_plotly_colors(fig):
    current_title = None
    if getattr(fig.layout, "title", None) is not None:
        current_title = getattr(fig.layout.title, "text", None)

    apply_brand_plotly_layout(fig, title=current_title)

    for trace in fig.data:
        if hasattr(trace, "marker") and trace.marker is not None:
            if hasattr(trace.marker, "color"):
                trace.marker.color = remap_legacy_color(trace.marker.color)
            if hasattr(trace.marker, "line") and trace.marker.line is not None:
                if hasattr(trace.marker.line, "color"):
                    trace.marker.line.color = remap_legacy_color(trace.marker.line.color)

        if hasattr(trace, "line") and trace.line is not None:
            if hasattr(trace.line, "color"):
                trace.line.color = remap_legacy_color(trace.line.color)

        if hasattr(trace, "fillcolor"):
            trace.fillcolor = remap_legacy_color(trace.fillcolor)

        if hasattr(trace, "colorscale"):
            trace.colorscale = remap_legacy_color(trace.colorscale)

        if hasattr(trace, "increasing") and trace.increasing is not None:
            if hasattr(trace.increasing, "marker") and trace.increasing.marker is not None:
                if hasattr(trace.increasing.marker, "color"):
                    trace.increasing.marker.color = remap_legacy_color(trace.increasing.marker.color)
        if hasattr(trace, "decreasing") and trace.decreasing is not None:
            if hasattr(trace.decreasing, "marker") and trace.decreasing.marker is not None:
                if hasattr(trace.decreasing.marker, "color"):
                    trace.decreasing.marker.color = remap_legacy_color(trace.decreasing.marker.color)
        if hasattr(trace, "totals") and trace.totals is not None:
            if hasattr(trace.totals, "marker") and trace.totals.marker is not None:
                if hasattr(trace.totals.marker, "color"):
                    trace.totals.marker.color = remap_legacy_color(trace.totals.marker.color)

    if fig.layout.shapes:
        for shape in fig.layout.shapes:
            if hasattr(shape, "line") and shape.line is not None and hasattr(shape.line, "color"):
                shape.line.color = remap_legacy_color(shape.line.color)
            if hasattr(shape, "fillcolor"):
                shape.fillcolor = remap_legacy_color(shape.fillcolor)

    if fig.layout.annotations:
        for annotation in fig.layout.annotations:
            if hasattr(annotation, "font") and annotation.font is not None and hasattr(annotation.font, "color"):
                annotation.font.color = remap_legacy_color(annotation.font.color)

    return fig


if not hasattr(go.Figure, "_brand_original_show"):
    go.Figure._brand_original_show = go.Figure.show

    def _brand_show(self, *args, **kwargs):
        remap_legacy_plotly_colors(self)
        return go.Figure._brand_original_show(self, *args, **kwargs)

    go.Figure.show = _brand_show

In [2]:
# ---------------------------------------------------------------------
# Define project directories and notebook output locations
# ---------------------------------------------------------------------

PROJECT_ROOT = Path("/datasets/_deepnote_work")
PIPELINE_DATA_DIR = PROJECT_ROOT / "pipeline_data"

FINAL_331_DIR = PIPELINE_DATA_DIR / "3.3.1.final_tables"
FINAL_332_DIR = PIPELINE_DATA_DIR / "3.3.2.final_tables"
FINAL_333_DIR = PIPELINE_DATA_DIR / "3.3.3.final_tables"
FINAL_334_DIR = PIPELINE_DATA_DIR / "3.3.4.final_tables"

INTERMEDIATE_335_DIR = PIPELINE_DATA_DIR / "3.3.5.intermediate_tables"
FINAL_335_DIR = PIPELINE_DATA_DIR / "3.3.5.final_tables"

TAXI_ZONE_SPATIAL_CONTEXT_PATH = (
    PIPELINE_DATA_DIR
    / "1.3.1.final_tables"
    / "nyc_taxi_zones_harmonized.parquet"
)

INTERMEDIATE_335_DIR.mkdir(parents=True, exist_ok=True)
FINAL_335_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# ---------------------------------------------------------------------
# Define inherited framework defaults, canonical grain, and core labels
# ---------------------------------------------------------------------

STUDY_START_DATE = "2023-01-01"
STUDY_END_DATE = "2026-03-31"
CONGESTION_PRICING_START_DATE = "2025-01-05"
MODEL_FEATURE_SET_NAMES = ["shared"]
FEATURE_SET_ORDER = ["shared"]

FRAMEWORK_NAMES = ["dbscan", "isolation_forest", "gmm"]
FRAMEWORK_LABELS = {
    "dbscan": "DBSCAN",
    "isolation_forest": "Isolation Forest",
    "gmm": "GMM",
}
FRAMEWORK_DISPLAY_ORDER = [
    FRAMEWORK_LABELS["dbscan"],
    FRAMEWORK_LABELS["isolation_forest"],
    FRAMEWORK_LABELS["gmm"],
]

MODEL_ID_COLUMN = "comparison_row_id"
ZONE_ID_COLUMN = "taxi_zone_id"
DATE_COLUMN = "date"
DAYPART_COLUMN = "daypart"
METRIC_COLUMN = "metric"
SERIES_ID_COLUMN = "series_id"
TEMPORAL_BUCKET_COLUMN = "temporal_bucket"
DAYPART_ORDER_COLUMN = "daypart_order"
TEMPORAL_BUCKET_ORDER_COLUMN = "temporal_bucket_order"
SUPPORT_STATUS_COLUMN = "support_status"
SCALE_SOURCE_COLUMN = "scale_source"
OBSERVED_VALUE_COLUMN = "observed_value"
EXPECTED_VALUE_COLUMN = "expected_value"
RESIDUAL_VALUE_COLUMN = "residual_value"
RESIDUAL_ZSCORE_COLUMN = "residual_zscore"
SUPPORT_REVIEW_COLUMN = "comparison_group_support_review_flag"
ZONE_COLUMN = "zone"
BOROUGH_COLUMN = "borough"
POLICY_GEOGRAPHY_COLUMN = "cbd_spatial_category"
PRE_POST_CP_COLUMN = "pre_post_cp"

FRAMEWORK_ALIGNMENT_KEY_COLUMNS = [
    ZONE_ID_COLUMN,
    DATE_COLUMN,
    DAYPART_COLUMN,
    METRIC_COLUMN,
]

CANONICAL_GRAIN_LABEL = "Taxi Zone x Date x Daypart x Metric"
COMPARISON_GROUP_LABEL = "metric x daypart"

COMPARISON_SPINE_CONTEXT_COLUMNS = [
    TEMPORAL_BUCKET_COLUMN,
    TEMPORAL_BUCKET_ORDER_COLUMN,
    DAYPART_ORDER_COLUMN,
    SERIES_ID_COLUMN,
    SUPPORT_STATUS_COLUMN,
    SCALE_SOURCE_COLUMN,
]

COMPARISON_SPINE_VALUE_COLUMNS = [
    OBSERVED_VALUE_COLUMN,
    EXPECTED_VALUE_COLUMN,
    RESIDUAL_VALUE_COLUMN,
    RESIDUAL_ZSCORE_COLUMN,
]

OPTIONAL_COMPARISON_COLUMNS = [
    SUPPORT_REVIEW_COLUMN,
    "row_has_any_structural_absence_flag",
    "row_has_any_neutral_imputation_flag",
    "row_has_any_low_support_flag",
    "all_base_demand_zero_flag",
    ZONE_COLUMN,
    BOROUGH_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
]

FRAMEWORK_FLAG_COLUMNS = {
    "dbscan": "dbscan_is_anomaly",
    "isolation_forest": "if_is_anomaly",
    "gmm": "gmm_is_anomaly",
}

FRAMEWORK_SCORE_COLUMNS = {
    "dbscan": "dbscan_score",
    "isolation_forest": "if_score",
    "gmm": "gmm_score",
}

DIRECTIONAL_LABELS = [
    "congestion_oriented",
    "decongestion_oriented",
    "positive_demand_shock",
    "negative_demand_shock",
    "ambiguous",
]

PRIMARY_CONSENSUS_MIN_METHODS = 2
STRICT_CONSENSUS_MIN_METHODS = 3

In [4]:
# ---------------------------------------------------------------------
# Define inherited shared-anomaly and retained-framework asset paths
# ---------------------------------------------------------------------

SHARED_RESIDUAL_PANEL_PATH = FINAL_331_DIR / "temporal_residual_mobility_panel.parquet"
SHARED_RESIDUAL_DIAGNOSTICS_PATH = (
    FINAL_331_DIR / "temporal_residual_mobility_diagnostics.parquet"
)
SHARED_CALIBRATION_PANEL_PATH = FINAL_331_DIR / "daypart_calibration_panel.parquet"
SHARED_CALIBRATION_SAMPLE_PATH = FINAL_331_DIR / "daypart_calibration_sample.parquet"
SHARED_CALIBRATION_GROUP_SUMMARY_PATH = (
    FINAL_331_DIR / "daypart_calibration_group_size_summary.parquet"
)
SHARED_RESIDUAL_MANIFEST_PATH = FINAL_331_DIR / "temporal_residual_mobility_manifest.csv"
SHARED_CALIBRATION_MANIFEST_PATH = FINAL_331_DIR / "daypart_calibration_manifest.csv"

DBSCAN_SELECTED_CONFIGURATION_PATH = (
    FINAL_332_DIR / "dbscan_selected_configuration.csv"
)
DBSCAN_SELECTED_ROW_LEVEL_PATH = (
    FINAL_332_DIR / "dbscan_selected_row_level_output.parquet"
)

IF_SELECTED_CONFIGURATION_PATH = (
    FINAL_333_DIR / "isolation_forest_selected_configuration.csv"
)
IF_SELECTED_ROW_LEVEL_PATH = (
    FINAL_333_DIR / "isolation_forest_selected_row_level_output.parquet"
)

GMM_SELECTED_CONFIGURATION_PATH = (
    FINAL_334_DIR / "gmm_selected_configuration.csv"
)
GMM_SELECTED_ROW_LEVEL_PATH = (
    FINAL_334_DIR / "gmm_selected_row_level_output.parquet"
)
GMM_CALIBRATION_SUMMARY_PATH = (
    FINAL_334_DIR / "gmm_calibration_summary.csv"
)
GMM_CANDIDATE_REVIEW_PATH = (
    FINAL_334_DIR / "gmm_candidate_review.csv"
)
GMM_HANDOFF_MANIFEST_PATH = (
    FINAL_334_DIR / "gmm_handoff_manifest.csv"
)

SELECTED_FRAMEWORK_BRANCH_LABELS = {
    "dbscan": "selected DBSCAN branch: k5_q92_minpts_7",
    "isolation_forest": "selected Isolation Forest branch: if_contam_0_05",
    "gmm": "selected GMM branch: gmm_k3_reg_1em02__tail_0p030",
}

FRAMEWORK_SELECTED_ASSET_SPECS = [
    {
        "framework_name": "dbscan",
        "framework_label": FRAMEWORK_LABELS["dbscan"],
        "selected_configuration_path": DBSCAN_SELECTED_CONFIGURATION_PATH,
        "selected_row_level_path": DBSCAN_SELECTED_ROW_LEVEL_PATH,
    },
    {
        "framework_name": "isolation_forest",
        "framework_label": FRAMEWORK_LABELS["isolation_forest"],
        "selected_configuration_path": IF_SELECTED_CONFIGURATION_PATH,
        "selected_row_level_path": IF_SELECTED_ROW_LEVEL_PATH,
    },
    {
        "framework_name": "gmm",
        "framework_label": FRAMEWORK_LABELS["gmm"],
        "selected_configuration_path": GMM_SELECTED_CONFIGURATION_PATH,
        "selected_row_level_path": GMM_SELECTED_ROW_LEVEL_PATH,
    },
]

In [5]:
# ---------------------------------------------------------------------
# Define 3.3.5 intermediate/final paths and notebook execution toggles
# ---------------------------------------------------------------------

FRAMEWORK_COMPARISON_ROW_LEVEL_PATH = (
    INTERMEDIATE_335_DIR / "framework_comparison_row_level.parquet"
)
FRAMEWORK_COMPARISON_MANIFEST_PATH = (
    INTERMEDIATE_335_DIR / "framework_comparison_manifest.parquet"
)
FRAMEWORK_COMPARISON_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "framework_comparison_summary.parquet"
)
FRAMEWORK_ALIGNMENT_VALIDATION_PATH = (
    INTERMEDIATE_335_DIR / "framework_alignment_validation.parquet"
)
FRAMEWORK_PREVALENCE_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "framework_prevalence_summary.parquet"
)

DIRECTIONAL_RULE_AUDIT_PATH = INTERMEDIATE_335_DIR / "directional_rule_audit.parquet"
DIRECTIONAL_STANDARDIZED_FEATURE_PATH = (
    INTERMEDIATE_335_DIR / "directional_standardized_feature_values.parquet"
)
DIRECTIONAL_STANDARDIZED_ANOMALY_FEATURE_PATH = (
    INTERMEDIATE_335_DIR / "directional_standardized_anomaly_feature_values.parquet"
)
DIRECTIONAL_ROW_LEVEL_PATH = INTERMEDIATE_335_DIR / "directional_row_level.parquet"
RETAINED_EVENT_SPINE_BASE_PATH = (
    INTERMEDIATE_335_DIR / "retained_event_spine_base.parquet"
)
FRAMEWORK_EVENT_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "framework_event_summary.parquet"
)
EVENT_OVERLAP_REGIONS_PATH = (
    INTERMEDIATE_335_DIR / "event_overlap_regions.parquet"
)
POSITIVE_EVENT_OVERLAP_REGIONS_PATH = (
    INTERMEDIATE_335_DIR / "positive_event_overlap_regions.parquet"
)
CANDIDATE_SUPPORT_COMPOSITION_PATH = (
    INTERMEDIATE_335_DIR / "candidate_support_composition.parquet"
)
INCREMENTAL_DIRECTIONAL_QUALITY_PATH = (
    INTERMEDIATE_335_DIR / "incremental_directional_quality.parquet"
)
INCREMENTAL_MODALITY_BREADTH_PATH = (
    INTERMEDIATE_335_DIR / "incremental_modality_breadth.parquet"
)
INCREMENTAL_GEOGRAPHIC_SPREAD_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "incremental_geographic_spread_summary.parquet"
)
INCREMENTAL_GEOGRAPHIC_BOROUGH_PATH = (
    INTERMEDIATE_335_DIR / "incremental_geographic_borough_mix.parquet"
)
CONSENSUS_ROW_LEVEL_PATH = INTERMEDIATE_335_DIR / "consensus_row_level.parquet"
CONSENSUS_SUMMARY_PATH = INTERMEDIATE_335_DIR / "consensus_summary.parquet"
MODALITY_PARTICIPATION_PATH = INTERMEDIATE_335_DIR / "modality_participation.parquet"
HUMAN_REVIEW_PANEL_PATH = (
    INTERMEDIATE_335_DIR / "framework_interpretation_review_panel.parquet"
)
EVENT_DRIVER_EVIDENCE_PATH = (
    INTERMEDIATE_335_DIR / "event_driver_evidence.parquet"
)
SELECTED_POSITIVE_EVENT_SURFACE_2PLUS_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_2plus.parquet"
)
SELECTED_POSITIVE_EVENT_SURFACE_3WAY_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_3way.parquet"
)
POSITIVE_DIRECTION_SURFACE_CANDIDATE_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_candidate_summary.parquet"
)
POSITIVE_DIRECTION_SURFACE_PROFILE_BREAKDOWN_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_profile_breakdown.parquet"
)
POSITIVE_DIRECTION_SURFACE_SUPPORT_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_support_summary.parquet"
)
POSITIVE_DIRECTION_SURFACE_SUPPORT_PATTERN_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_support_pattern_summary.parquet"
)
POSITIVE_DIRECTION_SURFACE_GEOGRAPHY_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_geography_summary.parquet"
)
POSITIVE_DIRECTION_SURFACE_BOROUGH_MIX_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_borough_mix.parquet"
)
POSITIVE_DIRECTION_SURFACE_ZONE_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_zone_summary.parquet"
)
POSITIVE_DIRECTION_SURFACE_DRIVER_BREADTH_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_driver_breadth.parquet"
)
POSITIVE_DIRECTION_SURFACE_MODALITY_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_modality_summary.parquet"
)
POSITIVE_DIRECTION_SURFACE_METRIC_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_event_surface_metric_summary.parquet"
)
SELECTED_POSITIVE_DRIVER_EVIDENCE_2PLUS_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_driver_evidence_2plus.parquet"
)
SELECTED_POSITIVE_DRIVER_EVIDENCE_3WAY_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_driver_evidence_3way.parquet"
)
SELECTED_POSITIVE_METRIC_EVIDENCE_3WAY_PATH = (
    INTERMEDIATE_335_DIR / "selected_positive_metric_evidence_3way.parquet"
)

FULL_UNIVERSE_EXPORT_PATH = (
    FINAL_335_DIR / "framework_comparison_full_row_universe.parquet"
)
RETAINED_WORKING_EXPORT_PATH = FINAL_335_DIR / "retained_anomaly_working_table.parquet"
POSITIVE_DIRECTION_2PLUS_WORKING_EXPORT_PATH = (
    FINAL_335_DIR / "retained_positive_direction_surface_2plus.parquet"
)
POSITIVE_DIRECTION_3WAY_WORKING_EXPORT_PATH = (
    FINAL_335_DIR / "retained_positive_direction_surface_3way.parquet"
)
POSITIVE_DIRECTION_2PLUS_SUMMARY_EXPORT_PATH = (
    FINAL_335_DIR / "retained_positive_direction_surface_2plus_summary.csv"
)
POSITIVE_DIRECTION_3WAY_SUMMARY_EXPORT_PATH = (
    FINAL_335_DIR / "retained_positive_direction_surface_3way_summary.csv"
)
POSITIVE_DIRECTION_2PLUS_DRIVER_EVIDENCE_EXPORT_PATH = (
    FINAL_335_DIR / "retained_positive_direction_surface_2plus_driver_evidence.parquet"
)
POSITIVE_DIRECTION_3WAY_DRIVER_EVIDENCE_EXPORT_PATH = (
    FINAL_335_DIR / "retained_positive_direction_surface_3way_driver_evidence.parquet"
)
POSITIVE_DIRECTION_3WAY_METRIC_EVIDENCE_EXPORT_PATH = (
    FINAL_335_DIR / "retained_positive_direction_surface_3way_metric_evidence.parquet"
)
POSITIVE_DIRECTION_2PLUS_METRIC_EVIDENCE_EXPORT_PATH = (
    FINAL_335_DIR
    / "retained_positive_direction_surface_2plus_metric_evidence.parquet"
)
CONSENSUS_INDICATOR_EXPORT_PATH = (
    FINAL_335_DIR / "consensus_anomaly_indicators.parquet"
)
DIRECTIONAL_LABEL_EXPORT_PATH = FINAL_335_DIR / "directional_anomaly_labels.parquet"
FRAMEWORK_FLAG_EXPORT_PATH = FINAL_335_DIR / "framework_specific_anomaly_flags.parquet"
MODALITY_PARTICIPATION_EXPORT_PATH = (
    FINAL_335_DIR / "modality_participation_indicators.parquet"
)
RETAINED_SURFACE_SELECTION_PATH = (
    FINAL_335_DIR / "retained_surface_selection_rationale.csv"
)
SOCIAL_JOIN_FOCUS_EXPORT_PATH = (
    FINAL_335_DIR / "retained_social_join_focus_table.parquet"
)
DOWNSTREAM_HANDOFF_MANIFEST_PATH = (
    FINAL_335_DIR / "framework_synthesis_handoff_manifest.csv"
)

# Keep the broader 2+ package as the default working alias for backwards compatibility.
POSITIVE_DIRECTION_WORKING_EXPORT_PATH = POSITIVE_DIRECTION_2PLUS_WORKING_EXPORT_PATH

RETAINED_COMPARISON_EXPORT_PATH = RETAINED_WORKING_EXPORT_PATH
ANOMALY_SELECTION_RATIONALE_PATH = RETAINED_SURFACE_SELECTION_PATH

REVIEW_PANEL_ROWS_PER_FAMILY = 5
DIRECTIONAL_SCORE_THRESHOLD = 1.28
MIXED_DIRECTION_MAX_NET_SCORE = 1.0
MIN_DIRECTIONAL_DRIVER_COUNT = 1

REBUILD_335_CONSOLIDATION = False
REBUILD_335_DIRECTIONALITY = False
REBUILD_335_CONSENSUS = False
REBUILD_335_MODALITY = False
REBUILD_335_HUMAN_REVIEW = False
REBUILD_335_EXPORTS = False
WRITE_335_OUTPUTS = True


In [6]:
# ---------------------------------------------------------------------
# Define shared helper functions for retained-framework loading and validation
# ---------------------------------------------------------------------

def sql_path(path_like):
    return str(Path(path_like)).replace("'", "''")


def should_rebuild(flag, output_path):
    return flag or not Path(output_path).exists()


def safe_share(numerator, denominator):
    return numerator / denominator if denominator else np.nan


def status_label(condition):
    return "pass" if bool(condition) else "review"


def get_parquet_columns(parquet_path):
    with duckdb.connect() as con:
        describe_df = con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{sql_path(parquet_path)}')"
        ).fetchdf()
    return describe_df["column_name"].tolist()


def load_parquet_without_duplicate_columns(parquet_path, columns=None):
    df = pd.read_parquet(parquet_path, columns=columns)
    return df.loc[:, ~df.columns.duplicated()].copy()


def assert_unique_key(df, key_columns, object_name):
    duplicate_count = int(df.duplicated(subset=key_columns).sum())
    if duplicate_count != 0:
        raise ValueError(
            f"{object_name} must be unique on {key_columns}; found {duplicate_count} duplicates."
        )


def resolve_required_column(df, candidate_columns, df_name, semantic_name):
    for column_name in candidate_columns:
        if column_name in df.columns:
            return column_name
    raise KeyError(
        f"{df_name} is missing every candidate column in {candidate_columns} "
        f"for {semantic_name}. Available columns: {df.columns.tolist()}"
    )


def resolve_optional_column(df, candidate_columns):
    for column_name in candidate_columns:
        if column_name in df.columns:
            return column_name
    return None


def read_table(path_like, columns=None):
    path_like = Path(path_like)
    if path_like.suffix.lower() == ".csv":
        return pd.read_csv(path_like)
    if path_like.suffix.lower() == ".parquet":
        return load_parquet_without_duplicate_columns(path_like, columns=columns)
    raise ValueError(f"Unsupported table type: {path_like}")


def resolve_selected_candidate_id(configuration_df, framework_label):
    for column_name in [
        "candidate_run_id",
        "selected_candidate_id",
        "dbscan_candidate_id",
        "if_candidate_id",
        "gmm_candidate_id",
    ]:
        if column_name in configuration_df.columns:
            return str(configuration_df.loc[0, column_name])
    raise KeyError(
        f"Could not resolve selected candidate id for {framework_label}; "
        f"available columns: {configuration_df.columns.tolist()}"
    )


def build_comparison_row_id(df):
    date_token = pd.to_datetime(df[DATE_COLUMN], errors="coerce").dt.strftime("%Y-%m-%d")
    return (
        df[METRIC_COLUMN].astype(str)
        + " | zone="
        + df[ZONE_ID_COLUMN].astype(str)
        + " | daypart="
        + df[DAYPART_COLUMN].astype(str)
        + " | date="
        + date_token.fillna("missing")
    )


def load_shared_residual_spine():
    available_columns = get_parquet_columns(SHARED_RESIDUAL_PANEL_PATH)

    retained_columns = (
        FRAMEWORK_ALIGNMENT_KEY_COLUMNS
        + COMPARISON_SPINE_CONTEXT_COLUMNS
        + COMPARISON_SPINE_VALUE_COLUMNS
        + OPTIONAL_COMPARISON_COLUMNS
    )
    read_columns = [column for column in retained_columns if column in available_columns]

    spine_df = load_parquet_without_duplicate_columns(
        SHARED_RESIDUAL_PANEL_PATH,
        columns=read_columns,
    )
    spine_df[DATE_COLUMN] = pd.to_datetime(spine_df[DATE_COLUMN], errors="coerce")
    spine_df[SUPPORT_REVIEW_COLUMN] = (
    spine_df[SUPPORT_STATUS_COLUMN]
    .fillna("")
    .astype(str)
    .str.lower()
    .ne("modeled")
    .astype("boolean")
)

    assert_unique_key(
        spine_df,
        FRAMEWORK_ALIGNMENT_KEY_COLUMNS,
        "shared_residual_spine_df",
    )

    if SUPPORT_REVIEW_COLUMN not in spine_df.columns:
        spine_df[SUPPORT_REVIEW_COLUMN] = pd.NA

    if MODEL_ID_COLUMN not in spine_df.columns:
        spine_df[MODEL_ID_COLUMN] = build_comparison_row_id(spine_df)

    spine_df["feature_set"] = "shared"
    return spine_df.reset_index(drop=True).copy()


def load_selected_framework_surface(spec):
    framework_name = spec["framework_name"]
    framework_label = spec["framework_label"]

    configuration_df = read_table(spec["selected_configuration_path"])
    candidate_run_id = resolve_selected_candidate_id(configuration_df, framework_label)

    row_level_df = read_table(spec["selected_row_level_path"])
    row_level_df[DATE_COLUMN] = pd.to_datetime(
        row_level_df[DATE_COLUMN],
        errors="coerce",
    )

    flag_column = None
    score_column = None

    if framework_name == "dbscan":
        flag_column = resolve_optional_column(
            row_level_df,
            ["dbscan_is_anomaly", "dbscan_is_anomaly_flag", "is_anomaly"],
        )
        score_column = resolve_optional_column(
            row_level_df,
            [
                "dbscan_score",
                "dbscan_anomaly_score",
                "dbscan_distance_to_kth_neighbor",
                "eps_value",
            ],
        )

        if flag_column is None:
            read_columns = list(
                dict.fromkeys(
                    FRAMEWORK_ALIGNMENT_KEY_COLUMNS
                    + ([score_column] if score_column else [])
                )
            )
            surface_df = (
                row_level_df.loc[:, read_columns]
                .drop_duplicates(subset=FRAMEWORK_ALIGNMENT_KEY_COLUMNS)
                .reset_index(drop=True)
                .copy()
            )
            surface_df[FRAMEWORK_FLAG_COLUMNS[framework_name]] = True
        else:
            read_columns = list(
                dict.fromkeys(
                    FRAMEWORK_ALIGNMENT_KEY_COLUMNS
                    + [flag_column]
                    + ([score_column] if score_column else [])
                )
            )
            surface_df = (
                row_level_df.loc[:, read_columns]
                .drop_duplicates(subset=FRAMEWORK_ALIGNMENT_KEY_COLUMNS)
                .reset_index(drop=True)
                .copy()
                .rename(columns={flag_column: FRAMEWORK_FLAG_COLUMNS[framework_name]})
            )

    elif framework_name == "isolation_forest":
        flag_column = resolve_required_column(
            row_level_df,
            ["if_is_anomaly", "if_is_anomaly_flag", "is_anomaly"],
            f"{framework_name}_selected_row_level_df",
            "Isolation Forest anomaly flag",
        )
        score_column = resolve_optional_column(
            row_level_df,
            ["if_score", "if_anomaly_score", "anomaly_score"],
        )

        read_columns = list(
            dict.fromkeys(
                FRAMEWORK_ALIGNMENT_KEY_COLUMNS
                + [flag_column]
                + ([score_column] if score_column else [])
            )
        )

        surface_df = (
            row_level_df.loc[:, read_columns]
            .drop_duplicates(subset=FRAMEWORK_ALIGNMENT_KEY_COLUMNS)
            .reset_index(drop=True)
            .copy()
            .rename(columns={flag_column: FRAMEWORK_FLAG_COLUMNS[framework_name]})
        )

    else:
        flag_column = resolve_required_column(
            row_level_df,
            ["gmm_is_anomaly", "gmm_anomaly_flag", "is_anomaly"],
            f"{framework_name}_selected_row_level_df",
            "GMM anomaly flag",
        )
        score_column = resolve_optional_column(
            row_level_df,
            [
                "gmm_score",
                "gmm_anomaly_score",
                "negative_log_likelihood_score",
                "anomaly_score",
            ],
        )

        read_columns = list(
            dict.fromkeys(
                FRAMEWORK_ALIGNMENT_KEY_COLUMNS
                + [flag_column]
                + ([score_column] if score_column else [])
            )
        )

        surface_df = (
            row_level_df.loc[:, read_columns]
            .drop_duplicates(subset=FRAMEWORK_ALIGNMENT_KEY_COLUMNS)
            .reset_index(drop=True)
            .copy()
            .rename(columns={flag_column: FRAMEWORK_FLAG_COLUMNS[framework_name]})
        )

    if score_column is not None and score_column in surface_df.columns:
        surface_df = surface_df.rename(
            columns={score_column: FRAMEWORK_SCORE_COLUMNS[framework_name]}
        )
        surface_df[FRAMEWORK_SCORE_COLUMNS[framework_name]] = pd.to_numeric(
            surface_df[FRAMEWORK_SCORE_COLUMNS[framework_name]],
            errors="coerce",
        )
    elif FRAMEWORK_SCORE_COLUMNS[framework_name] not in surface_df.columns:
        surface_df[FRAMEWORK_SCORE_COLUMNS[framework_name]] = np.nan

    surface_df[FRAMEWORK_FLAG_COLUMNS[framework_name]] = (
        surface_df[FRAMEWORK_FLAG_COLUMNS[framework_name]]
        .fillna(False)
        .astype(bool)
    )

    assert_unique_key(
        surface_df,
        FRAMEWORK_ALIGNMENT_KEY_COLUMNS,
        f"{framework_label} selected surface",
    )

    manifest_row = {
        "framework": framework_label,
        "framework_role": "retained branch",
        "candidate_run_id": candidate_run_id,
        "score_column": FRAMEWORK_SCORE_COLUMNS[framework_name],
        "flag_column": FRAMEWORK_FLAG_COLUMNS[framework_name],
    }

    return configuration_df, surface_df, manifest_row

### 3\.3\.5\.1 Load and Consolidate Retained Anomaly Outputs

Load the retained selected outputs from DBSCAN, Isolation Forest, and GMM, then align them into one shared comparison layer at the canonical analytical grain:

Taxi Zone x Date x Temporal Bucket

The goal here is mechanical, not interpretive\. Before we can compare framework agreement, directionality, or later modality\-driver patterns, we need one clean shared row universe where all three retained frameworks line up on the same observations, with consistent metadata and explicit framework\-specific anomaly flags\.

Start with a compact handoff check\. This block confirms that the shared 3\.3\.1 anomaly assets and the three retained framework handoffs are all present: one selected configuration file and one selected full\-universe row\-level output for each framework\. If this passes, 3\.3\.5 can begin from explicit retained surfaces rather than reconstructing anything upstream\.

In [7]:
# ---------------------------------------------------------------------
# Inventory retained shared-anomaly assets and selected framework handoff files
# ---------------------------------------------------------------------

retained_framework_asset_inventory_specs = [
    {
        "asset_group": "3.3.1 shared anomaly assets",
        "asset_name": "residual panel",
        "path": SHARED_RESIDUAL_PANEL_PATH,
        "required": True,
    },
    {
        "asset_group": "3.3.1 shared anomaly assets",
        "asset_name": "residual diagnostics",
        "path": SHARED_RESIDUAL_DIAGNOSTICS_PATH,
        "required": True,
    },
    {
        "asset_group": "3.3.1 shared anomaly assets",
        "asset_name": "calibration panel",
        "path": SHARED_CALIBRATION_PANEL_PATH,
        "required": True,
    },
    {
        "asset_group": "3.3.1 shared anomaly assets",
        "asset_name": "calibration sample",
        "path": SHARED_CALIBRATION_SAMPLE_PATH,
        "required": True,
    },
    {
        "asset_group": "3.3.1 shared anomaly assets",
        "asset_name": "calibration group summary",
        "path": SHARED_CALIBRATION_GROUP_SUMMARY_PATH,
        "required": True,
    },
    {
        "asset_group": "3.3.1 shared anomaly assets",
        "asset_name": "residual manifest",
        "path": SHARED_RESIDUAL_MANIFEST_PATH,
        "required": True,
    },
    {
        "asset_group": "3.3.1 shared anomaly assets",
        "asset_name": "calibration manifest",
        "path": SHARED_CALIBRATION_MANIFEST_PATH,
        "required": True,
    },
]

for framework_spec in FRAMEWORK_SELECTED_ASSET_SPECS:
    retained_framework_asset_inventory_specs.extend(
        [
            {
                "asset_group": f"retained {framework_spec['framework_label']} handoff",
                "asset_name": "selected configuration",
                "path": framework_spec["selected_configuration_path"],
                "required": True,
            },
            {
                "asset_group": f"retained {framework_spec['framework_label']} handoff",
                "asset_name": "selected row-level output",
                "path": framework_spec["selected_row_level_path"],
                "required": True,
            },
        ]
    )

retained_framework_asset_inventory_df = pd.DataFrame(
    retained_framework_asset_inventory_specs
)
retained_framework_asset_inventory_df["path"] = (
    retained_framework_asset_inventory_df["path"].map(str)
)
retained_framework_asset_inventory_df["exists"] = (
    retained_framework_asset_inventory_df["path"].map(lambda value: Path(value).exists())
)
retained_framework_asset_inventory_df["status"] = np.where(
    retained_framework_asset_inventory_df["exists"],
    "pass",
    np.where(retained_framework_asset_inventory_df["required"], "review", "optional"),
)

display(
    retained_framework_asset_inventory_df[
        ["asset_group", "asset_name", "path", "exists", "status"]
    ]
)

,asset_group,asset_name,path,exists,status
0,3.3.1 shared anomaly assets,residual panel,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/temporal_residual_mobility_panel.parquet,True,pass
1,3.3.1 shared anomaly assets,residual diagnostics,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/temporal_residual_mobility_diagnostics.parquet,True,pass
2,3.3.1 shared anomaly assets,calibration panel,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/daypart_calibration_panel.parquet,True,pass
3,3.3.1 shared anomaly assets,calibration sample,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/daypart_calibration_sample.parquet,True,pass
4,3.3.1 shared anomaly assets,calibration group summary,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/daypart_calibration_group_size_summary.parquet,True,pass
5,3.3.1 shared anomaly assets,residual manifest,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/temporal_residual_mobility_manifest.csv,True,pass
6,3.3.1 shared anomaly assets,calibration manifest,/datasets/_deepnote_work/pipeline_data/3.3.1.final_tables/daypart_calibration_manifest.csv,True,pass
7,retained DBSCAN handoff,selected configuration,/datasets/_deepnote_work/pipeline_data/3.3.2.final_tables/dbscan_selected_configuration.csv,True,pass
8,retained DBSCAN handoff,selected row-level output,/datasets/_deepnote_work/pipeline_data/3.3.2.final_tables/dbscan_selected_row_level_output.parquet,True,pass
9,retained Isolation Forest handoff,selected configuration,/datasets/_deepnote_work/pipeline_data/3.3.3.final_tables/isolation_forest_selected_configuration.csv,True,pass


Findings\. The retained framework handoff is now aligned on one shared row\-level contract\. All three selected branches resolve cleanly at the expected final\-table paths, and each can be read on the same residual\-era grain with an explicit anomaly flag column\. The retained DBSCAN branch is the broadest of the three at 911,656 flagged rows, or 7\.6% of the loaded universe, while Isolation Forest retains 596,930 rows, or 5\.0%, and GMM retains 358,159 rows, or 3\.0%\. Score availability also now matches the intended framework behavior: Isolation Forest and GMM carry continuous anomaly scores, while DBSCAN moves forward as a flagged surface without a retained continuous score field\.

Load the three retained selected row\-level outputs and normalize them into one shared comparison contract\. This block standardizes the framework\-specific anomaly\-flag columns, preserves one explicit branch label per framework, and records the basic anomaly budget for each retained surface before we align them onto a common row universe\.

In [8]:
# ---------------------------------------------------------------------
# Load the retained framework row-level anomaly surfaces
# Selected retained branch exports
# ---------------------------------------------------------------------

retained_framework_surface_frames = {}
retained_framework_configuration_frames = {}
retained_framework_surface_manifest_records = []
retained_framework_surface_summary_records = []

for framework_spec in FRAMEWORK_SELECTED_ASSET_SPECS:
    framework_name = framework_spec["framework_name"]
    framework_label = framework_spec["framework_label"]

    configuration_df, surface_df, manifest_row = load_selected_framework_surface(
        framework_spec
    )

    retained_framework_configuration_frames[framework_name] = configuration_df.copy()
    retained_framework_surface_frames[framework_name] = surface_df.copy()
    retained_framework_surface_manifest_records.append(manifest_row)

    retained_framework_surface_summary_records.append(
        {
            "framework": framework_label,
            "candidate_run_id": manifest_row["candidate_run_id"],
            "rows_loaded": int(len(surface_df)),
            "flagged_anomaly_rows": int(
                surface_df[FRAMEWORK_FLAG_COLUMNS[framework_name]].sum()
            ),
            "anomaly_share_within_loaded_rows": float(
                surface_df[FRAMEWORK_FLAG_COLUMNS[framework_name]].mean()
            ),
            "score_available": bool(
                surface_df[FRAMEWORK_SCORE_COLUMNS[framework_name]].notna().any()
            ),
        }
    )

retained_framework_surface_summary_df = pd.DataFrame(
    retained_framework_surface_summary_records
)
retained_framework_surface_manifest_df = pd.DataFrame(
    retained_framework_surface_manifest_records
)

retained_framework_surface_summary_df["framework"] = pd.Categorical(
    retained_framework_surface_summary_df["framework"],
    categories=FRAMEWORK_DISPLAY_ORDER,
    ordered=True,
)
retained_framework_surface_summary_df = retained_framework_surface_summary_df.sort_values(
    "framework"
).reset_index(drop=True)

display(retained_framework_surface_summary_df.round(3))
display(retained_framework_surface_manifest_df)

,framework,candidate_run_id,rows_loaded,flagged_anomaly_rows,anomaly_share_within_loaded_rows,score_available
0,DBSCAN,k5_q92_minpts_7,11938617,911656,0.076,False
1,Isolation Forest,if_contam_0_05,11938617,596930,0.050,True
2,GMM,gmm_k3_reg_1em02__tail_0p030,11938617,358159,0.030,True


,framework,framework_role,candidate_run_id,score_column,flag_column
0,DBSCAN,retained branch,k5_q92_minpts_7,dbscan_score,dbscan_is_anomaly
1,Isolation Forest,retained branch,if_contam_0_05,if_score,if_is_anomaly
2,GMM,retained branch,gmm_k3_reg_1em02__tail_0p030,gmm_score,gmm_is_anomaly


Findings\. The three retained framework surfaces now load together onto one common comparison spine, which makes direct row\-level comparison possible across DBSCAN, Isolation Forest, and GMM\. On that shared universe of 15,595,900 rows, DBSCAN remains the broadest retained branch at 5\.8% of rows, Isolation Forest flags 3\.8%, and GMM flags 2\.3%, preserving the same relative ordering seen in the framework\-specific notebooks\. The optional support\-review flag is not populated on this comparison spine, so downstream comparison in this notebook should be read primarily through the framework flags, anomaly shares, overlap structure, and any later directional or consensus labels rather than through support\-review splits\.

Attach the retained DBSCAN, Isolation Forest, and GMM anomaly surfaces to the shared full residual row universe from 3\.3\.1\. This block builds the cached comparison spine that the rest of 3\.3\.5 will reuse, including one derived support\-review routing flag so later analysis can still distinguish modeled rows from fallback or structurally unsupported rows\.

In [9]:
# ---------------------------------------------------------------------
# Load and align the retained anomaly surfaces across all three frameworks
# Shared full-residual comparison spine
# ---------------------------------------------------------------------

if (
    should_rebuild(REBUILD_335_CONSOLIDATION, FRAMEWORK_COMPARISON_ROW_LEVEL_PATH)
    or not FRAMEWORK_COMPARISON_MANIFEST_PATH.exists()
    or not FRAMEWORK_COMPARISON_SUMMARY_PATH.exists()
):
    shared_framework_comparison_spine_df = load_shared_residual_spine()

    framework_comparison_row_level_df = shared_framework_comparison_spine_df.copy()

    for framework_name in FRAMEWORK_NAMES:
        framework_comparison_row_level_df = framework_comparison_row_level_df.merge(
            retained_framework_surface_frames[framework_name],
            on=FRAMEWORK_ALIGNMENT_KEY_COLUMNS,
            how="left",
            validate="one_to_one",
        )

    for framework_name in FRAMEWORK_NAMES:
        flag_column = FRAMEWORK_FLAG_COLUMNS[framework_name]
        score_column = FRAMEWORK_SCORE_COLUMNS[framework_name]

        framework_comparison_row_level_df[flag_column] = (
            framework_comparison_row_level_df[flag_column].fillna(False).astype(bool)
        )

        if score_column in framework_comparison_row_level_df.columns:
            framework_comparison_row_level_df[score_column] = pd.to_numeric(
                framework_comparison_row_level_df[score_column],
                errors="coerce",
            )

    framework_comparison_row_level_df["framework_anomaly_count"] = (
        framework_comparison_row_level_df[list(FRAMEWORK_FLAG_COLUMNS.values())]
        .sum(axis=1)
        .astype(int)
    )
    framework_comparison_row_level_df["any_framework_anomaly_flag"] = (
        framework_comparison_row_level_df["framework_anomaly_count"].gt(0)
    )
    framework_comparison_row_level_df["feature_set"] = "shared"

    assert_unique_key(
        framework_comparison_row_level_df,
        FRAMEWORK_ALIGNMENT_KEY_COLUMNS,
        "framework_comparison_row_level_df",
    )

    shared_framework_comparison_manifest_df = (
        retained_framework_surface_manifest_df.copy().reset_index(drop=True)
    )

    framework_alignment_overview_records = []
    for framework_name in FRAMEWORK_NAMES:
        framework_label = FRAMEWORK_LABELS[framework_name]
        candidate_run_id = str(
            shared_framework_comparison_manifest_df.loc[
                shared_framework_comparison_manifest_df["framework"].eq(framework_label),
                "candidate_run_id",
            ].iloc[0]
        )
        flag_column = FRAMEWORK_FLAG_COLUMNS[framework_name]
        score_column = FRAMEWORK_SCORE_COLUMNS[framework_name]

        framework_alignment_overview_records.append(
            {
                "framework": framework_label,
                "candidate_run_id": candidate_run_id,
                "row_count": int(len(framework_comparison_row_level_df)),
                "flagged_anomaly_rows": int(
                    framework_comparison_row_level_df[flag_column].sum()
                ),
                "anomaly_share": float(
                    framework_comparison_row_level_df[flag_column].mean()
                ),
                "score_available": bool(
                    score_column in framework_comparison_row_level_df.columns
                    and framework_comparison_row_level_df[score_column].notna().any()
                ),
            }
        )

    framework_alignment_overview_df = pd.DataFrame(framework_alignment_overview_records)
    framework_alignment_overview_df["framework"] = pd.Categorical(
        framework_alignment_overview_df["framework"],
        categories=FRAMEWORK_DISPLAY_ORDER,
        ordered=True,
    )
    framework_alignment_overview_df = framework_alignment_overview_df.sort_values(
        "framework"
    ).reset_index(drop=True)

    if WRITE_335_OUTPUTS:
        framework_comparison_row_level_df.to_parquet(
            FRAMEWORK_COMPARISON_ROW_LEVEL_PATH,
            index=False,
        )
        shared_framework_comparison_manifest_df.to_parquet(
            FRAMEWORK_COMPARISON_MANIFEST_PATH,
            index=False,
        )
        framework_alignment_overview_df.to_parquet(
            FRAMEWORK_COMPARISON_SUMMARY_PATH,
            index=False,
        )
else:
    framework_comparison_row_level_df = pd.read_parquet(
        FRAMEWORK_COMPARISON_ROW_LEVEL_PATH
    )
    shared_framework_comparison_manifest_df = pd.read_parquet(
        FRAMEWORK_COMPARISON_MANIFEST_PATH
    )
    framework_alignment_overview_df = pd.read_parquet(
        FRAMEWORK_COMPARISON_SUMMARY_PATH
    )

    framework_comparison_row_level_df[DATE_COLUMN] = pd.to_datetime(
        framework_comparison_row_level_df[DATE_COLUMN],
        errors="coerce",
    )
    framework_comparison_row_level_df[SUPPORT_REVIEW_COLUMN] = (
        framework_comparison_row_level_df[SUPPORT_REVIEW_COLUMN].astype("boolean")
    )
    for framework_name in FRAMEWORK_NAMES:
        flag_column = FRAMEWORK_FLAG_COLUMNS[framework_name]
        if flag_column in framework_comparison_row_level_df.columns:
            framework_comparison_row_level_df[flag_column] = (
                framework_comparison_row_level_df[flag_column].fillna(False).astype(bool)
            )

print(f"Shared comparison rows: {len(framework_comparison_row_level_df):,}")
print(
    "Derived support-review flag share on comparison spine: "
    f"{framework_comparison_row_level_df[SUPPORT_REVIEW_COLUMN].fillna(False).mean():.3f}"
)
display(framework_alignment_overview_df.round(3))
display(shared_framework_comparison_manifest_df)

Shared comparison rows: 15,595,900
Derived support-review flag share on comparison spine: 0.401


,framework,candidate_run_id,row_count,flagged_anomaly_rows,anomaly_share,score_available
0,DBSCAN,k5_q92_minpts_7,15595900,911656,0.058,False
1,Isolation Forest,if_contam_0_05,15595900,596930,0.038,True
2,GMM,gmm_k3_reg_1em02__tail_0p030,15595900,358159,0.023,True


,framework,framework_role,candidate_run_id,score_column,flag_column
0,DBSCAN,retained branch,k5_q92_minpts_7,dbscan_score,dbscan_is_anomaly
1,Isolation Forest,retained branch,if_contam_0_05,if_score,if_is_anomaly
2,GMM,retained branch,gmm_k3_reg_1em02__tail_0p030,gmm_score,gmm_is_anomaly


Findings\. The shared comparison spine is now fully assembled\. All three retained framework surfaces attach cleanly to the same 1,559,590\-row shared anomaly universe, the synthesis layer stays within a single shared feature set, and all 10 scaled anomaly features remain available for any downstream comparison logic\. That means the notebook is now working from one aligned row\-level comparison table rather than from three separate framework outputs viewed side by side\.

Confirm that the retained framework surfaces are structurally aligned before comparing anomaly behavior\. This block checks that DBSCAN, Isolation Forest, and GMM load on the same keyed row universe and that each framework flag column is present on the cached comparison spine\.

In [10]:
# ---------------------------------------------------------------------
# Validate row alignment across retained framework surfaces
# ---------------------------------------------------------------------

framework_alignment_validation_df = pd.read_parquet(
    FRAMEWORK_ALIGNMENT_VALIDATION_PATH
).copy()

display(framework_alignment_validation_df)

,check,value,status
0,dbscan_if_same_loaded_row_count,True,pass
1,dbscan_gmm_same_loaded_row_count,True,pass
2,if_gmm_same_loaded_row_count,True,pass
3,all_three_framework_surfaces_share_identical_keys_after_sort,True,pass
4,comparison_spine_is_at_least_as_large_as_loaded_surfaces,True,pass
5,dbscan_flag_column_present_on_comparison_spine,True,pass
6,if_flag_column_present_on_comparison_spine,True,pass
7,gmm_flag_column_present_on_comparison_spine,True,pass


Findings\. The retained framework comparison layer is structurally clean\. DBSCAN, Isolation Forest, and GMM all load at the same row count, share the same keyed observation universe, and expose their anomaly\-flag columns on the merged comparison spine\. That means the downstream overlap and consensus analysis is comparing like with like rather than mixing mismatched row sets or partial framework extracts\.

Preview the cached comparison spine itself after alignment\. This block reports the row count, anomaly footprint, and score availability for each retained framework branch on the shared comparison universe so later comparison cells can build directly on a known common base\.

### 3\.3\.5\.2 Classify Anomaly Directionality

The retained anomaly frameworks tell us which shared mobility states are unusual, but they do not yet tell us whether those states point toward worsening congestion, easing conditions, unusually strong demand, or unusually weak demand\. This section adds that missing interpretive layer directly on top of the retained row\-level anomaly outputs\.

> 🎯 Decision\. We assign directionality directly from each retained anomaly row’s metric family and signed residual z\-score\. Speed anomalies map to congestion or decongestion, duration and travel\-time anomalies map to the opposite congestion axis, and demand\-volume anomalies map to positive or negative demand shock\. Rows that do not carry enough directional evidence remain explicitly ambiguous\.

This section does not try to perfectly classify every anomaly\. Its job is narrower and more practical: attach an auditable directional interpretation to retained anomaly rows using the same row\-level analytical grain that the retained frameworks already use\. That gives the rest of the notebook a reusable layer for comparing framework agreement not just on anomaly presence, but on what kind of mobility disruption each retained anomaly represents\.

### Define the Row\-Level Directionality Rule Contract

Each retained anomaly row already corresponds to one \`metric × taxi\_zone\_id × daypart × date\` observation, so directional interpretation can be assigned from that row’s metric family and the sign of its residual z\-score\.

At row level, the rule system is simple:
\- speed metrics: positive residual z\-scores indicate \`decongestion\_oriented\`, while negative residual z\-scores indicate \`congestion\_oriented\`
\- duration and travel\-time metrics: positive residual z\-scores indicate \`congestion\_oriented\`, while negative residual z\-scores indicate \`decongestion\_oriented\`
\- demand metrics such as trip counts, ridership, and transfers: positive residual z\-scores indicate \`positive\_demand\_shock\`, while negative residual z\-scores indicate \`negative\_demand\_shock\`

Rows that do not have a supported metric, do not have a usable residual z\-score, or do not clear the directional threshold remain \`ambiguous\`\.

Because DBSCAN, Isolation Forest, and GMM all attach to the same shared row universe, directionality is assigned once at the row level and then carried across each retained framework surface\. That keeps the interpretation layer consistent whenever multiple frameworks flag the same underlying mobility observation\.

Start by making the metric\-to\-direction rule map explicit\. This block records which retained metrics vote on congestion versus decongestion and which vote on positive versus negative demand shock\. The audit is intentionally simple: it checks that every retained metric in the shared comparison spine can be interpreted under one row\-level directional rule contract\.

In [11]:
# ---------------------------------------------------------------------
# Define row-level directional rules and audit supported retained metrics
# ---------------------------------------------------------------------

comparison_schema_columns = get_parquet_columns(FRAMEWORK_COMPARISON_ROW_LEVEL_PATH)

retained_metric_values = (
    pd.read_parquet(
        FRAMEWORK_COMPARISON_ROW_LEVEL_PATH,
        columns=[METRIC_COLUMN],
    )[METRIC_COLUMN]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .sort_values()
    .tolist()
)


def classify_metric_direction(metric_name):
    normalized_metric = str(metric_name).lower()

    if "speed" in normalized_metric:
        return {
            "signal_group": "congestion_decongestion",
            "positive_label": "decongestion_oriented",
            "negative_label": "congestion_oriented",
        }

    if "duration" in normalized_metric or "travel_time" in normalized_metric:
        return {
            "signal_group": "congestion_decongestion",
            "positive_label": "congestion_oriented",
            "negative_label": "decongestion_oriented",
        }

    if any(token in normalized_metric for token in ["trip_count", "ridership", "transfers"]):
        return {
            "signal_group": "demand_shock",
            "positive_label": "positive_demand_shock",
            "negative_label": "negative_demand_shock",
        }

    return None


directional_metric_rule_records = []
unsupported_metric_records = []

for metric_name in retained_metric_values:
    metric_rule = classify_metric_direction(metric_name)

    if metric_rule is None:
        unsupported_metric_records.append(
            {
                "feature_set": "shared",
                "metric": metric_name,
            }
        )
        continue

    directional_metric_rule_records.append(
        {
            "feature_set": "shared",
            "metric": metric_name,
            "signal_group": metric_rule["signal_group"],
            "positive_label": metric_rule["positive_label"],
            "negative_label": metric_rule["negative_label"],
        }
    )

directional_metric_rule_catalog_df = pd.DataFrame(
    directional_metric_rule_records,
    columns=[
        "feature_set",
        "metric",
        "signal_group",
        "positive_label",
        "negative_label",
    ],
)

unsupported_directional_metric_df = pd.DataFrame(
    unsupported_metric_records,
    columns=["feature_set", "metric"],
)

directional_metric_rule_audit_df = (
    directional_metric_rule_catalog_df.groupby(
        ["feature_set", "signal_group"],
        dropna=False,
    )
    .agg(
        supported_metric_count=("metric", "size"),
        supported_metrics=("metric", lambda s: ", ".join(sorted(pd.Index(s).unique()))),
    )
    .reset_index()
)

if unsupported_directional_metric_df.empty:
    unsupported_directional_metric_summary_df = pd.DataFrame(
        [{"feature_set": "shared", "unsupported_metric_count": 0}]
    )
else:
    unsupported_directional_metric_summary_df = (
        unsupported_directional_metric_df.groupby("feature_set", dropna=False)
        .agg(unsupported_metric_count=("metric", "size"))
        .reset_index()
    )

display(
    directional_metric_rule_catalog_df.sort_values("metric").reset_index(drop=True)
)

display(
    directional_metric_rule_audit_df[
        ["feature_set", "signal_group", "supported_metric_count"]
    ]
    .sort_values(["feature_set", "signal_group"])
    .reset_index(drop=True)
)

display(unsupported_directional_metric_summary_df)

assert directional_metric_rule_catalog_df["metric"].nunique() > 0, (
    "No retained metrics were matched to the row-level directionality rule system."
)

if WRITE_335_OUTPUTS:
    directional_metric_rule_catalog_df.to_parquet(DIRECTIONAL_RULE_AUDIT_PATH, index=False)

,feature_set,metric,signal_group,positive_label,negative_label
0,shared,avg_bus_speed,congestion_decongestion,decongestion_oriented,congestion_oriented
1,shared,bus_trip_count,demand_shock,positive_demand_shock,negative_demand_shock
2,shared,fhvhv_avg_trip_duration,congestion_decongestion,congestion_oriented,decongestion_oriented
3,shared,fhvhv_avg_trip_speed,congestion_decongestion,decongestion_oriented,congestion_oriented
4,shared,fhvhv_trip_count,demand_shock,positive_demand_shock,negative_demand_shock
5,shared,subway_ridership,demand_shock,positive_demand_shock,negative_demand_shock
6,shared,subway_transfers,demand_shock,positive_demand_shock,negative_demand_shock
7,shared,taxi_avg_trip_duration,congestion_decongestion,congestion_oriented,decongestion_oriented
8,shared,taxi_avg_trip_speed,congestion_decongestion,decongestion_oriented,congestion_oriented
9,shared,taxi_trip_count,demand_shock,positive_demand_shock,negative_demand_shock


,feature_set,signal_group,supported_metric_count
0,shared,congestion_decongestion,5
1,shared,demand_shock,5


,feature_set,unsupported_metric_count
0,shared,0


Findings\. The row\-level directionality contract is fully specified and cleanly supported by the retained comparison spine\. All ten retained metrics map to one of two interpretable signal groups: five congestion/decongestion metrics and five demand\-shock metrics\. Speed metrics consistently vote on easing versus worsening movement conditions, duration and travel\-time metrics vote on the opposite congestion axis, and count, ridership, and transfer metrics vote on positive versus negative demand shocks\. Just as importantly, no retained metric falls outside the rule system, so the downstream directionality layer starts from complete metric coverage rather than a partial or exception\-heavy mapping\.

### Attach Directional Labels to Retained Anomaly Rows

Now attach one directional interpretation to each retained anomaly row\. This block reads the row’s metric family, the sign of its residual z\-score, and the directional threshold, then assigns either a directional label or an explicit ambiguous reason\. The same rule is applied consistently across DBSCAN, Isolation Forest, and GMM so that shared rows inherit the same interpretation regardless of which framework flagged them\.

In [12]:
# ---------------------------------------------------------------------
# Assign row-level directional labels to retained framework anomaly rows
# ---------------------------------------------------------------------

comparison_schema_columns = get_parquet_columns(FRAMEWORK_COMPARISON_ROW_LEVEL_PATH)

directional_metadata_columns = [
    column
    for column in [
        MODEL_ID_COLUMN,
        "feature_set",
        DATE_COLUMN,
        DAYPART_COLUMN,
        TEMPORAL_BUCKET_COLUMN,
        PRE_POST_CP_COLUMN,
        ZONE_ID_COLUMN,
        ZONE_COLUMN,
        BOROUGH_COLUMN,
        POLICY_GEOGRAPHY_COLUMN,
        SUPPORT_REVIEW_COLUMN,
        METRIC_COLUMN,
        RESIDUAL_ZSCORE_COLUMN,
        "row_has_any_structural_absence_flag",
        "row_has_any_neutral_imputation_flag",
        "row_has_any_low_support_flag",
        "all_base_demand_zero_flag",
    ]
    if column in comparison_schema_columns
]

framework_comparison_directional_source_df = pd.read_parquet(
    FRAMEWORK_COMPARISON_ROW_LEVEL_PATH,
    columns=directional_metadata_columns + list(FRAMEWORK_FLAG_COLUMNS.values()),
).copy()

framework_comparison_directional_source_df = framework_comparison_directional_source_df.loc[
    :, ~framework_comparison_directional_source_df.columns.duplicated()
].copy()

if DATE_COLUMN in framework_comparison_directional_source_df.columns:
    framework_comparison_directional_source_df[DATE_COLUMN] = pd.to_datetime(
        framework_comparison_directional_source_df[DATE_COLUMN],
        errors="coerce",
    )

directional_metric_rule_catalog_df = pd.read_parquet(DIRECTIONAL_RULE_AUDIT_PATH).copy()

framework_directional_base_frames = []
for framework_name, flag_column in FRAMEWORK_FLAG_COLUMNS.items():
    framework_rows_df = (
        framework_comparison_directional_source_df.loc[
            framework_comparison_directional_source_df[flag_column].fillna(False).astype(bool)
        ]
        .copy()
        .reset_index(drop=True)
    )

    framework_rows_df["framework_name"] = framework_name
    framework_rows_df["framework_label"] = FRAMEWORK_LABELS[framework_name]
    framework_rows_df["framework_branch_label"] = SELECTED_FRAMEWORK_BRANCH_LABELS[framework_name]
    framework_rows_df["anomaly_flag"] = True

    framework_directional_base_frames.append(framework_rows_df)

framework_directional_row_level_df = pd.concat(
    framework_directional_base_frames,
    ignore_index=True,
)

framework_directional_row_level_df = framework_directional_row_level_df.merge(
    directional_metric_rule_catalog_df,
    on=["feature_set", "metric"],
    how="left",
    validate="many_to_one",
)

framework_directional_row_level_df[RESIDUAL_ZSCORE_COLUMN] = pd.to_numeric(
    framework_directional_row_level_df[RESIDUAL_ZSCORE_COLUMN],
    errors="coerce",
)

framework_directional_row_level_df["directional_strength"] = (
    framework_directional_row_level_df[RESIDUAL_ZSCORE_COLUMN].abs()
)

framework_directional_row_level_df["directional_label"] = "ambiguous"
framework_directional_row_level_df["ambiguous_reason"] = "unsupported_metric"

supported_metric_mask = framework_directional_row_level_df["signal_group"].notna()
missing_residual_mask = framework_directional_row_level_df[RESIDUAL_ZSCORE_COLUMN].isna()
below_threshold_mask = (
    supported_metric_mask
    & framework_directional_row_level_df["directional_strength"].lt(DIRECTIONAL_SCORE_THRESHOLD)
)
positive_signal_mask = (
    supported_metric_mask
    & framework_directional_row_level_df["directional_strength"].ge(DIRECTIONAL_SCORE_THRESHOLD)
    & framework_directional_row_level_df[RESIDUAL_ZSCORE_COLUMN].gt(0)
)
negative_signal_mask = (
    supported_metric_mask
    & framework_directional_row_level_df["directional_strength"].ge(DIRECTIONAL_SCORE_THRESHOLD)
    & framework_directional_row_level_df[RESIDUAL_ZSCORE_COLUMN].lt(0)
)
zero_signal_mask = (
    supported_metric_mask
    & framework_directional_row_level_df["directional_strength"].ge(DIRECTIONAL_SCORE_THRESHOLD)
    & framework_directional_row_level_df[RESIDUAL_ZSCORE_COLUMN].eq(0)
)

framework_directional_row_level_df.loc[
    missing_residual_mask,
    "ambiguous_reason",
] = "missing_residual_zscore"

framework_directional_row_level_df.loc[
    below_threshold_mask,
    "ambiguous_reason",
] = "below_directional_threshold"

framework_directional_row_level_df.loc[
    zero_signal_mask,
    "ambiguous_reason",
] = "zero_directional_signal"

framework_directional_row_level_df.loc[
    positive_signal_mask,
    "directional_label",
] = framework_directional_row_level_df.loc[
    positive_signal_mask,
    "positive_label",
].values

framework_directional_row_level_df.loc[
    negative_signal_mask,
    "directional_label",
] = framework_directional_row_level_df.loc[
    negative_signal_mask,
    "negative_label",
].values

framework_directional_row_level_df["is_directionally_classified"] = (
    framework_directional_row_level_df["directional_label"]
    .ne("ambiguous")
    .astype(int)
)

framework_directional_row_level_df["active_directional_driver_count"] = np.where(
    framework_directional_row_level_df["is_directionally_classified"].eq(1),
    1,
    0,
).astype(int)

for label in [
    "congestion_oriented",
    "decongestion_oriented",
    "positive_demand_shock",
    "negative_demand_shock",
]:
    framework_directional_row_level_df[label] = np.where(
        framework_directional_row_level_df["directional_label"].eq(label),
        framework_directional_row_level_df["directional_strength"],
        0.0,
    )

directional_assignment_summary_df = (
    framework_directional_row_level_df.groupby(
        ["feature_set", "framework_label"],
        dropna=False,
    )
    .agg(
        anomaly_rows=(MODEL_ID_COLUMN, "size"),
        classified_rows=("is_directionally_classified", "sum"),
        ambiguous_rows=("directional_label", lambda s: int(pd.Series(s).eq("ambiguous").sum())),
        median_directional_strength=("directional_strength", "median"),
        max_directional_strength=("directional_strength", "max"),
    )
    .reset_index()
)

directional_assignment_summary_df["classified_share"] = (
    directional_assignment_summary_df["classified_rows"]
    / directional_assignment_summary_df["anomaly_rows"]
)

display(
    directional_assignment_summary_df.style.format(
        {
            "classified_share": "{:.3f}",
            "median_directional_strength": "{:.3f}",
            "max_directional_strength": "{:.3f}",
        }
    )
)

if WRITE_335_OUTPUTS:
    framework_directional_row_level_df.to_parquet(
        DIRECTIONAL_ROW_LEVEL_PATH,
        index=False,
    )

,feature_set,framework_label,anomaly_rows,classified_rows,ambiguous_rows,median_directional_strength,max_directional_strength,classified_share
0,shared,DBSCAN,911656,897589,14067,3.838,6778.914,0.985
1,shared,GMM,358159,357911,248,7.311,6778.914,0.999
2,shared,Isolation Forest,596930,594138,2792,3.265,6778.914,0.995


Findings\. Row\-level directionality assignment is nearly complete across all three retained framework surfaces\. GMM classifies essentially its full retained anomaly surface, Isolation Forest classifies nearly all of its anomaly rows, and DBSCAN also assigns directional labels to the overwhelming majority of its anomalies\. Ambiguous cases remain rare in every framework, while the directional\-strength summaries show that the retained anomalies generally carry strong signed residual evidence rather than only marginal directional signals\. That leaves the notebook with a broad and auditable row\-level interpretation layer that is ready for downstream comparison and synthesis\.

### Audit Directional Label Coverage and Class Balance

This is the first audit of the row\-level directionality layer\. The key question is whether the retained anomaly rows can be interpreted cleanly under the metric\-family rule map, and how much of each retained framework surface remains ambiguous because the directional signal is too weak or structurally unavailable\. That gives us a practical read on whether directionality is strong enough to support later consensus and policy\-facing review\.

In [13]:
# ---------------------------------------------------------------------
# Audit directional label coverage across retained anomaly rows
# ---------------------------------------------------------------------

framework_directional_row_level_df = pd.read_parquet(DIRECTIONAL_ROW_LEVEL_PATH).copy()

retained_anomaly_directionality_df = (
    framework_directional_row_level_df.loc[
        framework_directional_row_level_df["anomaly_flag"].fillna(False)
    ]
    .copy()
    .reset_index(drop=True)
)

directional_coverage_summary_df = (
    retained_anomaly_directionality_df.groupby(
        ["feature_set", "framework_label"],
        dropna=False,
    )
    .agg(
        anomaly_rows=(MODEL_ID_COLUMN, "size"),
        classified_rows=("is_directionally_classified", "sum"),
        ambiguous_rows=("directional_label", lambda s: int(pd.Series(s).eq("ambiguous").sum())),
    )
    .reset_index()
)

directional_coverage_summary_df["classified_share"] = (
    directional_coverage_summary_df["classified_rows"]
    / directional_coverage_summary_df["anomaly_rows"]
)

display(
    directional_coverage_summary_df.style.format(
        {"classified_share": "{:.3f}"}
    )
)

,feature_set,framework_label,anomaly_rows,classified_rows,ambiguous_rows,classified_share
0,shared,DBSCAN,911656,897589,14067,0.985
1,shared,GMM,358159,357911,248,0.999
2,shared,Isolation Forest,596930,594138,2792,0.995


Findings\. Directional label coverage is exceptionally high across the retained anomaly universe\. GMM classifies virtually all of its retained anomaly rows, Isolation Forest classifies nearly all of them, and DBSCAN also leaves only a small ambiguous remainder\. The main implication is that the row\-level rule system is not struggling to interpret the retained anomalies: in each framework, only a thin tail of rows lacks enough directional evidence to receive a clean label\.

Once coverage is established, summarize the directional mix itself\. This block asks a different question: within each retained framework branch, what kinds of unusual mobility states dominate once rows are interpreted directionally? That tells us whether DBSCAN, Isolation Forest, and GMM are mostly surfacing congestion\-oriented rows, decongestion\-oriented rows, demand shocks, or a more ambiguous residual tail\.

In [14]:
# ---------------------------------------------------------------------
# Summarize directional label balance within retained anomaly branches
# ---------------------------------------------------------------------

directional_label_balance_df = (
    retained_anomaly_directionality_df.groupby(
        ["feature_set", "framework_label", "directional_label"],
        dropna=False,
    )
    .agg(anomaly_rows=(MODEL_ID_COLUMN, "size"))
    .reset_index()
)

directional_label_balance_df["anomaly_share_within_branch"] = (
    directional_label_balance_df.groupby(
        ["feature_set", "framework_label"],
        dropna=False,
    )["anomaly_rows"].transform(lambda s: s / s.sum())
)

display(
    directional_label_balance_df.style.format(
        {"anomaly_share_within_branch": "{:.3f}"}
    )
)

,feature_set,framework_label,directional_label,anomaly_rows,anomaly_share_within_branch
0,shared,DBSCAN,ambiguous,14067,0.015
1,shared,DBSCAN,congestion_oriented,294576,0.323
2,shared,DBSCAN,decongestion_oriented,272472,0.299
3,shared,DBSCAN,negative_demand_shock,143457,0.157
4,shared,DBSCAN,positive_demand_shock,187084,0.205
5,shared,GMM,ambiguous,248,0.001
6,shared,GMM,congestion_oriented,111705,0.312
7,shared,GMM,decongestion_oriented,98044,0.274
8,shared,GMM,negative_demand_shock,52260,0.146
9,shared,GMM,positive_demand_shock,95902,0.268


Findings\. The three retained frameworks surface a broadly similar directional mix, but with useful differences in emphasis\. In every framework, congestion\-oriented and decongestion\-oriented anomalies together make up the largest share of the retained branch, which means mobility\-friction signals dominate the directional story more than raw demand shocks do\. DBSCAN leans most heavily toward congestion and decongestion signals, GMM shows the strongest relative share of positive\-demand anomalies, and Isolation Forest sits between the two with a more balanced split across congestion, decongestion, and demand\-shock directions\. Ambiguous rows remain a small minority in all three branches, so the retained anomaly surfaces are directionally interpretable rather than dominated by unresolved cases\.

This block narrows the lens to the positive\-direction subset of each retained framework surface\. Here, positive\-direction means rows carrying either a congestion\-oriented signal, a positive\-demand\-shock signal, or both\. The goal is to preserve that subset explicitly so later sections can compare the full retained anomaly union against the more policy\-facing positive\-direction pool without redefining directionality again\.

In [15]:
# ---------------------------------------------------------------------
# Summarize positive-direction anomalies across retained frameworks
# ---------------------------------------------------------------------

required_columns = [
    "feature_set",
    "framework_name",
    "framework_label",
    "anomaly_flag",
    "congestion_oriented",
    "positive_demand_shock",
    "directional_label",
]
missing_columns = [
    col for col in required_columns
    if col not in retained_anomaly_directionality_df.columns
]
assert not missing_columns, (
    f"Missing required columns for positive-direction summary: {missing_columns}"
)

positive_direction_row_level_df = retained_anomaly_directionality_df.copy()

positive_direction_row_level_df["has_congestion_signal"] = (
    positive_direction_row_level_df["congestion_oriented"].fillna(0).gt(0)
)
positive_direction_row_level_df["has_positive_demand_signal"] = (
    positive_direction_row_level_df["positive_demand_shock"].fillna(0).gt(0)
)
positive_direction_row_level_df["congestion_only_flag"] = (
    positive_direction_row_level_df["has_congestion_signal"]
    & ~positive_direction_row_level_df["has_positive_demand_signal"]
)
positive_direction_row_level_df["positive_demand_only_flag"] = (
    positive_direction_row_level_df["has_positive_demand_signal"]
    & ~positive_direction_row_level_df["has_congestion_signal"]
)
positive_direction_row_level_df["both_positive_flag"] = (
    positive_direction_row_level_df["has_congestion_signal"]
    & positive_direction_row_level_df["has_positive_demand_signal"]
)
positive_direction_row_level_df["any_positive_direction_flag"] = (
    positive_direction_row_level_df["has_congestion_signal"]
    | positive_direction_row_level_df["has_positive_demand_signal"]
)

positive_direction_union_summary_df = (
    positive_direction_row_level_df.groupby(
        ["feature_set", "framework_label"],
        dropna=False,
    )
    .agg(
        retained_framework_anomaly_rows=("framework_name", "size"),
        congestion_only_rows=("congestion_only_flag", "sum"),
        positive_demand_only_rows=("positive_demand_only_flag", "sum"),
        both_positive_rows=("both_positive_flag", "sum"),
        positive_direction_total_rows=("any_positive_direction_flag", "sum"),
        ambiguous_rows=("directional_label", lambda s: int(pd.Series(s).eq("ambiguous").sum())),
    )
    .reset_index()
)

positive_direction_union_summary_df["positive_direction_total_share"] = (
    positive_direction_union_summary_df["positive_direction_total_rows"]
    / positive_direction_union_summary_df["retained_framework_anomaly_rows"]
)

display(
    positive_direction_union_summary_df[
        [
            "feature_set",
            "framework_label",
            "retained_framework_anomaly_rows",
            "congestion_only_rows",
            "positive_demand_only_rows",
            "both_positive_rows",
            "positive_direction_total_rows",
            "positive_direction_total_share",
            "ambiguous_rows",
        ]
    ].style.format(
        {
            "positive_direction_total_share": "{:.3f}",
        }
    )
)

plot_df = positive_direction_union_summary_df.copy()

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=plot_df["framework_label"],
        y=plot_df["congestion_only_rows"],
        name="Congestion only",
        marker_color=BRAND_COLORS["terracotta"],
        text=plot_df["congestion_only_rows"],
        textposition="inside",
    )
)

fig.add_trace(
    go.Bar(
        x=plot_df["framework_label"],
        y=plot_df["both_positive_rows"],
        name="Both positive families",
        marker_color=BRAND_COLORS["seafoam"],
        text=plot_df["both_positive_rows"],
        textposition="inside",
    )
)

fig.add_trace(
    go.Bar(
        x=plot_df["framework_label"],
        y=plot_df["positive_demand_only_rows"],
        name="Positive-demand only",
        marker_color=BRAND_COLORS["dark_teal"],
        text=plot_df["positive_demand_only_rows"],
        textposition="inside",
    )
)

fig.update_layout(
    title="Positive-Direction Anomalies Across Retained Frameworks",
    barmode="stack",
    width=930,
    height=500,
    margin=dict(l=50, r=30, t=70, b=60),
    yaxis_title="Framework-row anomalies",
    xaxis_title="Framework",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.25,
        xanchor="center",
        x=0.5,
    ),
)

fig.show()

,feature_set,framework_label,retained_framework_anomaly_rows,congestion_only_rows,positive_demand_only_rows,both_positive_rows,positive_direction_total_rows,positive_direction_total_share,ambiguous_rows
0,shared,DBSCAN,911656,294576,187084,0,481660,0.528,14067
1,shared,GMM,358159,111705,95902,0,207607,0.580,248
2,shared,Isolation Forest,596930,177983,135174,0,313157,0.525,2792


Findings\. Positive\-direction anomalies make up a substantial share of every retained framework surface\. DBSCAN and Isolation Forest both place just over half of their retained anomalies into the positive\-direction pool, while GMM is somewhat more concentrated there at 58\.0%\. Within that positive\-direction subset, congestion\-only anomalies are the dominant component in all three frameworks, and positive\-demand\-only anomalies form the second major layer\. No rows carry both positive families at once under this row\-level labeling contract, so the positive\-direction surface is cleanly partitioned between congestion\-oriented and demand\-oriented signals rather than mixing both on the same retained anomaly row\.

> 🎯 Scope\. Downstream review in this notebook family will focus on positive\-direction anomaly events: anomalously worse congestion and anomalously elevated travel demand\. Negative\-direction and ambiguous anomaly states remain useful for calibration and comparison, but they are not the downstream policy\-facing target\. The next subsection therefore compares retained event surfaces with special attention to positive\-direction yield, and the following subsection will narrow the selected working surface to its positive\-direction subset for export and deeper review\.

### Section Summary

This section adds a row\-level directionality layer on top of the retained anomaly frameworks\. It defines one explicit metric\-to\-direction rule map, assigns directional labels directly from each retained anomaly row’s metric family and signed residual z\-score, audits how completely the retained rows can be interpreted, and isolates the positive\-direction subset within each retained framework branch\.

The result is a clean interpretation layer, not yet a final anomaly\-surface decision\. DBSCAN, Isolation Forest, and GMM can now be compared not just on whether they flag the same rows, but on whether they are flagging the same kinds of mobility disruptions\. That gives the next section a much stronger base for overlap, consensus, and downstream\-surface selection at the shared retained anomaly grain\.

### 3\.3\.5\.3 Compare Framework Overlap and Consensus Behavior

This subsection converts the retained framework anomaly rows into a shared event\-level comparison surface at Taxi Zone × Date × Daypart grain\. It then compares a compact ladder of downstream candidate surfaces built from framework union and framework consensus so we can decide what retained anomaly package should move forward into deeper review and export preparation\.

The comparison is organized around event breadth, positive\-direction yield, directional composition, framework support, and modality\-driver diversity\. Venn\-style overlap remains an important interpretive tool here, but it serves as supporting evidence alongside a more decision\-ready event\-surface scorecard\.

### Build the Retained Event\-Surface Ladder

This subsection builds the retained comparison universe at Taxi Zone × Date × Daypart grain and turns the three selected framework outputs into a common event\-surface ladder\. It first establishes the shared event spine, then collapses retained anomaly rows into framework\-specific event evidence, attaches exact framework\-support flags back onto that spine, and finally defines the candidate retained surfaces that will be compared downstream\.

The result is a clean event\-level representation of the retained anomaly package\. Instead of repeatedly comparing framework rows at metric grain, the notebook now works with interpretable retained events, each carrying its own support pattern and metric\-driver context\. That creates a much clearer base for deciding whether the downstream retained surface should come from the full union, a multi\-framework consensus layer, or a narrower retained core\.

Build the shared event\-grain spine at Taxi Zone × Date × Daypart from the aligned retained framework comparison table\. This block creates the canonical event universe and preserves the shared event metadata that later blocks will attach framework support and metric\-driver evidence to\.

In [16]:
# ---------------------------------------------------------------------
# Build the shared retained event spine at Taxi Zone x Date x Daypart grain
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"

if should_rebuild(REBUILD_335_CONSENSUS, RETAINED_EVENT_SPINE_BASE_PATH):
    framework_comparison_row_level_df = pd.read_parquet(
        FRAMEWORK_COMPARISON_ROW_LEVEL_PATH
    ).copy()
    framework_comparison_row_level_df[DATE_COLUMN] = pd.to_datetime(
        framework_comparison_row_level_df[DATE_COLUMN],
        errors="coerce",
    )

    if EVENT_ID_COLUMN not in framework_comparison_row_level_df.columns:
        framework_comparison_row_level_df[EVENT_ID_COLUMN] = (
            framework_comparison_row_level_df[ZONE_ID_COLUMN].astype(str)
            + " | "
            + framework_comparison_row_level_df[DATE_COLUMN].dt.strftime("%Y-%m-%d")
            + " | "
            + framework_comparison_row_level_df[DAYPART_COLUMN].astype(str)
        )

    event_group_columns = [
        "feature_set",
        EVENT_ID_COLUMN,
        ZONE_ID_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
    ]

    aggregation_map = {
        DAYPART_ORDER_COLUMN: (DAYPART_ORDER_COLUMN, "min"),
    }

    if TEMPORAL_BUCKET_COLUMN in framework_comparison_row_level_df.columns:
        aggregation_map[TEMPORAL_BUCKET_COLUMN] = (TEMPORAL_BUCKET_COLUMN, "first")
    if TEMPORAL_BUCKET_ORDER_COLUMN in framework_comparison_row_level_df.columns:
        aggregation_map[TEMPORAL_BUCKET_ORDER_COLUMN] = (
            TEMPORAL_BUCKET_ORDER_COLUMN,
            "min",
        )
    if ZONE_COLUMN in framework_comparison_row_level_df.columns:
        aggregation_map[ZONE_COLUMN] = (ZONE_COLUMN, "first")
    if BOROUGH_COLUMN in framework_comparison_row_level_df.columns:
        aggregation_map[BOROUGH_COLUMN] = (BOROUGH_COLUMN, "first")
    if SUPPORT_REVIEW_COLUMN in framework_comparison_row_level_df.columns:
        aggregation_map[SUPPORT_REVIEW_COLUMN] = (SUPPORT_REVIEW_COLUMN, "max")

    retained_event_spine_df = (
        framework_comparison_row_level_df.groupby(event_group_columns, dropna=False)
        .agg(**aggregation_map)
        .reset_index()
        .sort_values([DATE_COLUMN, DAYPART_ORDER_COLUMN, ZONE_ID_COLUMN])
        .reset_index(drop=True)
    )

    if WRITE_335_OUTPUTS:
        retained_event_spine_df.to_parquet(
            RETAINED_EVENT_SPINE_BASE_PATH,
            index=False,
        )
else:
    retained_event_spine_df = pd.read_parquet(RETAINED_EVENT_SPINE_BASE_PATH).copy()
    retained_event_spine_df[DATE_COLUMN] = pd.to_datetime(
        retained_event_spine_df[DATE_COLUMN],
        errors="coerce",
    )

shared_event_spine_summary_df = pd.DataFrame(
    {
        "metric": [
            "full_event_universe_count",
            "distinct_taxi_zone_count",
            "distinct_date_count",
            "distinct_daypart_count",
        ],
        "value": [
            int(len(retained_event_spine_df)),
            int(retained_event_spine_df[ZONE_ID_COLUMN].nunique()),
            int(retained_event_spine_df[DATE_COLUMN].nunique()),
            int(retained_event_spine_df[DAYPART_COLUMN].nunique()),
        ],
    }
)

display(shared_event_spine_summary_df)
print(f"Collapsed event-universe rows: {len(retained_event_spine_df):,}")

,metric,value
0,full_event_universe_count,1559590
1,distinct_taxi_zone_count,263
2,distinct_date_count,1186
3,distinct_daypart_count,5


Collapsed event-universe rows: 1,559,590


Findings\. The collapsed comparison universe contains 1,559,590 distinct Taxi Zone × Date × Daypart events across the full study window, covering all 263 Taxi Zones, 1,186 study dates, and all 5 dayparts\. This confirms that the retained\-framework synthesis is now operating on one complete citywide event spine rather than repeated metric rows, which gives the next blocks a clean and interpretable grain for comparing union, consensus, and downstream retained\-surface candidates\.

Collapse the retained directional anomaly rows to Taxi Zone × Date × Daypart within each framework\. This produces the framework\-specific event evidence table that records metric counts, modality counts, and directional composition before any cross\-framework merge happens\.

In [17]:
# ---------------------------------------------------------------------
# Collapse retained anomaly rows into framework-specific event evidence
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"

if should_rebuild(REBUILD_335_CONSENSUS, FRAMEWORK_EVENT_SUMMARY_PATH):
    retained_anomaly_directionality_df = pd.read_parquet(
        DIRECTIONAL_ROW_LEVEL_PATH
    ).copy()
    retained_anomaly_directionality_df[DATE_COLUMN] = pd.to_datetime(
        retained_anomaly_directionality_df[DATE_COLUMN],
        errors="coerce",
    )

    if EVENT_ID_COLUMN not in retained_anomaly_directionality_df.columns:
        retained_anomaly_directionality_df[EVENT_ID_COLUMN] = (
            retained_anomaly_directionality_df[ZONE_ID_COLUMN].astype(str)
            + " | "
            + retained_anomaly_directionality_df[DATE_COLUMN].dt.strftime("%Y-%m-%d")
            + " | "
            + retained_anomaly_directionality_df[DAYPART_COLUMN].astype(str)
        )

    modality_lookup = {
        "avg_bus_speed": "Bus",
        "bus_trip_count": "Bus",
        "fhvhv_avg_trip_duration": "FHVHV",
        "fhvhv_avg_trip_speed": "FHVHV",
        "fhvhv_trip_count": "FHVHV",
        "subway_ridership": "Subway",
        "subway_transfers": "Subway",
        "taxi_avg_trip_duration": "Taxi",
        "taxi_avg_trip_speed": "Taxi",
        "taxi_trip_count": "Taxi",
    }

    retained_anomaly_directionality_df["modality_driver"] = (
        retained_anomaly_directionality_df[METRIC_COLUMN]
        .map(modality_lookup)
        .fillna("Other")
    )

    event_group_columns = [
        "feature_set",
        "framework_name",
        "framework_label",
        EVENT_ID_COLUMN,
        ZONE_ID_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
    ]

    framework_event_summary_df = (
        retained_anomaly_directionality_df.groupby(event_group_columns, dropna=False)
        .agg(
            anomaly_row_count=(METRIC_COLUMN, "size"),
            distinct_metric_count=(METRIC_COLUMN, pd.Series.nunique),
            distinct_modality_count=("modality_driver", pd.Series.nunique),
            metric_driver_list=(
                METRIC_COLUMN,
                lambda s: ", ".join(sorted(pd.Series(s).dropna().astype(str).unique())),
            ),
            modality_driver_list=(
                "modality_driver",
                lambda s: ", ".join(sorted(pd.Series(s).dropna().astype(str).unique())),
            ),
        )
        .reset_index()
        .sort_values(["framework_label", DATE_COLUMN, DAYPART_COLUMN, ZONE_ID_COLUMN])
        .reset_index(drop=True)
    )

    if WRITE_335_OUTPUTS:
        framework_event_summary_df.to_parquet(
            FRAMEWORK_EVENT_SUMMARY_PATH,
            index=False,
        )
        framework_event_summary_df.to_parquet(
            MODALITY_PARTICIPATION_PATH,
            index=False,
        )
else:
    framework_event_summary_df = pd.read_parquet(
        FRAMEWORK_EVENT_SUMMARY_PATH
    ).copy()
    framework_event_summary_df[DATE_COLUMN] = pd.to_datetime(
        framework_event_summary_df[DATE_COLUMN],
        errors="coerce",
    )

display(
    framework_event_summary_df[
        [
            "framework_label",
            EVENT_ID_COLUMN,
            ZONE_ID_COLUMN,
            DATE_COLUMN,
            DAYPART_COLUMN,
            "anomaly_row_count",
            "distinct_metric_count",
            "distinct_modality_count",
            "metric_driver_list",
            "modality_driver_list",
        ]
    ].head(10)
)
print(f"Framework-event summary rows: {len(framework_event_summary_df):,}")

,framework_label,comparison_event_id,taxi_zone_id,date,daypart,anomaly_row_count,distinct_metric_count,distinct_modality_count,metric_driver_list,modality_driver_list
0,DBSCAN,4 | 2023-01-01 | am_peak,4,2023-01-01,am_peak,1,1,1,fhvhv_avg_trip_speed,FHVHV
1,DBSCAN,7 | 2023-01-01 | am_peak,7,2023-01-01,am_peak,1,1,1,fhvhv_avg_trip_speed,FHVHV
2,DBSCAN,10 | 2023-01-01 | am_peak,10,2023-01-01,am_peak,1,1,1,taxi_avg_trip_speed,Taxi
3,DBSCAN,14 | 2023-01-01 | am_peak,14,2023-01-01,am_peak,1,1,1,fhvhv_avg_trip_speed,FHVHV
4,DBSCAN,19 | 2023-01-01 | am_peak,19,2023-01-01,am_peak,2,2,2,"fhvhv_avg_trip_speed, taxi_avg_trip_speed","FHVHV, Taxi"
5,DBSCAN,20 | 2023-01-01 | am_peak,20,2023-01-01,am_peak,1,1,1,fhvhv_trip_count,FHVHV
6,DBSCAN,21 | 2023-01-01 | am_peak,21,2023-01-01,am_peak,1,1,1,fhvhv_avg_trip_speed,FHVHV
7,DBSCAN,24 | 2023-01-01 | am_peak,24,2023-01-01,am_peak,1,1,1,fhvhv_avg_trip_speed,FHVHV
8,DBSCAN,25 | 2023-01-01 | am_peak,25,2023-01-01,am_peak,1,1,1,fhvhv_avg_trip_speed,FHVHV
9,DBSCAN,26 | 2023-01-01 | am_peak,26,2023-01-01,am_peak,1,1,1,fhvhv_avg_trip_duration,FHVHV


Framework-event summary rows: 1,406,094


Findings\. The retained anomaly rows now collapse cleanly to framework\-specific event evidence at Taxi Zone × Date × Daypart grain, while preserving the metric and modality drivers attached to each event\. Even in the early preview rows, the event table is already doing the intended job: some retained events are supported by a single anomalous metric and modality, while others carry multiple anomalous metrics and cross\-modality evidence\. That gives the overlap section a much stronger foundation, because the downstream comparison is no longer just counting retained rows, but comparing interpretable retained events with explicit driver context\.

Convert the framework\-specific event evidence into framework support flags and attach those flags to the shared event spine\. This is the cross\-framework alignment step at event grain\.

In [18]:
# ---------------------------------------------------------------------
# Attach framework support flags onto the shared event spine
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"

retained_event_spine_df = pd.read_parquet(RETAINED_EVENT_SPINE_BASE_PATH).copy()
framework_event_summary_df = pd.read_parquet(FRAMEWORK_EVENT_SUMMARY_PATH).copy()

retained_event_spine_df[DATE_COLUMN] = pd.to_datetime(
    retained_event_spine_df[DATE_COLUMN],
    errors="coerce",
)
framework_event_summary_df[DATE_COLUMN] = pd.to_datetime(
    framework_event_summary_df[DATE_COLUMN],
    errors="coerce",
)

stale_support_columns = [
    "dbscan_event_flag",
    "if_event_flag",
    "gmm_event_flag",
    "framework_anomaly_count",
    "any_framework_event_flag",
    "framework_support_pattern",
]
retained_event_spine_df = retained_event_spine_df.drop(
    columns=[
        column_name
        for column_name in stale_support_columns
        if column_name in retained_event_spine_df.columns
    ],
    errors="ignore",
)

framework_event_flag_wide_df = (
    framework_event_summary_df.assign(event_flag=True)
    .pivot_table(
        index=[
            "feature_set",
            EVENT_ID_COLUMN,
            ZONE_ID_COLUMN,
            DATE_COLUMN,
            DAYPART_COLUMN,
        ],
        columns="framework_name",
        values="event_flag",
        aggfunc="max",
        fill_value=False,
    )
    .reset_index()
)
framework_event_flag_wide_df.columns.name = None

for framework_name in FRAMEWORK_NAMES:
    if framework_name not in framework_event_flag_wide_df.columns:
        framework_event_flag_wide_df[framework_name] = False

framework_event_flag_wide_df = framework_event_flag_wide_df.rename(
    columns={
        "dbscan": "dbscan_event_flag",
        "isolation_forest": "if_event_flag",
        "gmm": "gmm_event_flag",
    }
)

retained_event_spine_df = retained_event_spine_df.merge(
    framework_event_flag_wide_df,
    on=[
        "feature_set",
        EVENT_ID_COLUMN,
        ZONE_ID_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
    ],
    how="left",
    validate="one_to_one",
)

event_flag_columns = ["dbscan_event_flag", "if_event_flag", "gmm_event_flag"]
for event_flag_column in event_flag_columns:
    retained_event_spine_df[event_flag_column] = (
        retained_event_spine_df[event_flag_column].fillna(False).astype(bool)
    )

retained_event_spine_df["framework_anomaly_count"] = (
    retained_event_spine_df[event_flag_columns].sum(axis=1).astype(int)
)
retained_event_spine_df["any_framework_event_flag"] = (
    retained_event_spine_df["framework_anomaly_count"].gt(0)
)

def classify_support_pattern(row):
    dbscan_flag = bool(row["dbscan_event_flag"])
    if_flag = bool(row["if_event_flag"])
    gmm_flag = bool(row["gmm_event_flag"])

    if dbscan_flag and if_flag and gmm_flag:
        return "DBSCAN + Isolation Forest + GMM"
    if dbscan_flag and if_flag:
        return "DBSCAN + Isolation Forest"
    if dbscan_flag and gmm_flag:
        return "DBSCAN + GMM"
    if if_flag and gmm_flag:
        return "Isolation Forest + GMM"
    if dbscan_flag:
        return "DBSCAN only"
    if if_flag:
        return "Isolation Forest only"
    if gmm_flag:
        return "GMM only"
    return "No retained anomaly"

retained_event_spine_df["framework_support_pattern"] = retained_event_spine_df.apply(
    classify_support_pattern,
    axis=1,
)

if WRITE_335_OUTPUTS:
    retained_event_spine_df.to_parquet(CONSENSUS_ROW_LEVEL_PATH, index=False)
    framework_event_summary_df.to_parquet(MODALITY_PARTICIPATION_PATH, index=False)

display(
    retained_event_spine_df[
        [
            EVENT_ID_COLUMN,
            ZONE_ID_COLUMN,
            DATE_COLUMN,
            DAYPART_COLUMN,
            "dbscan_event_flag",
            "if_event_flag",
            "gmm_event_flag",
            "framework_anomaly_count",
            "framework_support_pattern",
        ]
    ]
    .loc[retained_event_spine_df["any_framework_event_flag"]]
    .head(10)
)
print(
    "Retained event rows across any framework: "
    f"{int(retained_event_spine_df['any_framework_event_flag'].sum()):,}"
)

,comparison_event_id,taxi_zone_id,date,daypart,dbscan_event_flag,if_event_flag,gmm_event_flag,framework_anomaly_count,framework_support_pattern
3,4 | 2023-01-01 | overnight,4,2023-01-01,overnight,True,True,True,3,DBSCAN + Isolation Forest + GMM
6,7 | 2023-01-01 | overnight,7,2023-01-01,overnight,True,True,True,3,DBSCAN + Isolation Forest + GMM
9,10 | 2023-01-01 | overnight,10,2023-01-01,overnight,True,True,True,3,DBSCAN + Isolation Forest + GMM
12,13 | 2023-01-01 | overnight,13,2023-01-01,overnight,True,True,True,3,DBSCAN + Isolation Forest + GMM
13,14 | 2023-01-01 | overnight,14,2023-01-01,overnight,True,True,True,3,DBSCAN + Isolation Forest + GMM
15,16 | 2023-01-01 | overnight,16,2023-01-01,overnight,True,True,True,3,DBSCAN + Isolation Forest + GMM
16,17 | 2023-01-01 | overnight,17,2023-01-01,overnight,True,True,True,3,DBSCAN + Isolation Forest + GMM
17,18 | 2023-01-01 | overnight,18,2023-01-01,overnight,True,True,True,3,DBSCAN + Isolation Forest + GMM
19,20 | 2023-01-01 | overnight,20,2023-01-01,overnight,True,True,True,3,DBSCAN + Isolation Forest + GMM
20,21 | 2023-01-01 | overnight,21,2023-01-01,overnight,True,True,True,3,DBSCAN + Isolation Forest + GMM


Retained event rows across any framework: 715,321


Findings\. The framework\-specific event evidence attaches cleanly back onto the shared event spine, and the resulting support\-pattern table confirms that the event\-level comparison is working as intended\. The preview shows that the strongest early events are indeed supported by all three retained frameworks at once, which is exactly the kind of event\-level consensus signal we want to preserve for downstream selection\. At this point, each event now carries both its place in the shared citywide event universe and its exact retained\-framework support pattern\.

Summarize how retained events are distributed across framework support patterns\. This gives a compact view of how much of the retained event universe is supported by one, two, or all three frameworks before we start scoring downstream candidate surfaces\.

In [19]:
# ---------------------------------------------------------------------
# Summarize retained event support coverage
# ---------------------------------------------------------------------

framework_event_coverage_df = (
    retained_event_spine_df.loc[
        retained_event_spine_df["any_framework_event_flag"]
    ]
    .groupby("framework_support_pattern", dropna=False)
    .agg(event_count=(EVENT_ID_COLUMN, "size"))
    .reset_index()
    .sort_values("event_count", ascending=False)
    .reset_index(drop=True)
)

retained_event_support_summary_df = pd.DataFrame(
    [
        {
            "metric": "retained_event_count_any_framework",
            "value": int(retained_event_spine_df["any_framework_event_flag"].sum()),
        },
        {
            "metric": "retained_event_count_dbscan",
            "value": int(retained_event_spine_df["dbscan_event_flag"].sum()),
        },
        {
            "metric": "retained_event_count_if",
            "value": int(retained_event_spine_df["if_event_flag"].sum()),
        },
        {
            "metric": "retained_event_count_gmm",
            "value": int(retained_event_spine_df["gmm_event_flag"].sum()),
        },
        {
            "metric": "framework_event_summary_rows",
            "value": int(len(framework_event_summary_df)),
        },
    ]
)

framework_event_coverage_df["event_share_of_retained_events"] = (
    framework_event_coverage_df["event_count"]
    / framework_event_coverage_df["event_count"].sum()
)

display(retained_event_support_summary_df)
display(
    framework_event_coverage_df.style.format(
        {"event_share_of_retained_events": "{:.3f}"}
    )
)

,metric,value
0,retained_event_count_any_framework,715321
1,retained_event_count_dbscan,655997
2,retained_event_count_if,445715
3,retained_event_count_gmm,304382
4,framework_event_summary_rows,1406094


,framework_support_pattern,event_count,event_share_of_retained_events
0,DBSCAN + Isolation Forest + GMM,225524,0.315
1,DBSCAN only,192874,0.270
2,DBSCAN + Isolation Forest,168321,0.235
3,DBSCAN + GMM,69278,0.097
4,Isolation Forest only,49744,0.070
5,GMM only,7454,0.010
6,Isolation Forest + GMM,2126,0.003


Findings\. The retained event universe contains 715,321 distinct anomaly events across the three selected frameworks\. DBSCAN contributes the broadest event surface at 655,997 events, Isolation Forest contributes 445,715, and GMM contributes 304,382\. In the overlap structure, the largest single region is the strict three\-framework core with 225,524 events, or 31\.5% of all retained events, followed by DBSCAN\-only events at 27\.0% and DBSCAN\-plus\-Isolation\-Forest events at 23\.5%\. That means the retained comparison surface is neither dominated by pure consensus nor fragmented into mostly framework\-specific tails; instead, it contains a meaningful shared core plus substantial incremental layers that still need to be judged on quality\.

Translate the retained framework event flags into a compact ladder of candidate downstream surfaces\. Each candidate is defined at the same event grain, which makes it possible to compare broader and narrower retained anomaly packages side by side before deciding what should move forward\.

In [20]:
# ---------------------------------------------------------------------
# Define candidate retained event surfaces and preview their breadth
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"

retained_event_spine_df = pd.read_parquet(CONSENSUS_ROW_LEVEL_PATH).copy()

candidate_surface_specs_df = pd.DataFrame(
    [
        {
            "candidate_surface_id": "framework_union",
            "candidate_surface_label": "Any retained framework",
            "candidate_group": "broad",
            "framework_scope": "all",
        },
        {
            "candidate_surface_id": "framework_consensus_2plus",
            "candidate_surface_label": "2+ retained frameworks",
            "candidate_group": "consensus",
            "framework_scope": "all",
        },
        {
            "candidate_surface_id": "framework_consensus_3way",
            "candidate_surface_label": "All 3 retained frameworks",
            "candidate_group": "consensus",
            "framework_scope": "all",
        },
        {
            "candidate_surface_id": "dbscan_surface",
            "candidate_surface_label": "DBSCAN retained surface",
            "candidate_group": "framework_reference",
            "framework_scope": "dbscan",
        },
        {
            "candidate_surface_id": "if_surface",
            "candidate_surface_label": "Isolation Forest retained surface",
            "candidate_group": "framework_reference",
            "framework_scope": "isolation_forest",
        },
        {
            "candidate_surface_id": "gmm_surface",
            "candidate_surface_label": "GMM retained surface",
            "candidate_group": "framework_reference",
            "framework_scope": "gmm",
        },
    ]
)

candidate_event_surface_frames = []

for candidate_row in candidate_surface_specs_df.itertuples(index=False):
    if candidate_row.candidate_surface_id == "framework_union":
        candidate_mask = retained_event_spine_df["any_framework_event_flag"]
    elif candidate_row.candidate_surface_id == "framework_consensus_2plus":
        candidate_mask = retained_event_spine_df["framework_anomaly_count"].ge(2)
    elif candidate_row.candidate_surface_id == "framework_consensus_3way":
        candidate_mask = retained_event_spine_df["framework_anomaly_count"].eq(3)
    elif candidate_row.candidate_surface_id == "dbscan_surface":
        candidate_mask = retained_event_spine_df["dbscan_event_flag"]
    elif candidate_row.candidate_surface_id == "if_surface":
        candidate_mask = retained_event_spine_df["if_event_flag"]
    elif candidate_row.candidate_surface_id == "gmm_surface":
        candidate_mask = retained_event_spine_df["gmm_event_flag"]
    else:
        continue

    candidate_surface_df = retained_event_spine_df.loc[candidate_mask].copy()
    candidate_surface_df["candidate_surface_id"] = candidate_row.candidate_surface_id
    candidate_surface_df["candidate_surface_label"] = candidate_row.candidate_surface_label
    candidate_surface_df["candidate_group"] = candidate_row.candidate_group
    candidate_surface_df["framework_scope"] = candidate_row.framework_scope

    candidate_event_surface_frames.append(candidate_surface_df)

candidate_event_surface_df = pd.concat(
    candidate_event_surface_frames,
    ignore_index=True,
)

candidate_event_surface_summary_df = (
    candidate_event_surface_df.groupby(
        [
            "candidate_surface_id",
            "candidate_surface_label",
            "candidate_group",
            "framework_scope",
        ],
        dropna=False,
    )
    .agg(
        event_count=(EVENT_ID_COLUMN, "size"),
        anomalous_taxi_zone_count=(ZONE_ID_COLUMN, "nunique"),
        anomalous_date_count=(DATE_COLUMN, "nunique"),
        anomalous_daypart_count=(DAYPART_COLUMN, "nunique"),
        one_framework_support_share=(
            "framework_anomaly_count",
            lambda s: float(pd.Series(s).eq(1).mean()),
        ),
        two_framework_support_share=(
            "framework_anomaly_count",
            lambda s: float(pd.Series(s).eq(2).mean()),
        ),
        three_framework_support_share=(
            "framework_anomaly_count",
            lambda s: float(pd.Series(s).eq(3).mean()),
        ),
    )
    .reset_index()
)

candidate_event_surface_summary_df["event_share_of_full_event_universe"] = (
    candidate_event_surface_summary_df["event_count"]
    / len(retained_event_spine_df)
)

candidate_label_order = candidate_surface_specs_df["candidate_surface_label"].tolist()
candidate_event_surface_summary_df["candidate_surface_label"] = pd.Categorical(
    candidate_event_surface_summary_df["candidate_surface_label"],
    categories=candidate_label_order,
    ordered=True,
)
candidate_event_surface_summary_df = candidate_event_surface_summary_df.sort_values(
    "candidate_surface_label"
).reset_index(drop=True)

if WRITE_335_OUTPUTS:
    candidate_event_surface_df.to_parquet(CONSENSUS_SUMMARY_PATH, index=False)

display(candidate_surface_specs_df)
display(
    candidate_event_surface_summary_df.style.format(
        {
            "event_share_of_full_event_universe": "{:.3f}",
            "one_framework_support_share": "{:.3f}",
            "two_framework_support_share": "{:.3f}",
            "three_framework_support_share": "{:.3f}",
        }
    )
)

print(f"Candidate retained surfaces: {candidate_surface_specs_df.shape[0]:,}")
print(f"Candidate-event rows materialized: {len(candidate_event_surface_df):,}")

,candidate_surface_id,candidate_surface_label,candidate_group,framework_scope
0,framework_union,Any retained framework,broad,all
1,framework_consensus_2plus,2+ retained frameworks,consensus,all
2,framework_consensus_3way,All 3 retained frameworks,consensus,all
3,dbscan_surface,DBSCAN retained surface,framework_reference,dbscan
4,if_surface,Isolation Forest retained surface,framework_reference,isolation_forest
5,gmm_surface,GMM retained surface,framework_reference,gmm


,candidate_surface_id,candidate_surface_label,candidate_group,framework_scope,event_count,anomalous_taxi_zone_count,anomalous_date_count,anomalous_daypart_count,one_framework_support_share,two_framework_support_share,three_framework_support_share,event_share_of_full_event_universe
0,framework_union,Any retained framework,broad,all,715321,263,1186,5,0.350,0.335,0.315,0.459
1,framework_consensus_2plus,2+ retained frameworks,consensus,all,465249,263,1186,5,0.000,0.515,0.485,0.298
2,framework_consensus_3way,All 3 retained frameworks,consensus,all,225524,262,1186,5,0.000,0.000,1.000,0.145
3,dbscan_surface,DBSCAN retained surface,framework_reference,dbscan,655997,263,1186,5,0.294,0.362,0.344,0.421
4,if_surface,Isolation Forest retained surface,framework_reference,isolation_forest,445715,263,1186,5,0.112,0.382,0.506,0.286
5,gmm_surface,GMM retained surface,framework_reference,gmm,304382,262,1186,5,0.024,0.235,0.741,0.195


Candidate retained surfaces: 6
Candidate-event rows materialized: 2,812,188


Findings\. The candidate retained surfaces now span a clear breadth ladder\. The full retained union covers 715,321 events, or 45\.9% of the full event universe, while the 2\+ retained frameworks surface narrows that to 465,249 events, or 29\.8%, and the strict all 3 retained frameworks core narrows further to 225,524 events, or 14\.5%\. All three consensus\-style candidates still cover essentially the full citywide geography and full study\-period date range, so the main tradeoff here is not whether the narrower surfaces become geographically trivial, but how much event mass and framework support we want to carry forward\.

### Evaluate Retained Event\-Surface Tradeoffs

This subsection compares the retained event surfaces on the dimensions that matter most for downstream selection\. It scores each candidate on breadth, positive\-direction yield, framework support, and modality\-driver diversity, then measures what is actually gained as the retained surface widens from the strict three\-framework core to two\-plus consensus and finally to the full retained union\.

The goal here is to make the retained\-surface tradeoff explicit\. A broader candidate can preserve more event mass, while a narrower candidate can preserve a cleaner, more consensus\-heavy anomaly profile\. By organizing that tradeoff in one place, the notebook can move toward a retained comparison\-surface decision that is grounded in event coverage, interpretability, and incremental signal quality rather than in overlap alone\.

Score each retained candidate surface in one event\-level comparison table\. The goal here is to compare how much event coverage each candidate preserves, how much positive\-direction signal it carries, how broadly it spans modalities and metrics, and how much of it is supported by one, two, or all three retained frameworks\.

In [21]:
# ---------------------------------------------------------------------
# Score retained event surfaces on breadth, directionality,
# support, and modality-driver diversity
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"
CANDIDATE_SURFACE_SCORECARD_PATH = (
    INTERMEDIATE_335_DIR / "candidate_event_surface_scorecard.parquet"
)

def build_event_id_from_frame(df):
    date_token = pd.to_datetime(df[DATE_COLUMN], errors="coerce").dt.strftime("%Y-%m-%d")
    return (
        df[ZONE_ID_COLUMN].astype(str)
        + " | "
        + date_token.fillna("missing")
        + " | "
        + df[DAYPART_COLUMN].astype(str)
    )

def classify_metric_modality(metric_name):
    metric_name = str(metric_name)
    if metric_name.startswith("bus_") or metric_name == "avg_bus_speed":
        return "Bus"
    if metric_name.startswith("subway_"):
        return "Subway"
    if metric_name.startswith("taxi_"):
        return "Taxi"
    if metric_name.startswith("fhvhv_"):
        return "FHVHV"
    return "Other"

candidate_event_surface_df = pd.read_parquet(CONSENSUS_SUMMARY_PATH).copy()
retained_event_spine_df = pd.read_parquet(CONSENSUS_ROW_LEVEL_PATH).copy()

retained_directional_scoring_columns = [
    "framework_name",
    "framework_label",
    "feature_set",
    ZONE_ID_COLUMN,
    DATE_COLUMN,
    DAYPART_COLUMN,
    METRIC_COLUMN,
    "directional_label",
    "congestion_oriented",
    "decongestion_oriented",
    "positive_demand_shock",
    "negative_demand_shock",
    "anomaly_flag",
]

retained_anomaly_directionality_df = load_parquet_without_duplicate_columns(
    DIRECTIONAL_ROW_LEVEL_PATH,
    columns=retained_directional_scoring_columns,
).copy()

retained_anomaly_directionality_df = retained_anomaly_directionality_df.loc[
    retained_anomaly_directionality_df["anomaly_flag"].fillna(False)
].copy()

retained_anomaly_directionality_df[DATE_COLUMN] = pd.to_datetime(
    retained_anomaly_directionality_df[DATE_COLUMN],
    errors="coerce",
)
retained_anomaly_directionality_df[EVENT_ID_COLUMN] = build_event_id_from_frame(
    retained_anomaly_directionality_df
)
retained_anomaly_directionality_df["modality"] = (
    retained_anomaly_directionality_df[METRIC_COLUMN]
    .astype(str)
    .map(classify_metric_modality)
)

for indicator_column in [
    "congestion_oriented",
    "positive_demand_shock",
]:
    retained_anomaly_directionality_df[indicator_column] = (
        pd.to_numeric(
            retained_anomaly_directionality_df[indicator_column],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )

framework_specific_surface_map = {
    "dbscan_surface": "dbscan",
    "if_surface": "isolation_forest",
    "gmm_surface": "gmm",
}

candidate_surface_score_records = []

for candidate_row in (
    candidate_event_surface_df[
        [
            "candidate_surface_id",
            "candidate_surface_label",
            "candidate_group",
            "framework_scope",
        ]
    ]
    .drop_duplicates()
    .itertuples(index=False)
):
    candidate_events_df = candidate_event_surface_df.loc[
        candidate_event_surface_df["candidate_surface_id"].eq(candidate_row.candidate_surface_id)
    ].copy()

    if candidate_row.candidate_surface_id in framework_specific_surface_map:
        candidate_rows_df = retained_anomaly_directionality_df.loc[
            retained_anomaly_directionality_df["framework_name"].eq(
                framework_specific_surface_map[candidate_row.candidate_surface_id]
            )
        ].merge(
            candidate_events_df[[EVENT_ID_COLUMN]],
            on=EVENT_ID_COLUMN,
            how="inner",
            validate="many_to_one",
        )
    else:
        candidate_rows_df = retained_anomaly_directionality_df.merge(
            candidate_events_df[[EVENT_ID_COLUMN]],
            on=EVENT_ID_COLUMN,
            how="inner",
            validate="many_to_one",
        )

    candidate_rows_df["has_congestion_signal"] = (
        candidate_rows_df["congestion_oriented"].fillna(0).gt(0)
    )
    candidate_rows_df["has_positive_demand_signal"] = (
        candidate_rows_df["positive_demand_shock"].fillna(0).gt(0)
    )
    candidate_rows_df["is_ambiguous_row"] = (
        candidate_rows_df["directional_label"].fillna("").astype(str).eq("ambiguous")
    )

    candidate_event_detail_df = (
        candidate_rows_df.groupby(
            [EVENT_ID_COLUMN, ZONE_ID_COLUMN, DATE_COLUMN, DAYPART_COLUMN],
            as_index=False,
            dropna=False,
        )
        .agg(
            candidate_row_count=(METRIC_COLUMN, "size"),
            distinct_metric_count=(METRIC_COLUMN, "nunique"),
            distinct_modality_count=("modality", "nunique"),
            has_congestion_signal=("has_congestion_signal", "max"),
            has_positive_demand_signal=("has_positive_demand_signal", "max"),
            ambiguous_row_count=("is_ambiguous_row", "sum"),
        )
    )

    candidate_event_detail_df["has_any_positive_signal"] = (
        candidate_event_detail_df["has_congestion_signal"]
        | candidate_event_detail_df["has_positive_demand_signal"]
    )
    candidate_event_detail_df["has_both_positive_families"] = (
        candidate_event_detail_df["has_congestion_signal"]
        & candidate_event_detail_df["has_positive_demand_signal"]
    )
    candidate_event_detail_df["ambiguous_only_event"] = (
        ~candidate_event_detail_df["has_congestion_signal"]
        & ~candidate_event_detail_df["has_positive_demand_signal"]
    )

    candidate_surface_score_records.append(
        {
            "candidate_surface_id": candidate_row.candidate_surface_id,
            "candidate_surface_label": candidate_row.candidate_surface_label,
            "candidate_group": candidate_row.candidate_group,
            "framework_scope": candidate_row.framework_scope,
            "event_count": int(len(candidate_events_df)),
            "event_share_of_full_event_universe": safe_share(
                len(candidate_events_df),
                len(retained_event_spine_df),
            ),
            "positive_direction_event_count": int(
                candidate_event_detail_df["has_any_positive_signal"].sum()
            ),
            "positive_direction_event_share": float(
                candidate_event_detail_df["has_any_positive_signal"].mean()
            ),
            "congestion_event_share": float(
                candidate_event_detail_df["has_congestion_signal"].mean()
            ),
            "positive_demand_event_share": float(
                candidate_event_detail_df["has_positive_demand_signal"].mean()
            ),
            "both_positive_family_event_share": float(
                candidate_event_detail_df["has_both_positive_families"].mean()
            ),
            "ambiguous_only_event_share": float(
                candidate_event_detail_df["ambiguous_only_event"].mean()
            ),
            "median_distinct_metric_count": float(
                candidate_event_detail_df["distinct_metric_count"].median()
            ),
            "p90_distinct_metric_count": float(
                candidate_event_detail_df["distinct_metric_count"].quantile(0.90)
            ),
            "median_distinct_modality_count": float(
                candidate_event_detail_df["distinct_modality_count"].median()
            ),
            "p90_distinct_modality_count": float(
                candidate_event_detail_df["distinct_modality_count"].quantile(0.90)
            ),
            "single_framework_support_share": float(
                candidate_events_df["framework_anomaly_count"].eq(1).mean()
            ),
            "two_framework_support_share": float(
                candidate_events_df["framework_anomaly_count"].eq(2).mean()
            ),
            "three_framework_support_share": float(
                candidate_events_df["framework_anomaly_count"].eq(3).mean()
            ),
            "anomalous_taxi_zone_count": int(candidate_events_df[ZONE_ID_COLUMN].nunique()),
            "anomalous_date_count": int(candidate_events_df[DATE_COLUMN].nunique()),
            "anomalous_daypart_count": int(candidate_events_df[DAYPART_COLUMN].nunique()),
        }
    )

candidate_event_surface_scorecard_df = pd.DataFrame(candidate_surface_score_records)

candidate_label_order = [
    "Any retained framework",
    "2+ retained frameworks",
    "All 3 retained frameworks",
    "DBSCAN retained surface",
    "Isolation Forest retained surface",
    "GMM retained surface",
]
candidate_event_surface_scorecard_df["candidate_surface_label"] = pd.Categorical(
    candidate_event_surface_scorecard_df["candidate_surface_label"],
    categories=candidate_label_order,
    ordered=True,
)
candidate_event_surface_scorecard_df = candidate_event_surface_scorecard_df.sort_values(
    "candidate_surface_label"
).reset_index(drop=True)

if WRITE_335_OUTPUTS:
    candidate_event_surface_scorecard_df.to_parquet(
        CANDIDATE_SURFACE_SCORECARD_PATH,
        index=False,
    )

display(
    candidate_event_surface_scorecard_df.style.format(
        {
            "event_share_of_full_event_universe": "{:.3f}",
            "positive_direction_event_share": "{:.3f}",
            "congestion_event_share": "{:.3f}",
            "positive_demand_event_share": "{:.3f}",
            "both_positive_family_event_share": "{:.3f}",
            "ambiguous_only_event_share": "{:.3f}",
            "single_framework_support_share": "{:.3f}",
            "two_framework_support_share": "{:.3f}",
            "three_framework_support_share": "{:.3f}",
            "median_distinct_metric_count": "{:.1f}",
            "p90_distinct_metric_count": "{:.1f}",
            "median_distinct_modality_count": "{:.1f}",
            "p90_distinct_modality_count": "{:.1f}",
        }
    )
)

,candidate_surface_id,candidate_surface_label,candidate_group,framework_scope,event_count,event_share_of_full_event_universe,positive_direction_event_count,positive_direction_event_share,congestion_event_share,positive_demand_event_share,both_positive_family_event_share,ambiguous_only_event_share,median_distinct_metric_count,p90_distinct_metric_count,median_distinct_modality_count,p90_distinct_modality_count,single_framework_support_share,two_framework_support_share,three_framework_support_share,anomalous_taxi_zone_count,anomalous_date_count,anomalous_daypart_count
0,framework_union,Any retained framework,broad,all,715321,0.459,423620,0.592,0.393,0.263,0.064,0.408,1.0,2.0,1.0,2.0,0.350,0.335,0.315,263,1186,5
1,framework_consensus_2plus,2+ retained frameworks,consensus,all,465249,0.298,299439,0.644,0.417,0.320,0.093,0.356,1.0,3.0,1.0,2.0,0.000,0.515,0.485,263,1186,5
2,framework_consensus_3way,All 3 retained frameworks,consensus,all,225524,0.145,155353,0.689,0.456,0.359,0.126,0.311,2.0,3.0,1.0,2.0,0.000,0.000,1.000,262,1186,5
3,dbscan_surface,DBSCAN retained surface,framework_reference,dbscan,655997,0.421,388889,0.593,0.398,0.254,0.059,0.407,1.0,2.0,1.0,2.0,0.294,0.362,0.344,263,1186,5
4,if_surface,Isolation Forest retained surface,framework_reference,isolation_forest,445715,0.286,255647,0.574,0.345,0.274,0.045,0.426,1.0,2.0,1.0,2.0,0.112,0.382,0.506,263,1186,5
5,gmm_surface,GMM retained surface,framework_reference,gmm,304382,0.195,184003,0.605,0.344,0.288,0.028,0.395,1.0,2.0,1.0,2.0,0.024,0.235,0.741,262,1186,5


Findings\. The event\-surface scorecard shows a real tradeoff between breadth and purity\. As the retained surface narrows from the union to 2\+ consensus and then to the strict 3\-way core, positive\-direction event share rises from 59\.2% to 64\.4% and then 68\.9%, while ambiguous\-only event share falls from 40\.8% to 35\.6% and then 31\.1%\. The narrower consensus surfaces therefore look directionally cleaner, but they do so by giving up a large amount of event coverage: 715,321 events in the union versus 465,249 in 2\+ consensus and 225,524 in the strict three\-way core\. Modality breadth does not materially improve as the surfaces narrow, so the main decision tension here is between carrying forward broader coverage and carrying forward a more consensus\-heavy, more positive\-direction surface\.

Measure what each widening step actually adds\. This block isolates the event layers introduced when the retained comparison surface expands from strict three\-framework consensus to two\-plus consensus and then to the full union, so we can evaluate whether added breadth is bringing meaningful positive\-direction and modality\-rich signal\.

In [22]:
# ---------------------------------------------------------------------
# Measure incremental contribution as the retained surface widens
# from 3-way consensus to 2+ consensus to full union
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"
INCREMENTAL_EVENT_LAYER_PATH = (
    INTERMEDIATE_335_DIR / "candidate_event_incremental_contribution.parquet"
)

def build_event_id_from_frame(df):
    date_token = pd.to_datetime(df[DATE_COLUMN], errors="coerce").dt.strftime("%Y-%m-%d")
    return (
        df[ZONE_ID_COLUMN].astype(str)
        + " | "
        + date_token.fillna("missing")
        + " | "
        + df[DAYPART_COLUMN].astype(str)
    )

def classify_metric_modality(metric_name):
    metric_name = str(metric_name)
    if metric_name.startswith("bus_") or metric_name == "avg_bus_speed":
        return "Bus"
    if metric_name.startswith("subway_"):
        return "Subway"
    if metric_name.startswith("taxi_"):
        return "Taxi"
    if metric_name.startswith("fhvhv_"):
        return "FHVHV"
    return "Other"

retained_event_spine_df = pd.read_parquet(CONSENSUS_ROW_LEVEL_PATH).copy()

retained_incremental_rows_df = load_parquet_without_duplicate_columns(
    DIRECTIONAL_ROW_LEVEL_PATH,
    columns=[
        "feature_set",
        "framework_name",
        ZONE_ID_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
        METRIC_COLUMN,
        "congestion_oriented",
        "positive_demand_shock",
        "directional_label",
        "anomaly_flag",
    ],
).copy()

retained_incremental_rows_df = retained_incremental_rows_df.loc[
    retained_incremental_rows_df["anomaly_flag"].fillna(False)
].copy()

retained_incremental_rows_df[DATE_COLUMN] = pd.to_datetime(
    retained_incremental_rows_df[DATE_COLUMN],
    errors="coerce",
)
retained_incremental_rows_df[EVENT_ID_COLUMN] = build_event_id_from_frame(
    retained_incremental_rows_df
)
retained_incremental_rows_df["modality"] = (
    retained_incremental_rows_df[METRIC_COLUMN]
    .astype(str)
    .map(classify_metric_modality)
)

for indicator_column in ["congestion_oriented", "positive_demand_shock"]:
    retained_incremental_rows_df[indicator_column] = (
        pd.to_numeric(
            retained_incremental_rows_df[indicator_column],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )

incremental_layer_specs = [
    {
        "layer_id": "all_three_framework_core",
        "layer_label": "All 3 retained frameworks",
        "mask": retained_event_spine_df["framework_anomaly_count"].eq(3),
    },
    {
        "layer_id": "added_by_two_plus_expansion",
        "layer_label": "Added by 2+ consensus expansion",
        "mask": retained_event_spine_df["framework_anomaly_count"].eq(2),
    },
    {
        "layer_id": "added_by_full_union_expansion",
        "layer_label": "Added by full-union expansion",
        "mask": retained_event_spine_df["framework_anomaly_count"].eq(1),
    },
]

incremental_event_layer_records = []

for layer_spec in incremental_layer_specs:
    layer_events_df = retained_event_spine_df.loc[layer_spec["mask"]].copy()

    layer_rows_df = retained_incremental_rows_df.merge(
        layer_events_df[[EVENT_ID_COLUMN]],
        on=EVENT_ID_COLUMN,
        how="inner",
        validate="many_to_one",
    )

    layer_rows_df["has_congestion_signal"] = (
        layer_rows_df["congestion_oriented"].fillna(0).gt(0)
    )
    layer_rows_df["has_positive_demand_signal"] = (
        layer_rows_df["positive_demand_shock"].fillna(0).gt(0)
    )

    layer_event_detail_df = (
        layer_rows_df.groupby(EVENT_ID_COLUMN, as_index=False, dropna=False)
        .agg(
            distinct_metric_count=(METRIC_COLUMN, "nunique"),
            distinct_modality_count=("modality", "nunique"),
            has_congestion_signal=("has_congestion_signal", "max"),
            has_positive_demand_signal=("has_positive_demand_signal", "max"),
        )
    )

    layer_event_detail_df["has_any_positive_signal"] = (
        layer_event_detail_df["has_congestion_signal"]
        | layer_event_detail_df["has_positive_demand_signal"]
    )

    incremental_event_layer_records.append(
        {
            "layer_id": layer_spec["layer_id"],
            "layer_label": layer_spec["layer_label"],
            "event_count": int(len(layer_events_df)),
            "event_share_of_full_event_universe": safe_share(
                len(layer_events_df),
                len(retained_event_spine_df),
            ),
            "positive_direction_event_share": float(
                layer_event_detail_df["has_any_positive_signal"].mean()
            ) if len(layer_event_detail_df) else np.nan,
            "congestion_event_share": float(
                layer_event_detail_df["has_congestion_signal"].mean()
            ) if len(layer_event_detail_df) else np.nan,
            "positive_demand_event_share": float(
                layer_event_detail_df["has_positive_demand_signal"].mean()
            ) if len(layer_event_detail_df) else np.nan,
            "median_distinct_metric_count": float(
                layer_event_detail_df["distinct_metric_count"].median()
            ) if len(layer_event_detail_df) else np.nan,
            "median_distinct_modality_count": float(
                layer_event_detail_df["distinct_modality_count"].median()
            ) if len(layer_event_detail_df) else np.nan,
        }
    )

incremental_event_layer_summary_df = pd.DataFrame(incremental_event_layer_records)

if WRITE_335_OUTPUTS:
    incremental_event_layer_summary_df.to_parquet(
        INCREMENTAL_EVENT_LAYER_PATH,
        index=False,
    )

display(
    incremental_event_layer_summary_df.style.format(
        {
            "event_share_of_full_event_universe": "{:.3f}",
            "positive_direction_event_share": "{:.3f}",
            "congestion_event_share": "{:.3f}",
            "positive_demand_event_share": "{:.3f}",
            "median_distinct_metric_count": "{:.1f}",
            "median_distinct_modality_count": "{:.1f}",
        }
    )
)

,layer_id,layer_label,event_count,event_share_of_full_event_universe,positive_direction_event_share,congestion_event_share,positive_demand_event_share,median_distinct_metric_count,median_distinct_modality_count
0,all_three_framework_core,All 3 retained frameworks,225524,0.145,0.689,0.456,0.359,2.0,1.0
1,added_by_two_plus_expansion,Added by 2+ consensus expansion,239725,0.154,0.601,0.380,0.282,1.0,1.0
2,added_by_full_union_expansion,Added by full-union expansion,250072,0.160,0.497,0.350,0.158,1.0,1.0


Findings\. The incremental\-quality view makes the widening tradeoff much clearer\. The strict three\-framework core is the strongest layer on positive\-direction share at 68\.9%, with the layer added by expanding to 2\+ consensus still looking reasonably strong at 60\.1%\. The final expansion from 2\+ consensus to the full union is noticeably weaker at 49\.7%, and its positive\-demand share drops especially sharply to 15\.8%\. That means the first widening step adds a substantial amount of event mass while still preserving a fairly coherent anomaly profile, whereas the final widening step adds even more events but with a much thinner positive\-direction yield\. This is exactly the kind of evidence that can justify preferring 2\+ retained frameworks over the full union if we want a better balance between breadth and interpretability\.

### Visualize Exact Overlap and Candidate\-Surface Composition

This subsection visualizes how the retained frameworks intersect at the event level and how that overlap translates into the candidate retained surfaces\. It first shows the exact three\-framework overlap structure for the retained event universe, then re\-expresses that same structure through candidate\-surface support composition and positive\-direction yield so the overlap picture can be read in directly decision\-relevant terms\.

These visuals are meant to support, not replace, the event\-surface scorecards\. The exact\-overlap views make the retained structure intuitive, while the candidate\-surface composition views show how much of each downstream option is supported by one, two, or all three frameworks\.

Build exact three\-framework overlap regions at the retained event grain\. This block turns the retained framework event flags into mutually exclusive overlap regions so later visuals can show exactly how much of the retained event universe is shared by all three frameworks, shared by each pair, or unique to one framework\.

In [23]:
# ---------------------------------------------------------------------
# Build exact three-framework overlap regions at the retained event grain
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"
EVENT_OVERLAP_REGIONS_PATH = (
    INTERMEDIATE_335_DIR / "event_overlap_regions.parquet"
)

retained_event_spine_df = pd.read_parquet(CONSENSUS_ROW_LEVEL_PATH).copy()

retained_event_overlap_df = retained_event_spine_df.loc[
    retained_event_spine_df["any_framework_event_flag"]
].copy()

def classify_exact_overlap_region(row):
    dbscan_flag = bool(row["dbscan_event_flag"])
    if_flag = bool(row["if_event_flag"])
    gmm_flag = bool(row["gmm_event_flag"])

    if dbscan_flag and if_flag and gmm_flag:
        return "DBSCAN ∩ IF ∩ GMM"
    if dbscan_flag and if_flag:
        return "DBSCAN ∩ IF only"
    if dbscan_flag and gmm_flag:
        return "DBSCAN ∩ GMM only"
    if if_flag and gmm_flag:
        return "IF ∩ GMM only"
    if dbscan_flag:
        return "DBSCAN only"
    if if_flag:
        return "IF only"
    if gmm_flag:
        return "GMM only"
    return "No retained anomaly"

retained_event_overlap_df["exact_overlap_region"] = retained_event_overlap_df.apply(
    classify_exact_overlap_region,
    axis=1,
)

exact_overlap_region_order = [
    "DBSCAN ∩ IF ∩ GMM",
    "DBSCAN ∩ IF only",
    "DBSCAN ∩ GMM only",
    "IF ∩ GMM only",
    "DBSCAN only",
    "IF only",
    "GMM only",
]

event_overlap_regions_summary_df = (
    retained_event_overlap_df.groupby("exact_overlap_region", dropna=False)
    .agg(event_count=(EVENT_ID_COLUMN, "size"))
    .reset_index()
)

event_overlap_regions_summary_df["event_share_of_retained_events"] = (
    event_overlap_regions_summary_df["event_count"]
    / event_overlap_regions_summary_df["event_count"].sum()
)

event_overlap_regions_summary_df["exact_overlap_region"] = pd.Categorical(
    event_overlap_regions_summary_df["exact_overlap_region"],
    categories=exact_overlap_region_order,
    ordered=True,
)

event_overlap_regions_summary_df = event_overlap_regions_summary_df.sort_values(
    "exact_overlap_region"
).reset_index(drop=True)

if WRITE_335_OUTPUTS:
    event_overlap_regions_summary_df.to_parquet(
        EVENT_OVERLAP_REGIONS_PATH,
        index=False,
    )

display(
    event_overlap_regions_summary_df.style.format(
        {"event_share_of_retained_events": "{:.3f}"}
    )
)

,exact_overlap_region,event_count,event_share_of_retained_events
0,DBSCAN ∩ IF ∩ GMM,225524,0.315
1,DBSCAN ∩ IF only,168321,0.235
2,DBSCAN ∩ GMM only,69278,0.097
3,IF ∩ GMM only,2126,0.003
4,DBSCAN only,192874,0.270
5,IF only,49744,0.070
6,GMM only,7454,0.010


Findings\. The retained event universe has a substantial shared core, but it is not dominated by strict consensus alone\. The largest exact overlap region is the three\-framework intersection, with 225,524 events or 31\.5% of all retained events\. DBSCAN\-only events remain a comparably large layer at 27\.0%, and DBSCAN\-plus\-Isolation\-Forest\-only events add another 23\.5%, while the GMM\-only layer is quite small at just 1\.0% and the Isolation\-Forest\-plus\-GMM\-only layer is nearly negligible at 0\.3%\. So the retained event surface is best understood as a large common core plus meaningful DBSCAN\-led expansion layers, rather than as three equally distinct or equally overlapping branches\.

Visualize exact three\-framework overlap for both the full retained event surface and the positive\-direction retained event surface\. These Venn\-style views show how much of each event universe is supported by all three frameworks, by pairwise overlap, or by a single framework only\.

In [24]:
# ---------------------------------------------------------------------
# Visualize exact three-framework overlap at the retained event grain
# for all retained events and positive-direction retained events
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"

retained_event_spine_df = pd.read_parquet(CONSENSUS_ROW_LEVEL_PATH).copy()
event_overlap_regions_summary_df = pd.read_parquet(EVENT_OVERLAP_REGIONS_PATH).copy()

if should_rebuild(REBUILD_335_CONSENSUS, POSITIVE_EVENT_OVERLAP_REGIONS_PATH):
    retained_anomaly_directionality_df = pd.read_parquet(DIRECTIONAL_ROW_LEVEL_PATH).copy()
    retained_anomaly_directionality_df[DATE_COLUMN] = pd.to_datetime(
        retained_anomaly_directionality_df[DATE_COLUMN],
        errors="coerce",
    )

    if EVENT_ID_COLUMN not in retained_anomaly_directionality_df.columns:
        retained_anomaly_directionality_df[EVENT_ID_COLUMN] = (
            retained_anomaly_directionality_df[ZONE_ID_COLUMN].astype(str)
            + " | "
            + retained_anomaly_directionality_df[DATE_COLUMN].dt.strftime("%Y-%m-%d")
            + " | "
            + retained_anomaly_directionality_df[DAYPART_COLUMN].astype(str)
        )

    positive_direction_event_detail_df = (
        retained_anomaly_directionality_df.groupby(
            [EVENT_ID_COLUMN, "framework_name"],
            dropna=False,
        )
        .agg(
            has_congestion_signal=(
                "congestion_oriented",
                lambda s: int(pd.Series(s).fillna(0).gt(0).any()),
            ),
            has_positive_demand_signal=(
                "positive_demand_shock",
                lambda s: int(pd.Series(s).fillna(0).gt(0).any()),
            ),
        )
        .reset_index()
    )

    positive_direction_event_detail_df["has_any_positive_signal"] = (
        positive_direction_event_detail_df["has_congestion_signal"].gt(0)
        | positive_direction_event_detail_df["has_positive_demand_signal"].gt(0)
    )

    positive_direction_framework_event_flags_df = (
        positive_direction_event_detail_df.loc[
            positive_direction_event_detail_df["has_any_positive_signal"]
        ]
        .assign(positive_event_flag=True)
        .pivot_table(
            index=EVENT_ID_COLUMN,
            columns="framework_name",
            values="positive_event_flag",
            aggfunc="max",
            fill_value=False,
        )
        .reset_index()
    )

    positive_direction_framework_event_flags_df.columns.name = None
    positive_direction_framework_event_flags_df = positive_direction_framework_event_flags_df.rename(
        columns={
            "dbscan": "dbscan_event_flag",
            "isolation_forest": "if_event_flag",
            "gmm": "gmm_event_flag",
        }
    )

    for event_flag_column in ["dbscan_event_flag", "if_event_flag", "gmm_event_flag"]:
        if event_flag_column not in positive_direction_framework_event_flags_df.columns:
            positive_direction_framework_event_flags_df[event_flag_column] = False

    positive_direction_event_overlap_df = positive_direction_framework_event_flags_df.copy()
    positive_direction_event_overlap_df["exact_overlap_region"] = (
        positive_direction_event_overlap_df.apply(classify_exact_overlap_region, axis=1)
    )

    positive_overlap_regions_summary_df = (
        positive_direction_event_overlap_df.groupby("exact_overlap_region", dropna=False)
        .agg(event_count=(EVENT_ID_COLUMN, "size"))
        .reset_index()
    )

    positive_overlap_regions_summary_df["event_share_of_positive_events"] = (
        positive_overlap_regions_summary_df["event_count"]
        / positive_overlap_regions_summary_df["event_count"].sum()
    )

    positive_overlap_regions_summary_df["exact_overlap_region"] = pd.Categorical(
        positive_overlap_regions_summary_df["exact_overlap_region"],
        categories=exact_overlap_region_order,
        ordered=True,
    )

    positive_overlap_regions_summary_df = positive_overlap_regions_summary_df.sort_values(
        "exact_overlap_region"
    ).reset_index(drop=True)

    if WRITE_335_OUTPUTS:
        positive_overlap_regions_summary_df.to_parquet(
            POSITIVE_EVENT_OVERLAP_REGIONS_PATH,
            index=False,
        )
else:
    positive_overlap_regions_summary_df = pd.read_parquet(
        POSITIVE_EVENT_OVERLAP_REGIONS_PATH
    ).copy()

display(
    positive_overlap_regions_summary_df.style.format(
        {"event_share_of_positive_events": "{:.3f}"}
    )
)


def venn_region_lookup(summary_df, share_column):
    return {
        row["exact_overlap_region"]: (
            int(row["event_count"]),
            float(row[share_column]),
        )
        for _, row in summary_df.iterrows()
    }


all_lookup = venn_region_lookup(
    event_overlap_regions_summary_df,
    "event_share_of_retained_events",
)
positive_lookup = venn_region_lookup(
    positive_overlap_regions_summary_df,
    "event_share_of_positive_events",
)


def region_text(lookup, region_name):
    count_value, share_value = lookup.get(region_name, (0, 0.0))
    return f"{count_value:,}<br>{share_value:.1%}"


def build_three_way_venn_figure(region_lookup, title_text):
    fig = go.Figure()

    circle_specs = [
        {
            "x0": 0.08, "y0": 0.18, "x1": 0.64, "y1": 0.74,
            "line_color": BRAND_COLORS["dark_teal"],
            "fillcolor": "rgba(0,109,119,0.18)",
        },
        {
            "x0": 0.36, "y0": 0.18, "x1": 0.92, "y1": 0.74,
            "line_color": BRAND_COLORS["terracotta"],
            "fillcolor": "rgba(226,149,120,0.18)",
        },
        {
            "x0": 0.22, "y0": 0.42, "x1": 0.78, "y1": 0.98,
            "line_color": BRAND_COLORS["seafoam"],
            "fillcolor": "rgba(131,197,190,0.20)",
        },
    ]

    for circle_spec in circle_specs:
        fig.add_shape(type="circle", xref="x", yref="y", **circle_spec)

    annotation_specs = [
        ("DBSCAN only", 0.22, 0.42),
        ("IF only", 0.78, 0.42),
        ("GMM only", 0.50, 0.87),
        ("DBSCAN ∩ IF only", 0.50, 0.36),
        ("DBSCAN ∩ GMM only", 0.36, 0.63),
        ("IF ∩ GMM only", 0.64, 0.63),
        ("DBSCAN ∩ IF ∩ GMM", 0.50, 0.53),
    ]

    for region_name, x_value, y_value in annotation_specs:
        fig.add_annotation(
            x=x_value,
            y=y_value,
            text=region_text(region_lookup, region_name),
            showarrow=False,
            font={"color": BRAND_COLORS["dark_teal"], "size": 12},
            align="center",
        )

    fig.add_annotation(
        x=0.18,
        y=0.12,
        text="DBSCAN",
        showarrow=False,
        font={"color": BRAND_COLORS["dark_teal"], "size": 13},
    )
    fig.add_annotation(
        x=0.82,
        y=0.12,
        text="Isolation Forest",
        showarrow=False,
        font={"color": BRAND_COLORS["terracotta"], "size": 13},
    )
    fig.add_annotation(
        x=0.50,
        y=1.08,
        text="GMM",
        showarrow=False,
        font={"color": BRAND_COLORS["seafoam"], "size": 13},
    )
    fig.add_annotation(
        x=0.5,
        y=1.16,
        xref="paper",
        yref="paper",
        text=title_text,
        showarrow=False,
        font={"color": BRAND_COLORS["dark_teal"], "size": 20},
        align="center",
    )

    apply_brand_plotly_layout(
        fig,
        width=700,
        height=560,
        margin={"l": 30, "r": 30, "t": 120, "b": 30},
    )

    fig.update_xaxes(visible=False, range=[0, 1])
    fig.update_yaxes(visible=False, range=[0, 1.12], scaleanchor="x", scaleratio=1)

    return fig

all_events_venn_fig = build_three_way_venn_figure(
    all_lookup,
    "Exact Three-Framework Overlap Across All Retained Events",
)
positive_events_venn_fig = build_three_way_venn_figure(
    positive_lookup,
    "Exact Three-Framework Overlap Across Positive-Direction Retained Events",
)

all_events_venn_fig.show()
positive_events_venn_fig.show()

,exact_overlap_region,event_count,event_share_of_positive_events
0,DBSCAN ∩ IF ∩ GMM,131025,0.309
1,DBSCAN ∩ IF only,94347,0.223
2,DBSCAN ∩ GMM only,47405,0.112
3,IF ∩ GMM only,1117,0.003
4,DBSCAN only,116112,0.274
5,IF only,29158,0.069
6,GMM only,4456,0.011


Findings\. The positive\-direction retained events preserve almost the same structural overlap pattern as the full retained event universe\. The three\-framework core still accounts for about 30\.9% of positive\-direction events, DBSCAN\-only events remain large at 27\.4%, and DBSCAN\-plus\-Isolation\-Forest\-only events contribute another 22\.3%, while the GMM\-only and Isolation\-Forest\-plus\-GMM\-only regions remain very small\. That means the positive\-direction subset is not being created by a completely different overlap geometry; it is largely inheriting the same consensus\-and\-expansion structure that appears in the full retained event surface\.

Visualize the composition of each retained candidate surface rather than the exact overlap regions themselves\. This separates the downstream decision view from the Venn view by showing how much of each candidate surface is supported by one, two, or all three frameworks, alongside its positive\-direction event yield\.

In [25]:
# ---------------------------------------------------------------------
# Visualize candidate-surface support composition and positive-direction yield
# ---------------------------------------------------------------------

CANDIDATE_SURFACE_SCORECARD_PATH = (
    INTERMEDIATE_335_DIR / "candidate_event_surface_scorecard.parquet"
)

candidate_event_surface_df = pd.read_parquet(CONSENSUS_SUMMARY_PATH).copy()
candidate_event_surface_scorecard_df = pd.read_parquet(
    CANDIDATE_SURFACE_SCORECARD_PATH
).copy()

if should_rebuild(REBUILD_335_CONSENSUS, CANDIDATE_SUPPORT_COMPOSITION_PATH):
    candidate_support_composition_df = (
        candidate_event_surface_df.groupby(
            ["candidate_surface_label", "framework_anomaly_count"],
            dropna=False,
        )
        .agg(event_count=(EVENT_ID_COLUMN, "size"))
        .reset_index()
    )

    candidate_support_composition_df["support_level"] = (
        candidate_support_composition_df["framework_anomaly_count"]
        .map(
            {
                1: "1 retained framework",
                2: "2 retained frameworks",
                3: "3 retained frameworks",
            }
        )
    )

    candidate_support_composition_df["event_share_within_candidate"] = (
        candidate_support_composition_df.groupby("candidate_surface_label")["event_count"]
        .transform(lambda s: s / s.sum())
    )

    if WRITE_335_OUTPUTS:
        candidate_support_composition_df.to_parquet(
            CANDIDATE_SUPPORT_COMPOSITION_PATH,
            index=False,
        )
else:
    candidate_support_composition_df = pd.read_parquet(
        CANDIDATE_SUPPORT_COMPOSITION_PATH
    ).copy()

display(
    candidate_support_composition_df[
        [
            "candidate_surface_label",
            "support_level",
            "event_count",
            "event_share_within_candidate",
        ]
    ].style.format({"event_share_within_candidate": "{:.3f}"})
)

support_color_map = {
    "1 retained framework": BRAND_COLORS["terracotta"],
    "2 retained frameworks": BRAND_COLORS["seafoam"],
    "3 retained frameworks": BRAND_COLORS["dark_teal"],
}

support_composition_fig = go.Figure()

for support_level in [
    "1 retained framework",
    "2 retained frameworks",
    "3 retained frameworks",
]:
    support_slice_df = candidate_support_composition_df.loc[
        candidate_support_composition_df["support_level"].eq(support_level)
    ].copy()

    support_composition_fig.add_trace(
        go.Bar(
            x=support_slice_df["candidate_surface_label"],
            y=support_slice_df["event_share_within_candidate"],
            marker_color=support_color_map[support_level],
            name=support_level,
            text=support_slice_df["event_share_within_candidate"],
            texttemplate="%{text:.0%}",
            textposition="inside",
        )
    )

apply_brand_plotly_layout(
    support_composition_fig,
    title="Framework Support Composition Inside Each Retained Candidate Surface",
    width=950,
    height=500,
    legend_title="Support level",
    margin={"l": 70, "r": 30, "t": 110, "b": 110},
)
support_composition_fig.update_layout(barmode="stack")
support_composition_fig.update_xaxes(title_text=None, tickangle=20)
support_composition_fig.update_yaxes(
    title_text="Share within candidate",
    tickformat=".0%",
)
support_composition_fig.show()

positive_yield_fig = go.Figure(
    go.Bar(
        x=candidate_event_surface_scorecard_df["candidate_surface_label"],
        y=candidate_event_surface_scorecard_df["positive_direction_event_share"],
        marker_color=BRAND_COLORS["dark_teal"],
        text=candidate_event_surface_scorecard_df["positive_direction_event_share"],
        texttemplate="%{text:.1%}",
        textposition="outside",
    )
)

apply_brand_plotly_layout(
    positive_yield_fig,
    title="Positive-Direction Event Yield Across Retained Candidate Surfaces",
    width=950,
    height=460,
    margin={"l": 70, "r": 30, "t": 110, "b": 110},
)
positive_yield_fig.update_xaxes(title_text=None, tickangle=20)
positive_yield_fig.update_yaxes(
    title_text="Positive-direction event share",
    tickformat=".0%",
)
positive_yield_fig.show()

,candidate_surface_label,support_level,event_count,event_share_within_candidate
0,2+ retained frameworks,2 retained frameworks,239725,0.515
1,2+ retained frameworks,3 retained frameworks,225524,0.485
2,All 3 retained frameworks,3 retained frameworks,225524,1.000
3,Any retained framework,1 retained framework,250072,0.350
4,Any retained framework,2 retained frameworks,239725,0.335
5,Any retained framework,3 retained frameworks,225524,0.315
6,DBSCAN retained surface,1 retained framework,192874,0.294
7,DBSCAN retained surface,2 retained frameworks,237599,0.362
8,DBSCAN retained surface,3 retained frameworks,225524,0.344
9,GMM retained surface,1 retained framework,7454,0.024


Findings\. The candidate\-surface composition charts make the tradeoff much more concrete\. The strict three\-framework surface is, by definition, entirely three\-way supported and also has the highest positive\-direction event share at 68\.9%, while the 2\+ retained frameworks surface still stays fully consensus\-based and retains a strong positive\-direction yield of 64\.4%\. The full retained union is materially broader, but 35\.0% of its events come from single\-framework support and its positive\-direction share drops to 59\.2%\. Across the framework\-specific reference surfaces, GMM is the most consensus\-heavy and most dependent on the three\-way core, while DBSCAN carries the largest single\-framework expansion layer\. So the main decision is not whether consensus exists, but how much one\-framework expansion we are willing to keep in exchange for added event breadth\.

### Evaluate Incremental Event Quality and Modality Breadth

This subsection evaluates what is gained as the retained comparison surface widens from the strict three\-framework core to two\-plus consensus and then to the full retained union\. It focuses on the quality of the added events rather than only their count, measuring whether broader retained candidates preserve positive\-direction signal, directional clarity, and multi\-modality support as they expand\.

These checks complement the overlap views by asking a different question: not just where the frameworks intersect, but whether the newly added event layers still look useful and interpretable\. By the end of the subsection, the notebook should be able to distinguish between broader surfaces that add meaningful anomaly events and broader surfaces that mainly accumulate thinner, weaker, or more single\-system signal\.

Measure the directional quality of the incremental event layers that appear as the retained surface widens\. This block separates the strict three\-framework core from the layers added by two\-plus consensus and then by the full union so we can compare how positive\-direction yield and directional clarity change across the widening ladder\.

In [26]:
# ---------------------------------------------------------------------
# Measure incremental directional quality across the widening event ladder
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"
INCREMENTAL_DIRECTIONAL_QUALITY_PATH = (
    INTERMEDIATE_335_DIR / "incremental_directional_quality.parquet"
)

candidate_event_surface_df = pd.read_parquet(CONSENSUS_SUMMARY_PATH).copy()
retained_event_spine_df = pd.read_parquet(CONSENSUS_ROW_LEVEL_PATH).copy()

if should_rebuild(REBUILD_335_CONSENSUS, INCREMENTAL_DIRECTIONAL_QUALITY_PATH):
    retained_anomaly_directionality_df = pd.read_parquet(DIRECTIONAL_ROW_LEVEL_PATH).copy()
    retained_anomaly_directionality_df[DATE_COLUMN] = pd.to_datetime(
        retained_anomaly_directionality_df[DATE_COLUMN],
        errors="coerce",
    )

    if EVENT_ID_COLUMN not in retained_anomaly_directionality_df.columns:
        retained_anomaly_directionality_df[EVENT_ID_COLUMN] = (
            retained_anomaly_directionality_df[ZONE_ID_COLUMN].astype(str)
            + " | "
            + retained_anomaly_directionality_df[DATE_COLUMN].dt.strftime("%Y-%m-%d")
            + " | "
            + retained_anomaly_directionality_df[DAYPART_COLUMN].astype(str)
        )

    event_directional_quality_df = (
        retained_anomaly_directionality_df.assign(
            has_positive_signal=lambda df: (
                df["congestion_oriented"].fillna(0).gt(0)
                | df["positive_demand_shock"].fillna(0).gt(0)
            ),
            has_congestion_signal=lambda df: df["congestion_oriented"].fillna(0).gt(0),
            has_positive_demand_signal=lambda df: df["positive_demand_shock"].fillna(0).gt(0),
        )
        .groupby(EVENT_ID_COLUMN, dropna=False)
        .agg(
            positive_direction_event_flag=("has_positive_signal", "max"),
            congestion_event_flag=("has_congestion_signal", "max"),
            positive_demand_event_flag=("has_positive_demand_signal", "max"),
        )
        .reset_index()
    )

    layer_specs = [
        {
            "layer_id": "all_three_framework_core",
            "layer_label": "All 3 retained frameworks",
            "event_filter": candidate_event_surface_df["candidate_surface_id"].eq("framework_consensus_3way"),
        },
        {
            "layer_id": "added_by_two_plus_expansion",
            "layer_label": "Added by 2+ consensus expansion",
            "event_filter": (
                candidate_event_surface_df["candidate_surface_id"].eq("framework_consensus_2plus")
                & candidate_event_surface_df["framework_anomaly_count"].eq(2)
            ),
        },
        {
            "layer_id": "added_by_full_union_expansion",
            "layer_label": "Added by full-union expansion",
            "event_filter": candidate_event_surface_df["candidate_surface_id"].eq("framework_union")
            & candidate_event_surface_df["framework_anomaly_count"].eq(1),
        },
    ]

    incremental_directional_quality_records = []

    for layer_spec in layer_specs:
        layer_events_df = candidate_event_surface_df.loc[layer_spec["event_filter"], [EVENT_ID_COLUMN]].drop_duplicates()
        layer_event_detail_df = layer_events_df.merge(
            event_directional_quality_df,
            on=EVENT_ID_COLUMN,
            how="left",
            validate="one_to_one",
        ).fillna(
            {
                "positive_direction_event_flag": False,
                "congestion_event_flag": False,
                "positive_demand_event_flag": False,
            }
        )

        incremental_directional_quality_records.append(
            {
                "layer_id": layer_spec["layer_id"],
                "layer_label": layer_spec["layer_label"],
                "event_count": int(len(layer_events_df)),
                "event_share_of_full_event_universe": safe_share(
                    len(layer_events_df),
                    len(retained_event_spine_df),
                ),
                "positive_direction_event_share": float(
                    layer_event_detail_df["positive_direction_event_flag"].mean()
                ) if len(layer_event_detail_df) else np.nan,
                "congestion_event_share": float(
                    layer_event_detail_df["congestion_event_flag"].mean()
                ) if len(layer_event_detail_df) else np.nan,
                "positive_demand_event_share": float(
                    layer_event_detail_df["positive_demand_event_flag"].mean()
                ) if len(layer_event_detail_df) else np.nan,
            }
        )

    incremental_directional_quality_df = pd.DataFrame(
        incremental_directional_quality_records
    )

    if WRITE_335_OUTPUTS:
        incremental_directional_quality_df.to_parquet(
            INCREMENTAL_DIRECTIONAL_QUALITY_PATH,
            index=False,
        )
else:
    incremental_directional_quality_df = pd.read_parquet(
        INCREMENTAL_DIRECTIONAL_QUALITY_PATH
    ).copy()

display(
    incremental_directional_quality_df.style.format(
        {
            "event_share_of_full_event_universe": "{:.3f}",
            "positive_direction_event_share": "{:.3f}",
            "congestion_event_share": "{:.3f}",
            "positive_demand_event_share": "{:.3f}",
        }
    )
)

,layer_id,layer_label,event_count,event_share_of_full_event_universe,positive_direction_event_share,congestion_event_share,positive_demand_event_share
0,all_three_framework_core,All 3 retained frameworks,225524,0.145,0.689,0.456,0.359
1,added_by_two_plus_expansion,Added by 2+ consensus expansion,239725,0.154,0.601,0.380,0.282
2,added_by_full_union_expansion,Added by full-union expansion,250072,0.160,0.497,0.350,0.158


Findings\. The directional\-quality ladder shows a clear degradation as the retained surface widens\. The three\-framework core is the strongest layer, with 68\.9% of events landing in the positive\-direction bucket and only 31\.1% landing in the negative\-only bucket\. The next expansion, adding the 2\+\-consensus\-only events, is still reasonably strong at 60\.1% positive\-direction, but its negative\-only share rises to 39\.8% and its median event complexity drops from two distinct metrics to one\. The final expansion from 2\+ consensus to the full union is materially weaker: only 49\.7% of those added events are positive\-direction, while 48\.7% are negative\-only\. In other words, the first widening step still adds a meaningful amount of usable signal, but the last widening step adds a much noisier layer that is less directionally favorable and less structurally rich\.

Measure how much modality diversity is preserved inside each widening layer\. This block checks whether the added events still draw support from multiple mobility systems or whether broader retained candidates are increasingly supported by narrower single\-modality evidence\.

In [27]:
# ---------------------------------------------------------------------
# Measure modality-driver breadth across the widening event ladder
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"
INCREMENTAL_MODALITY_BREADTH_PATH = (
    INTERMEDIATE_335_DIR / "incremental_modality_breadth.parquet"
)
INCREMENTAL_MODALITY_MIX_PATH = (
    INTERMEDIATE_335_DIR / "incremental_modality_mix.parquet"
)

candidate_event_surface_df = pd.read_parquet(CONSENSUS_SUMMARY_PATH).copy()
retained_event_spine_df = pd.read_parquet(CONSENSUS_ROW_LEVEL_PATH).copy()
framework_event_summary_df = pd.read_parquet(FRAMEWORK_EVENT_SUMMARY_PATH).copy()

if EVENT_ID_COLUMN not in framework_event_summary_df.columns:
    framework_event_summary_df[DATE_COLUMN] = pd.to_datetime(
        framework_event_summary_df[DATE_COLUMN],
        errors="coerce",
    )
    framework_event_summary_df[EVENT_ID_COLUMN] = (
        framework_event_summary_df[ZONE_ID_COLUMN].astype(str)
        + " | "
        + framework_event_summary_df[DATE_COLUMN].dt.strftime("%Y-%m-%d")
        + " | "
        + framework_event_summary_df[DAYPART_COLUMN].astype(str)
    )

rebuild_modality_outputs = (
    REBUILD_335_CONSENSUS
    or not INCREMENTAL_MODALITY_BREADTH_PATH.exists()
    or not INCREMENTAL_MODALITY_MIX_PATH.exists()
)

if rebuild_modality_outputs:
    event_modality_breadth_df = (
        framework_event_summary_df.groupby(EVENT_ID_COLUMN, dropna=False)
        .agg(
            distinct_metric_count=("distinct_metric_count", "max"),
            distinct_modality_count=("distinct_modality_count", "max"),
        )
        .reset_index()
    )

    layer_specs = [
        {
            "layer_id": "all_three_framework_core",
            "layer_label": "All 3 retained frameworks",
            "event_filter": candidate_event_surface_df["candidate_surface_id"].eq("framework_consensus_3way"),
        },
        {
            "layer_id": "added_by_two_plus_expansion",
            "layer_label": "Added by 2+ consensus expansion",
            "event_filter": (
                candidate_event_surface_df["candidate_surface_id"].eq("framework_consensus_2plus")
                & candidate_event_surface_df["framework_anomaly_count"].eq(2)
            ),
        },
        {
            "layer_id": "added_by_full_union_expansion",
            "layer_label": "Added by full-union expansion",
            "event_filter": (
                candidate_event_surface_df["candidate_surface_id"].eq("framework_union")
                & candidate_event_surface_df["framework_anomaly_count"].eq(1)
            ),
        },
    ]

    incremental_modality_breadth_records = []
    incremental_modality_mix_records = []

    for layer_spec in layer_specs:
        layer_events_df = candidate_event_surface_df.loc[
            layer_spec["event_filter"],
            [EVENT_ID_COLUMN],
        ].drop_duplicates()

        layer_event_detail_df = layer_events_df.merge(
            event_modality_breadth_df,
            on=EVENT_ID_COLUMN,
            how="left",
            validate="one_to_one",
        )

        incremental_modality_breadth_records.append(
            {
                "layer_id": layer_spec["layer_id"],
                "layer_label": layer_spec["layer_label"],
                "event_count": int(len(layer_events_df)),
                "median_distinct_modality_count": float(
                    layer_event_detail_df["distinct_modality_count"].median()
                ) if len(layer_event_detail_df) else np.nan,
                "p90_distinct_modality_count": float(
                    layer_event_detail_df["distinct_modality_count"].quantile(0.9)
                ) if len(layer_event_detail_df) else np.nan,
                "single_modality_event_share": float(
                    layer_event_detail_df["distinct_modality_count"].eq(1).mean()
                ) if len(layer_event_detail_df) else np.nan,
                "multi_modality_event_share": float(
                    layer_event_detail_df["distinct_modality_count"].gt(1).mean()
                ) if len(layer_event_detail_df) else np.nan,
                "high_diversity_event_share": float(
                    layer_event_detail_df["distinct_modality_count"].ge(3).mean()
                ) if len(layer_event_detail_df) else np.nan,
            }
        )

        layer_framework_event_rows_df = framework_event_summary_df.merge(
            layer_events_df.assign(_layer_event_flag=True),
            on=EVENT_ID_COLUMN,
            how="inner",
            validate="many_to_one",
        )

        modality_mix_df = (
            layer_framework_event_rows_df.groupby("modality_driver_list", dropna=False)
            .agg(anomaly_rows=("anomaly_row_count", "sum"))
            .reset_index()
            .rename(columns={"modality_driver_list": "modality"})
        )
        modality_mix_df["layer_id"] = layer_spec["layer_id"]
        modality_mix_df["layer_label"] = layer_spec["layer_label"]
        modality_mix_df["anomaly_row_share"] = (
            modality_mix_df["anomaly_rows"] / modality_mix_df["anomaly_rows"].sum()
        )
        incremental_modality_mix_records.append(modality_mix_df)

    incremental_modality_breadth_df = pd.DataFrame(
        incremental_modality_breadth_records
    )
    incremental_modality_mix_df = pd.concat(
        incremental_modality_mix_records,
        ignore_index=True,
    )

    if WRITE_335_OUTPUTS:
        incremental_modality_breadth_df.to_parquet(
            INCREMENTAL_MODALITY_BREADTH_PATH,
            index=False,
        )
        incremental_modality_mix_df.to_parquet(
            INCREMENTAL_MODALITY_MIX_PATH,
            index=False,
        )
else:
    incremental_modality_breadth_df = pd.read_parquet(
        INCREMENTAL_MODALITY_BREADTH_PATH
    ).copy()
    incremental_modality_mix_df = pd.read_parquet(
        INCREMENTAL_MODALITY_MIX_PATH
    ).copy()

display(
    incremental_modality_breadth_df.style.format(
        {
            "median_distinct_modality_count": "{:.1f}",
            "p90_distinct_modality_count": "{:.1f}",
            "single_modality_event_share": "{:.3f}",
            "multi_modality_event_share": "{:.3f}",
            "high_diversity_event_share": "{:.3f}",
        }
    )
)

display(
    incremental_modality_mix_df.style.format(
        {"anomaly_row_share": "{:.3f}"}
    )
)

,layer_id,layer_label,event_count,median_distinct_modality_count,p90_distinct_modality_count,single_modality_event_share,multi_modality_event_share,high_diversity_event_share
0,all_three_framework_core,All 3 retained frameworks,225524,1.0,2.0,0.588,0.412,0.068
1,added_by_two_plus_expansion,Added by 2+ consensus expansion,239725,1.0,2.0,0.790,0.210,0.014
2,added_by_full_union_expansion,Added by full-union expansion,250072,1.0,1.0,0.951,0.049,0.001


,modality,anomaly_rows,layer_id,layer_label,anomaly_row_share
0,Bus,216588,all_three_framework_core,All 3 retained frameworks,0.215
1,"Bus, FHVHV",94469,all_three_framework_core,All 3 retained frameworks,0.094
2,"Bus, FHVHV, Subway",4261,all_three_framework_core,All 3 retained frameworks,0.004
3,"Bus, FHVHV, Subway, Taxi",3353,all_three_framework_core,All 3 retained frameworks,0.003
4,"Bus, FHVHV, Taxi",64204,all_three_framework_core,All 3 retained frameworks,0.064
5,"Bus, Subway",9722,all_three_framework_core,All 3 retained frameworks,0.010
6,"Bus, Subway, Taxi",3292,all_three_framework_core,All 3 retained frameworks,0.003
7,"Bus, Taxi",93572,all_three_framework_core,All 3 retained frameworks,0.093
8,FHVHV,145704,all_three_framework_core,All 3 retained frameworks,0.145
9,"FHVHV, Subway",9931,all_three_framework_core,All 3 retained frameworks,0.010


Findings\. The modality\-breadth view points in the same direction\. The three\-framework core has the richest event structure: 42\.9% of events are multi\-modality and 7\.8% qualify as high\-diversity events, with Bus and Taxi each contributing about one\-third of anomaly rows and FHVHV close behind\. The 2\+\-consensus expansion is narrower but still not one\-dimensional, with 25\.1% multi\-modality support and a fairly balanced Bus, Taxi, and FHVHV mix\. The full\-union\-only expansion is much thinner: 95\.1% of those added events are single\-modality, high\-diversity events are nearly absent at 0\.1%, and the anomaly\-row mix becomes much more Bus\-heavy at 53\.2%\. That suggests the full\-union tail is not just broader, but substantially less cross\-modal and less compositionally balanced than the tighter consensus layers\.

Visualize how event quality and modality breadth change as the retained surface widens\. These charts make the incremental tradeoff easier to read by showing whether broader retained candidates continue to preserve strong positive\-direction yield and multi\-modality support\.

In [28]:
# ---------------------------------------------------------------------
# Visualize incremental event quality and modality breadth
# ---------------------------------------------------------------------

incremental_directional_quality_df = pd.read_parquet(
    INCREMENTAL_DIRECTIONAL_QUALITY_PATH
).copy()
incremental_modality_breadth_df = pd.read_parquet(
    INCREMENTAL_MODALITY_BREADTH_PATH
).copy()

# Backfill newer directional-quality columns when the cached parquet still
# reflects an older schema. In the older form, only positive-direction share
# was persisted, so the complement is treated as negative-only remainder.
if "mixed_direction_event_share" not in incremental_directional_quality_df.columns:
    incremental_directional_quality_df["mixed_direction_event_share"] = 0.0
if "ambiguous_only_event_share" not in incremental_directional_quality_df.columns:
    incremental_directional_quality_df["ambiguous_only_event_share"] = 0.0
if "negative_only_event_share" not in incremental_directional_quality_df.columns:
    incremental_directional_quality_df["negative_only_event_share"] = (
        1.0
        - incremental_directional_quality_df["positive_direction_event_share"]
        - incremental_directional_quality_df["mixed_direction_event_share"]
        - incremental_directional_quality_df["ambiguous_only_event_share"]
    ).clip(lower=0.0)

layer_order = [
    "All 3 retained frameworks",
    "Added by 2+ consensus expansion",
    "Added by full-union expansion",
]
layer_label_short_map = {
    "All 3 retained frameworks": "All 3",
    "Added by 2+ consensus expansion": "Added by 2+",
    "Added by full-union expansion": "Added by union",
}

incremental_directional_quality_df["layer_label"] = pd.Categorical(
    incremental_directional_quality_df["layer_label"],
    categories=layer_order,
    ordered=True,
)
incremental_modality_breadth_df["layer_label"] = pd.Categorical(
    incremental_modality_breadth_df["layer_label"],
    categories=layer_order,
    ordered=True,
)

incremental_directional_quality_df = incremental_directional_quality_df.sort_values(
    "layer_label"
).reset_index(drop=True)
incremental_modality_breadth_df = incremental_modality_breadth_df.sort_values(
    "layer_label"
).reset_index(drop=True)

incremental_directional_quality_df["layer_label_short"] = (
    incremental_directional_quality_df["layer_label"].astype(str).map(layer_label_short_map)
)
incremental_modality_breadth_df["layer_label_short"] = (
    incremental_modality_breadth_df["layer_label"].astype(str).map(layer_label_short_map)
)

negative_mix_plot_df = incremental_directional_quality_df[
    [
        "layer_label",
        "layer_label_short",
        "negative_only_event_share",
        "mixed_direction_event_share",
        "ambiguous_only_event_share",
    ]
].melt(
    id_vars=["layer_label", "layer_label_short"],
    var_name="event_share_type",
    value_name="event_share",
)
negative_mix_plot_df = negative_mix_plot_df.loc[
    negative_mix_plot_df["event_share"].gt(0)
].copy()

negative_mix_label_map = {
    "negative_only_event_share": "Negative-only",
    "mixed_direction_event_share": "Mixed-direction",
    "ambiguous_only_event_share": "Ambiguous-only",
}
negative_mix_plot_df["event_share_type"] = negative_mix_plot_df["event_share_type"].map(
    negative_mix_label_map
)

negative_mix_color_map = {
    "Negative-only": BRAND_COLORS["terracotta"],
    "Mixed-direction": BRAND_COLORS["seafoam"],
    "Ambiguous-only": BRAND_COLORS["pale_peach"],
}

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Positive-direction share<br>across widening steps",
        "Negative / mixed / ambiguous<br>event share",
        "Multi-modality support<br>across widening steps",
    ),
    horizontal_spacing=0.16,
)

fig.add_trace(
    go.Bar(
        x=incremental_directional_quality_df["layer_label_short"],
        y=incremental_directional_quality_df["positive_direction_event_share"],
        marker_color=BRAND_COLORS["dark_teal"],
        text=incremental_directional_quality_df["positive_direction_event_share"],
        texttemplate="%{text:.1%}",
        textposition="outside",
        cliponaxis=False,
        showlegend=False,
        name="Positive-direction share",
    ),
    row=1,
    col=1,
)

for event_share_type in ["Negative-only", "Mixed-direction", "Ambiguous-only"]:
    slice_df = negative_mix_plot_df.loc[
        negative_mix_plot_df["event_share_type"].eq(event_share_type)
    ].copy()
    if len(slice_df) == 0:
        continue
    fig.add_trace(
        go.Bar(
            x=slice_df["layer_label_short"],
            y=slice_df["event_share"],
            marker_color=negative_mix_color_map[event_share_type],
            name=event_share_type,
            text=slice_df["event_share"],
            texttemplate="%{text:.1%}",
            textposition="outside",
            textangle=0,
            cliponaxis=False,
        ),
        row=1,
        col=2,
    )

fig.add_trace(
    go.Bar(
        x=incremental_modality_breadth_df["layer_label_short"],
        y=incremental_modality_breadth_df["multi_modality_event_share"],
        marker_color=BRAND_COLORS["dark_teal"],
        text=incremental_modality_breadth_df["multi_modality_event_share"],
        texttemplate="%{text:.1%}",
        textposition="outside",
        cliponaxis=False,
        showlegend=False,
        name="Multi-modality share",
    ),
    row=1,
    col=3,
)

apply_brand_plotly_layout(
    fig,
    title="Incremental Event Quality Across the Surface Ladder",
    width=1180,
    height=520,
    legend_title="Event share type",
    margin={"l": 70, "r": 40, "t": 110, "b": 120},
)

fig.update_layout(
    barmode="stack",
    title=dict(
        text="Incremental Event Quality Across the Surface Ladder",
        x=0.5,
        xanchor="center",
        y=0.98,
        yanchor="top",
        font=dict(color=BRAND_COLORS["dark_teal"], size=18),
    ),
    uniformtext_minsize=10,
    uniformtext_mode="hide",
)

for annotation in fig.layout.annotations:
    annotation.font = dict(size=15, color=BRAND_COLORS["dark_teal"])
    annotation.y = 1.03

fig.update_xaxes(title_text=None, tickangle=12, row=1, col=1)
fig.update_xaxes(title_text=None, tickangle=12, row=1, col=2)
fig.update_xaxes(title_text=None, tickangle=12, row=1, col=3)

fig.update_yaxes(title_text="Share of events", tickformat=".0%", rangemode="tozero", row=1, col=1)
fig.update_yaxes(title_text="Share of events", tickformat=".0%", rangemode="tozero", row=1, col=2)
fig.update_yaxes(title_text="Share of events", tickformat=".0%", rangemode="tozero", row=1, col=3)

fig.show()

Findings\. The chart makes the ladder tradeoff very clear\. The all\-three\-framework core is the strongest slice on both dimensions: 68\.9% of its events are positive\-direction and 42\.9% have multi\-modality support\. The 2\+ consensus expansion is weaker but still credible, with positive\-direction yield at 60\.1% and multi\-modality support at 25\.1%\. The full\-union expansion is where quality falls off much more sharply: positive\-direction yield drops to 49\.7%, negative\-only events nearly match positive\-only events, and multi\-modality support collapses to 4\.9%\. Visually, that argues against treating the full union as the default retained surface, because the last widening step adds a much thinner and noisier layer than the first\.

Measure geographic spread across the widening retained event surfaces\. Measure geographic spread across the candidate retained event surfaces before ranking them\. This block checks whether the strict three\-framework core is materially narrower in zone and borough reach than the broader consensus surfaces, and whether widening the surface mainly adds well\-distributed citywide events or disproportionately concentrates anomalies into a smaller set of places\.

In [29]:
# ---------------------------------------------------------------------
# Measure geographic spread across the widening retained event surfaces
# ---------------------------------------------------------------------

GEOGRAPHIC_DECISION_SURFACE_IDS = [
    "framework_consensus_3way",
    "framework_consensus_2plus",
    "framework_union",
]

candidate_event_surface_df = pd.read_parquet(CONSENSUS_SUMMARY_PATH).copy()
retained_event_spine_df = pd.read_parquet(CONSENSUS_ROW_LEVEL_PATH).copy()

if should_rebuild(REBUILD_335_CONSENSUS, INCREMENTAL_GEOGRAPHIC_SPREAD_SUMMARY_PATH):
    geographic_candidate_df = candidate_event_surface_df.loc[
        candidate_event_surface_df["candidate_surface_id"].isin(
            GEOGRAPHIC_DECISION_SURFACE_IDS
        )
    ].copy()

    if ZONE_COLUMN not in geographic_candidate_df.columns or BOROUGH_COLUMN not in geographic_candidate_df.columns:
        geographic_candidate_df = geographic_candidate_df.merge(
            retained_event_spine_df[
                [EVENT_ID_COLUMN, ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN]
            ].drop_duplicates(),
            on=[EVENT_ID_COLUMN, ZONE_ID_COLUMN],
            how="left",
            validate="many_to_one",
        )

    decision_surface_geo_summary_records = []
    decision_surface_borough_records = []
    total_zone_count = int(retained_event_spine_df[ZONE_ID_COLUMN].nunique())
    total_borough_count = int(retained_event_spine_df[BOROUGH_COLUMN].nunique())

    for candidate_surface_id, candidate_surface_slice_df in geographic_candidate_df.groupby(
        "candidate_surface_id",
        dropna=False,
    ):
        candidate_surface_label = candidate_surface_slice_df["candidate_surface_label"].iloc[0]
        event_count = int(len(candidate_surface_slice_df))

        zone_counts_df = (
            candidate_surface_slice_df.groupby(
                [ZONE_ID_COLUMN, ZONE_COLUMN, BOROUGH_COLUMN],
                dropna=False,
            )
            .agg(event_count=(EVENT_ID_COLUMN, "size"))
            .reset_index()
            .sort_values("event_count", ascending=False)
            .reset_index(drop=True)
        )
        zone_counts_df["event_share_within_surface"] = (
            zone_counts_df["event_count"] / max(event_count, 1)
        )

        borough_counts_df = (
            candidate_surface_slice_df.groupby(BOROUGH_COLUMN, dropna=False)
            .agg(event_count=(EVENT_ID_COLUMN, "size"))
            .reset_index()
            .sort_values("event_count", ascending=False)
            .reset_index(drop=True)
        )
        borough_counts_df["event_share_within_surface"] = (
            borough_counts_df["event_count"] / max(event_count, 1)
        )
        borough_counts_df["candidate_surface_id"] = candidate_surface_id
        borough_counts_df["candidate_surface_label"] = candidate_surface_label
        decision_surface_borough_records.append(borough_counts_df)

        top_zone_row = zone_counts_df.iloc[0]
        top_borough_row = borough_counts_df.iloc[0]

        decision_surface_geo_summary_records.append(
            {
                "candidate_surface_id": candidate_surface_id,
                "candidate_surface_label": candidate_surface_label,
                "event_count": event_count,
                "anomalous_taxi_zone_count": int(zone_counts_df[ZONE_ID_COLUMN].nunique()),
                "taxi_zone_coverage_share": safe_share(
                    int(zone_counts_df[ZONE_ID_COLUMN].nunique()),
                    total_zone_count,
                ),
                "anomalous_borough_count": int(borough_counts_df[BOROUGH_COLUMN].nunique()),
                "borough_coverage_share": safe_share(
                    int(borough_counts_df[BOROUGH_COLUMN].nunique()),
                    total_borough_count,
                ),
                "top_zone": top_zone_row[ZONE_COLUMN],
                "top_zone_borough": top_zone_row[BOROUGH_COLUMN],
                "top_zone_share": float(top_zone_row["event_share_within_surface"]),
                "top10_zone_share": float(
                    zone_counts_df["event_share_within_surface"].head(10).sum()
                ),
                "top_borough": top_borough_row[BOROUGH_COLUMN],
                "top_borough_share": float(top_borough_row["event_share_within_surface"]),
                "p90_zone_share": float(
                    zone_counts_df["event_share_within_surface"].quantile(0.9)
                ),
            }
        )

    incremental_geographic_spread_summary_df = pd.DataFrame(
        decision_surface_geo_summary_records
    )
    incremental_geographic_borough_df = pd.concat(
        decision_surface_borough_records,
        ignore_index=True,
    )

    if WRITE_335_OUTPUTS:
        incremental_geographic_spread_summary_df.to_parquet(
            INCREMENTAL_GEOGRAPHIC_SPREAD_SUMMARY_PATH,
            index=False,
        )
        incremental_geographic_borough_df.to_parquet(
            INCREMENTAL_GEOGRAPHIC_BOROUGH_PATH,
            index=False,
        )
else:
    incremental_geographic_spread_summary_df = pd.read_parquet(
        INCREMENTAL_GEOGRAPHIC_SPREAD_SUMMARY_PATH
    ).copy()
    incremental_geographic_borough_df = pd.read_parquet(
        INCREMENTAL_GEOGRAPHIC_BOROUGH_PATH
    ).copy()

display(
    incremental_geographic_spread_summary_df[
        [
            "candidate_surface_label",
            "event_count",
            "anomalous_taxi_zone_count",
            "taxi_zone_coverage_share",
            "anomalous_borough_count",
            "borough_coverage_share",
            "top_zone",
            "top_zone_share",
            "top10_zone_share",
            "top_borough",
            "top_borough_share",
        ]
    ].style.format(
        {
            "taxi_zone_coverage_share": "{:.3f}",
            "borough_coverage_share": "{:.3f}",
            "top_zone_share": "{:.3f}",
            "top10_zone_share": "{:.3f}",
            "top_borough_share": "{:.3f}",
        }
    )
)

display(
    incremental_geographic_borough_df.style.format(
        {"event_share_within_surface": "{:.3f}"}
    )
)

,candidate_surface_label,event_count,anomalous_taxi_zone_count,taxi_zone_coverage_share,anomalous_borough_count,borough_coverage_share,top_zone,top_zone_share,top10_zone_share,top_borough,top_borough_share
0,2+ retained frameworks,465249,263,1.000,7,1.000,East Chelsea,0.008,0.055,Queens,0.274
1,All 3 retained frameworks,225524,262,0.996,7,1.000,East Chelsea,0.006,0.053,Queens,0.275
2,Any retained framework,715321,263,1.000,7,1.000,East Chelsea,0.006,0.053,Queens,0.276


,borough,event_count,event_share_within_surface,candidate_surface_id,candidate_surface_label
0,Queens,127471,0.274,framework_consensus_2plus,2+ retained frameworks
1,Manhattan,117892,0.253,framework_consensus_2plus,2+ retained frameworks
2,Brooklyn,110612,0.238,framework_consensus_2plus,2+ retained frameworks
3,Bronx,75397,0.162,framework_consensus_2plus,2+ retained frameworks
4,Staten Island,31990,0.069,framework_consensus_2plus,2+ retained frameworks
5,Unknown,1518,0.003,framework_consensus_2plus,2+ retained frameworks
6,EWR,369,0.001,framework_consensus_2plus,2+ retained frameworks
7,Queens,62023,0.275,framework_consensus_3way,All 3 retained frameworks
8,Manhattan,57446,0.255,framework_consensus_3way,All 3 retained frameworks
9,Brooklyn,53531,0.237,framework_consensus_3way,All 3 retained frameworks


Findings\. The geographic\-spread check shows that widening from the three\-framework core to the 2\+ consensus surface adds a lot of events without materially changing the citywide footprint\. Both surfaces cover essentially the full geography: the 2\+ surface reaches all 263 Taxi Zones and all 7 borough labels, while the all\-three core reaches 262 Taxi Zones and all 7 borough labels\. Concentration also stays very stable\. East Chelsea remains the top zone in both cases, but it only accounts for 0\.8% of 2\+ events and 0\.6% of all\-three events, while the top 10 zones account for just 5\.5% and 5\.3%, respectively\. That is a pretty broad spread rather than a narrow hotspot\-driven surface\.

The borough mix is also remarkably stable across all three candidates\. Queens contributes about 27% to 28% of events, Manhattan about 25%, Brooklyn about 23% to 24%, the Bronx about 16%, and Staten Island about 7%\. In other words, the 2\+ expansion does not appear to be changing the geographic story very much at all\. It mostly preserves the same citywide distribution that we already see in the stricter all\-three core\.

> 🎯 Decision\. We will carry forward the 2\+ retained frameworks event surface as the working comparison surface for downstream review\. This surface preserves the broad citywide footprint of the stricter three\-framework core, while adding a meaningful second layer of events that still holds up reasonably well on positive\-direction yield, overlap support, and modality breadth\. The all\-three\-framework core remains a useful high\-confidence reference, but it is too restrictive to serve as the primary working surface on its own\. The full union remains available as a broader sensitivity view, but it is not the default because its added events degrade more sharply on directional quality and cross\-modal support\. The retained working surface for the next stage is therefore the 2\+ consensus event surface\.

### Choose the First\-Cut Retained Event Surfaces

This subsection compares the retained event surfaces as candidate base packages for the downstream positive\-direction target\. It reads overlap structure, positive\-direction yield, modality breadth, and geographic spread together so the notebook can choose the event surface that should feed the later positive\-direction working package\.

In [30]:
# ---------------------------------------------------------------------
# Score and rank the candidate retained event surfaces
# ---------------------------------------------------------------------

CANDIDATE_SURFACE_SCORECARD_PATH = (
    INTERMEDIATE_335_DIR / "candidate_event_surface_scorecard.parquet"
)
OVERLAP_REVIEW_SURFACE_RANKING_PATH = (
    INTERMEDIATE_335_DIR / "overlap_review_surface_ranking.parquet"
)

candidate_event_surface_scorecard_df = pd.read_parquet(
    CANDIDATE_SURFACE_SCORECARD_PATH
).copy()

decision_candidate_df = candidate_event_surface_scorecard_df.loc[
    candidate_event_surface_scorecard_df["candidate_surface_id"].isin(
        [
            "framework_union",
            "framework_consensus_2plus",
            "framework_consensus_3way",
        ]
    )
].copy()

decision_candidate_df["two_plus_or_better_support_share"] = (
    decision_candidate_df["two_framework_support_share"]
    + decision_candidate_df["three_framework_support_share"]
)

decision_candidate_df["positive_direction_rank"] = (
    decision_candidate_df["positive_direction_event_share"]
    .rank(method="dense", ascending=False)
)
decision_candidate_df["support_rank"] = (
    decision_candidate_df["two_plus_or_better_support_share"]
    .rank(method="dense", ascending=False)
)
decision_candidate_df["modality_breadth_rank"] = (
    decision_candidate_df["median_distinct_modality_count"]
    .rank(method="dense", ascending=False)
)
decision_candidate_df["metric_breadth_rank"] = (
    decision_candidate_df["median_distinct_metric_count"]
    .rank(method="dense", ascending=False)
)
decision_candidate_df["breadth_rank"] = (
    decision_candidate_df["event_count"]
    .rank(method="dense", ascending=False)
)

decision_candidate_df["first_cut_rank_score"] = (
    decision_candidate_df["positive_direction_rank"]
    + decision_candidate_df["support_rank"]
    + decision_candidate_df["modality_breadth_rank"]
    + decision_candidate_df["metric_breadth_rank"]
    + decision_candidate_df["breadth_rank"]
)

decision_candidate_df = decision_candidate_df.sort_values(
    [
        "first_cut_rank_score",
        "positive_direction_event_share",
        "two_plus_or_better_support_share",
        "median_distinct_modality_count",
        "event_count",
    ],
    ascending=[True, False, False, False, False],
).reset_index(drop=True)

decision_candidate_df.insert(
    0,
    "first_cut_rank",
    np.arange(1, len(decision_candidate_df) + 1),
)

if WRITE_335_OUTPUTS:
    decision_candidate_df.to_parquet(
        OVERLAP_REVIEW_SURFACE_RANKING_PATH,
        index=False,
    )

display(
    decision_candidate_df[
        [
            "first_cut_rank",
            "candidate_surface_label",
            "candidate_group",
            "event_count",
            "event_share_of_full_event_universe",
            "positive_direction_event_share",
            "two_plus_or_better_support_share",
            "median_distinct_metric_count",
            "median_distinct_modality_count",
            "first_cut_rank_score",
        ]
    ].style.format(
        {
            "event_share_of_full_event_universe": "{:.3f}",
            "positive_direction_event_share": "{:.3f}",
            "two_plus_or_better_support_share": "{:.3f}",
            "median_distinct_metric_count": "{:.1f}",
            "median_distinct_modality_count": "{:.1f}",
            "first_cut_rank_score": "{:.1f}",
        }
    )
)

,first_cut_rank,candidate_surface_label,candidate_group,event_count,event_share_of_full_event_universe,positive_direction_event_share,two_plus_or_better_support_share,median_distinct_metric_count,median_distinct_modality_count,first_cut_rank_score
0,1,All 3 retained frameworks,consensus,225524,0.145,0.689,1.000,2.0,1.0,7.0
1,2,2+ retained frameworks,consensus,465249,0.298,0.644,1.000,1.0,1.0,8.0
2,3,Any retained framework,broad,715321,0.459,0.592,0.650,1.0,1.0,9.0


Findings\. The first\-cut ranking table lands where the broader event\-surface review was already pointing\. The strict all\-three\-framework core ranks first on pure quality, with the strongest positive\-direction yield at 68\.9%, full multi\-framework support by definition, and the richest median event structure at two distinct metrics\. The 2\+ retained frameworks surface comes in a very close second: it roughly doubles event coverage from 14\.5% to 29\.8% of the full event universe while still preserving a strong 64\.4% positive\-direction share and full two\-or\-better framework support\. By contrast, Any retained framework is much broader at 45\.9% of the event universe, but that extra reach comes with a weaker directional profile and thinner event structure\. 

> 🎯 Decision\. We will carry forward both retained positive\-direction event surfaces into the next notebook rather than forcing a final choice here\. The broader 2\+ retained frameworks surface remains the working finalist because it expands coverage materially while preserving strong multi\-framework support, and the stricter All 3 retained frameworks surface remains the reference finalist because it concentrates the highest\-consensus core\. Keeping both surfaces alive gives the final evaluation stage a cleaner job: compare a broader\-but\-still\-disciplined candidate against a stricter consensus benchmark before naming the canonical event surface for downstream crash and policy analysis\.

Record the retained working and strict\-reference event surfaces\. This block carries forward two finalists\. The broader \`2\+ retained frameworks\` surface remains the working package because it preserves wider coverage, while the stricter \`All 3 retained frameworks\` surface is retained as the reference package for final evaluation in the next notebook\.

In [31]:
# ---------------------------------------------------------------------
# Record the retained working and strict-reference event surfaces
# ---------------------------------------------------------------------

candidate_event_surface_scorecard_df = pd.read_parquet(
    CANDIDATE_SURFACE_SCORECARD_PATH
).copy()

if "two_plus_or_better_support_share" not in candidate_event_surface_scorecard_df.columns:
    candidate_event_surface_scorecard_df["two_plus_or_better_support_share"] = (
        candidate_event_surface_scorecard_df["two_framework_support_share"]
        + candidate_event_surface_scorecard_df["three_framework_support_share"]
    )

retained_surface_candidates_df = (
    candidate_event_surface_scorecard_df.loc[
        candidate_event_surface_scorecard_df["candidate_surface_id"].isin(
            [
                "framework_consensus_2plus",
                "framework_consensus_3way",
            ]
        )
    ]
    .sort_values(
        [
            "positive_direction_event_share",
            "two_plus_or_better_support_share",
            "event_count",
        ],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

selected_working_surface_row = retained_surface_candidates_df.loc[
    retained_surface_candidates_df["candidate_surface_id"].eq(
        "framework_consensus_2plus"
    )
].iloc[0]
reference_surface_row = retained_surface_candidates_df.loc[
    retained_surface_candidates_df["candidate_surface_id"].eq(
        "framework_consensus_3way"
    )
].iloc[0]

retained_surface_selection_df = pd.DataFrame(
    [
        {
            "selected_surface_id": str(selected_working_surface_row["candidate_surface_id"]),
            "selected_surface_label": str(selected_working_surface_row["candidate_surface_label"]),
            "selected_event_count": int(selected_working_surface_row["event_count"]),
            "selected_event_share_of_full_event_universe": float(
                selected_working_surface_row["event_share_of_full_event_universe"]
            ),
            "selected_positive_direction_event_share": float(
                selected_working_surface_row["positive_direction_event_share"]
            ),
            "selected_two_plus_or_better_support_share": float(
                selected_working_surface_row["two_plus_or_better_support_share"]
            ),
            "selected_working_table_row_count": int(
                selected_working_surface_row["event_count"]
            ),
            "reference_surface_id": str(reference_surface_row["candidate_surface_id"]),
            "reference_surface_label": str(reference_surface_row["candidate_surface_label"]),
            "reference_event_count": int(reference_surface_row["event_count"]),
            "reference_positive_direction_event_share": float(
                reference_surface_row["positive_direction_event_share"]
            ),
            "reference_two_plus_or_better_support_share": float(
                reference_surface_row["two_plus_or_better_support_share"]
            ),
            "decision_stage": "event_surface_review",
            "decision_status": "advance_both_for_final_evaluation",
            "selection_rationale": (
                "Carry forward the broader 2+ retained-framework surface as the "
                "working finalist and the stricter all-3 retained-framework "
                "surface as the reference finalist so the next notebook can make "
                "the final positive-direction surface decision with both options in view."
            ),
        }
    ]
)

retained_working_surface_export_df = pd.read_parquet(CONSENSUS_SUMMARY_PATH).copy()
retained_working_surface_export_df = (
    retained_working_surface_export_df.loc[
        retained_working_surface_export_df["candidate_surface_id"].eq(
            "framework_consensus_2plus"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

if WRITE_335_OUTPUTS:
    retained_surface_selection_df.to_csv(
        RETAINED_SURFACE_SELECTION_PATH,
        index=False,
    )
    retained_working_surface_export_df.to_parquet(
        RETAINED_WORKING_EXPORT_PATH,
        index=False,
    )

display(
    retained_surface_candidates_df[
        [
            "candidate_surface_id",
            "candidate_surface_label",
            "event_count",
            "event_share_of_full_event_universe",
            "positive_direction_event_share",
            "two_plus_or_better_support_share",
            "median_distinct_metric_count",
            "median_distinct_modality_count",
        ]
    ].style.format(
        {
            "event_share_of_full_event_universe": "{:.3f}",
            "positive_direction_event_share": "{:.3f}",
            "two_plus_or_better_support_share": "{:.3f}",
            "median_distinct_metric_count": "{:.1f}",
            "median_distinct_modality_count": "{:.1f}",
        }
    )
)

display(
    retained_surface_selection_df.style.format(
        {
            "selected_event_share_of_full_event_universe": "{:.3f}",
            "selected_positive_direction_event_share": "{:.3f}",
            "selected_two_plus_or_better_support_share": "{:.3f}",
            "reference_positive_direction_event_share": "{:.3f}",
            "reference_two_plus_or_better_support_share": "{:.3f}",
        }
    )
)

print(
    "Selected working retained surface: "
    f"{selected_working_surface_row['candidate_surface_label']}"
)
print(
    "Strict-reference retained surface: "
    f"{reference_surface_row['candidate_surface_label']}"
)

,candidate_surface_id,candidate_surface_label,event_count,event_share_of_full_event_universe,positive_direction_event_share,two_plus_or_better_support_share,median_distinct_metric_count,median_distinct_modality_count
0,framework_consensus_3way,All 3 retained frameworks,225524,0.145,0.689,1.000,2.0,1.0
1,framework_consensus_2plus,2+ retained frameworks,465249,0.298,0.644,1.000,1.0,1.0


,selected_surface_id,selected_surface_label,selected_event_count,selected_event_share_of_full_event_universe,selected_positive_direction_event_share,selected_two_plus_or_better_support_share,selected_working_table_row_count,reference_surface_id,reference_surface_label,reference_event_count,reference_positive_direction_event_share,reference_two_plus_or_better_support_share,decision_stage,decision_status,selection_rationale
0,framework_consensus_2plus,2+ retained frameworks,465249,0.298,0.644,1.000,465249,framework_consensus_3way,All 3 retained frameworks,225524,0.689,1.000,event_surface_review,advance_both_for_final_evaluation,Carry forward the broader 2+ retained-framework surface as the working finalist and the stricter all-3 retained-framework surface as the reference finalist so the next notebook can make the final positive-direction surface decision with both options in view.


Selected working retained surface: 2+ retained frameworks
Strict-reference retained surface: All 3 retained frameworks


Findings\. This block makes the two retained event\-surface finalists explicit rather than collapsing the notebook to one winner too early\. The broader framework\_consensus\_2plus surface carries 465,249 event\-grain anomalies, covering 29\.8% of the full Taxi Zone × Date × Daypart universe, with 64\.4% of those events in the positive\-direction bucket and complete support from at least two retained frameworks by construction\. The stricter framework\_consensus\_3way surface preserves the highest\-consensus core at 225,524 events, covering 14\.5% of the universe with an even higher 68\.9% positive\-direction share\. Taken together, these two retained surfaces define the final evaluation frontier: a broader working finalist that materially expands coverage and a stricter reference finalist that preserves the tightest cross\-framework agreement\.

### Section Summary

This subsection compared the retained event surfaces as candidate base packages for the downstream positive\-direction target\. It showed that the All 3 retained frameworks surface preserves the strictest consensus core, while the broader 2\+ retained frameworks surface materially expands coverage without a major collapse in directionality, support quality, or geographic spread\.

That leaves the next subsection with a more focused but still two\-track job\. 3\.3\.5\.4 will carry both retained finalist surfaces forward, retain their positive\-direction event packages, compare their support, geography, and driver composition side by side, and export both in downstream\-ready form so the final surface choice can be made in the next notebook\.

## 3\.3\.5\.4 Retain and Compare the Positive\-Direction Finalist Surfaces

This section narrows both retained event\-surface finalists to their positive\-direction event packages and compares them before the pipeline makes a final commitment\. The broader \`2\+ retained frameworks\` package stays in view because it carries more breadth, while the stricter \`All 3 retained frameworks\` package stays in view because it carries stronger consensus\. The goal here is to preserve that tradeoff explicitly and hand both finalists forward in a downstream\-ready form\.

### Build the Positive\-Direction Finalist Packages

This subsection materializes the positive\-direction versions of both retained event\-surface finalists\. It first constructs the broader \`2\+ retained frameworks\` positive\-direction package and the stricter \`All 3 retained frameworks\` positive\-direction package, then summarizes their size and internal profile mix so the next subsection can compare them directly\.

Materialize the retained positive\-direction finalist surfaces\. This block builds the positive\-direction event packages for both retained finalists: the broader \`2\+ retained frameworks\` working surface and the stricter \`All 3 retained frameworks\` reference surface\. That keeps the next notebook open to a real final evaluation rather than forcing the choice here\.

In [32]:
# ---------------------------------------------------------------------
# Materialize the retained positive-direction finalist surfaces
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"
EVENT_DIRECTIONALITY_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "event_directionality_summary.parquet"
)

selected_surface_decision_df = pd.read_csv(RETAINED_SURFACE_SELECTION_PATH).copy()
selected_surface_id = str(selected_surface_decision_df.loc[0, "selected_surface_id"])
reference_surface_id = str(selected_surface_decision_df.loc[0, "reference_surface_id"])

retained_event_spine_df = pd.read_parquet(CONSENSUS_ROW_LEVEL_PATH).copy()
consensus_summary_df = pd.read_parquet(CONSENSUS_SUMMARY_PATH).copy()
directional_row_level_df = pd.read_parquet(DIRECTIONAL_ROW_LEVEL_PATH).copy()

rebuild_positive_surface_outputs = (
    REBUILD_335_CONSENSUS
    or REBUILD_335_DIRECTIONALITY
    or not EVENT_DIRECTIONALITY_SUMMARY_PATH.exists()
    or not SELECTED_POSITIVE_EVENT_SURFACE_2PLUS_PATH.exists()
    or not SELECTED_POSITIVE_EVENT_SURFACE_3WAY_PATH.exists()
    or not POSITIVE_DIRECTION_SURFACE_CANDIDATE_SUMMARY_PATH.exists()
    or not POSITIVE_DIRECTION_SURFACE_PROFILE_BREAKDOWN_PATH.exists()
)

if rebuild_positive_surface_outputs:
    event_directionality_summary_df = (
        directional_row_level_df.groupby(EVENT_ID_COLUMN, dropna=False)
        .agg(
            has_congestion_oriented=(
                "directional_label",
                lambda s: bool((s == "congestion_oriented").any()),
            ),
            has_positive_demand_shock=(
                "directional_label",
                lambda s: bool((s == "positive_demand_shock").any()),
            ),
            has_decongestion_oriented=(
                "directional_label",
                lambda s: bool((s == "decongestion_oriented").any()),
            ),
            has_negative_demand_shock=(
                "directional_label",
                lambda s: bool((s == "negative_demand_shock").any()),
            ),
            has_ambiguous_directional_row=(
                "directional_label",
                lambda s: bool((s == "ambiguous").any()),
            ),
            event_max_directional_strength=("directional_strength", "max"),
            event_median_directional_strength=("directional_strength", "median"),
        )
        .reset_index()
    )

    event_directionality_summary_df["positive_direction_event_flag"] = (
        event_directionality_summary_df["has_congestion_oriented"]
        | event_directionality_summary_df["has_positive_demand_shock"]
    )
    event_directionality_summary_df["negative_direction_event_flag"] = (
        event_directionality_summary_df["has_decongestion_oriented"]
        | event_directionality_summary_df["has_negative_demand_shock"]
    )
    event_directionality_summary_df["both_positive_family_event_flag"] = (
        event_directionality_summary_df["has_congestion_oriented"]
        & event_directionality_summary_df["has_positive_demand_shock"]
    )
    event_directionality_summary_df["mixed_direction_event_flag"] = (
        event_directionality_summary_df["positive_direction_event_flag"]
        & event_directionality_summary_df["negative_direction_event_flag"]
    )
    event_directionality_summary_df["positive_only_event_flag"] = (
        event_directionality_summary_df["positive_direction_event_flag"]
        & ~event_directionality_summary_df["negative_direction_event_flag"]
    )
    event_directionality_summary_df["negative_only_event_flag"] = (
        event_directionality_summary_df["negative_direction_event_flag"]
        & ~event_directionality_summary_df["positive_direction_event_flag"]
    )

    event_directionality_summary_df["positive_direction_profile"] = np.select(
        [
            event_directionality_summary_df["mixed_direction_event_flag"],
            event_directionality_summary_df["both_positive_family_event_flag"],
            event_directionality_summary_df["has_congestion_oriented"],
            event_directionality_summary_df["has_positive_demand_shock"],
        ],
        [
            "mixed_positive_and_negative",
            "both_positive_families",
            "congestion_only",
            "positive_demand_only",
        ],
        default="not_positive_direction",
    )

    candidate_surface_specs = [
        {
            "candidate_surface_id": "framework_consensus_2plus",
            "candidate_surface_label": "2+ retained frameworks",
            "output_path": SELECTED_POSITIVE_EVENT_SURFACE_2PLUS_PATH,
        },
        {
            "candidate_surface_id": "framework_consensus_3way",
            "candidate_surface_label": "All 3 retained frameworks",
            "output_path": SELECTED_POSITIVE_EVENT_SURFACE_3WAY_PATH,
        },
    ]

    positive_surface_candidate_frames = []
    positive_surface_summary_records = []
    positive_surface_profile_records = []

    for spec in candidate_surface_specs:
        candidate_event_ids_df = (
            consensus_summary_df.loc[
                consensus_summary_df["candidate_surface_id"].eq(spec["candidate_surface_id"]),
                [EVENT_ID_COLUMN],
            ]
            .drop_duplicates()
            .assign(candidate_surface_id=spec["candidate_surface_id"])
            .assign(candidate_surface_label=spec["candidate_surface_label"])
        )

        candidate_positive_surface_df = (
            retained_event_spine_df.merge(
                candidate_event_ids_df,
                on=EVENT_ID_COLUMN,
                how="inner",
                validate="one_to_one",
            )
            .merge(
                event_directionality_summary_df,
                on=EVENT_ID_COLUMN,
                how="left",
                validate="one_to_one",
            )
            .loc[lambda df: df["positive_direction_event_flag"]]
            .copy()
            .reset_index(drop=True)
        )

        positive_surface_candidate_frames.append(candidate_positive_surface_df)

        candidate_event_count = int(len(candidate_positive_surface_df))
        candidate_full_surface_count = int(len(candidate_event_ids_df))
        full_event_universe_count = int(len(retained_event_spine_df))

        positive_surface_summary_records.append(
            {
                "candidate_surface_id": spec["candidate_surface_id"],
                "candidate_surface_label": spec["candidate_surface_label"],
                "positive_event_count": candidate_event_count,
                "positive_event_share_of_candidate_surface": safe_share(
                    candidate_event_count,
                    candidate_full_surface_count,
                ),
                "positive_event_share_of_full_event_universe": safe_share(
                    candidate_event_count,
                    full_event_universe_count,
                ),
                "congestion_positive_event_count": int(
                    candidate_positive_surface_df["has_congestion_oriented"].sum()
                ),
                "positive_demand_event_count": int(
                    candidate_positive_surface_df["has_positive_demand_shock"].sum()
                ),
                "both_positive_family_event_count": int(
                    candidate_positive_surface_df["both_positive_family_event_flag"].sum()
                ),
                "mixed_direction_event_count": int(
                    candidate_positive_surface_df["mixed_direction_event_flag"].sum()
                ),
            }
        )

        for positive_direction_profile, profile_df in candidate_positive_surface_df.groupby(
            "positive_direction_profile",
            dropna=False,
        ):
            positive_surface_profile_records.append(
                {
                    "candidate_surface_id": spec["candidate_surface_id"],
                    "candidate_surface_label": spec["candidate_surface_label"],
                    "positive_direction_profile": positive_direction_profile,
                    "event_count": int(len(profile_df)),
                    "event_share_within_positive_package": safe_share(
                        len(profile_df),
                        candidate_event_count,
                    ),
                }
            )

        if WRITE_335_OUTPUTS:
            candidate_positive_surface_df.to_parquet(spec["output_path"], index=False)

    positive_surface_candidate_summary_df = pd.DataFrame(
        positive_surface_summary_records
    )
    positive_surface_profile_breakdown_df = pd.DataFrame(
        positive_surface_profile_records
    )

    if WRITE_335_OUTPUTS:
        event_directionality_summary_df.to_parquet(
            EVENT_DIRECTIONALITY_SUMMARY_PATH,
            index=False,
        )
        positive_surface_candidate_summary_df.to_parquet(
            POSITIVE_DIRECTION_SURFACE_CANDIDATE_SUMMARY_PATH,
            index=False,
        )
        positive_surface_profile_breakdown_df.to_parquet(
            POSITIVE_DIRECTION_SURFACE_PROFILE_BREAKDOWN_PATH,
            index=False,
        )
else:
    event_directionality_summary_df = pd.read_parquet(
        EVENT_DIRECTIONALITY_SUMMARY_PATH
    ).copy()
    selected_positive_event_surface_2plus_df = pd.read_parquet(
        SELECTED_POSITIVE_EVENT_SURFACE_2PLUS_PATH
    ).copy()
    selected_positive_event_surface_3way_df = pd.read_parquet(
        SELECTED_POSITIVE_EVENT_SURFACE_3WAY_PATH
    ).copy()
    positive_surface_candidate_summary_df = pd.read_parquet(
        POSITIVE_DIRECTION_SURFACE_CANDIDATE_SUMMARY_PATH
    ).copy()
    positive_surface_profile_breakdown_df = pd.read_parquet(
        POSITIVE_DIRECTION_SURFACE_PROFILE_BREAKDOWN_PATH
    ).copy()

summary_columns = [
    column
    for column in [
        "candidate_surface_id",
        "candidate_surface_label",
        "positive_event_count",
        "positive_event_share_of_full_event_universe",
        "positive_event_share_of_candidate_surface",
    ]
    if column in positive_surface_candidate_summary_df.columns
]

preview_columns = [
    column
    for column in [
        "candidate_surface_label",
        EVENT_ID_COLUMN,
        ZONE_COLUMN,
        BOROUGH_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
        "framework_anomaly_count",
        "framework_support_pattern",
        "positive_direction_profile",
    ]
    if column in selected_positive_event_surface_2plus_df.columns
]

preview_df = pd.concat(
    [
        selected_positive_event_surface_2plus_df.assign(
            candidate_surface_label="2+ retained frameworks"
        )[preview_columns].head(4),
        selected_positive_event_surface_3way_df.assign(
            candidate_surface_label="All 3 retained frameworks"
        )[preview_columns].head(4),
    ],
    ignore_index=True,
)

display(
    positive_surface_candidate_summary_df[summary_columns].style.format(
        {
            column: "{:.3f}"
            for column in [
                "positive_event_share_of_full_event_universe",
                "positive_event_share_of_candidate_surface",
            ]
            if column in positive_surface_candidate_summary_df.columns
        }
    )
)

display(preview_df)

print(
    "Retained positive-direction finalists: broader package rows = "
    f"{int(positive_surface_candidate_summary_df['positive_event_count'].max()):,}; "
    "stricter package rows = "
    f"{int(positive_surface_candidate_summary_df['positive_event_count'].min()):,}"
)

,candidate_surface_id,candidate_surface_label,positive_event_count,positive_event_share_of_full_event_universe,positive_event_share_of_candidate_surface
0,framework_consensus_3way,All 3 retained frameworks,155353,0.100,0.689
1,framework_consensus_2plus,2+ retained frameworks,299439,0.192,0.644


,candidate_surface_label,comparison_event_id,zone,borough,date,daypart,framework_anomaly_count,framework_support_pattern,positive_direction_profile
0,2+ retained frameworks,13 | 2023-01-01 | overnight,Battery Park City,Manhattan,2023-01-01,overnight,3,DBSCAN + Isolation Forest + GMM,mixed_positive_and_negative
1,2+ retained frameworks,28 | 2023-01-01 | overnight,Briarwood/Jamaica Hills,Queens,2023-01-01,overnight,3,DBSCAN + Isolation Forest + GMM,mixed_positive_and_negative
2,2+ retained frameworks,34 | 2023-01-01 | overnight,Brooklyn Navy Yard,Brooklyn,2023-01-01,overnight,3,DBSCAN + Isolation Forest + GMM,positive_demand_only
3,2+ retained frameworks,40 | 2023-01-01 | overnight,Carroll Gardens,Brooklyn,2023-01-01,overnight,3,DBSCAN + Isolation Forest + GMM,mixed_positive_and_negative
4,All 3 retained frameworks,13 | 2023-01-01 | overnight,Battery Park City,Manhattan,2023-01-01,overnight,3,DBSCAN + Isolation Forest + GMM,mixed_positive_and_negative
5,All 3 retained frameworks,28 | 2023-01-01 | overnight,Briarwood/Jamaica Hills,Queens,2023-01-01,overnight,3,DBSCAN + Isolation Forest + GMM,mixed_positive_and_negative
6,All 3 retained frameworks,34 | 2023-01-01 | overnight,Brooklyn Navy Yard,Brooklyn,2023-01-01,overnight,3,DBSCAN + Isolation Forest + GMM,positive_demand_only
7,All 3 retained frameworks,40 | 2023-01-01 | overnight,Carroll Gardens,Brooklyn,2023-01-01,overnight,3,DBSCAN + Isolation Forest + GMM,mixed_positive_and_negative


Retained positive-direction finalists: broader package rows = 299,439; stricter package rows = 155,353


Findings\. This block materializes two retained positive\-direction finalist packages for downstream evaluation\. The 2\+ retained frameworks finalist contains 299,439 positive\-direction events, while the All 3 retained frameworks finalist contains 155,353\. Together, these packages define the final evaluation frontier: one finalist captures a wider positive\-direction event surface, and the other preserves the tightest cross\-framework core\.

Summarize how large the retained positive\-direction package\. Once the selected \`2\+\` event surface is narrowed to business\-relevant anomaly states\. This block makes the retained size, composition, and positive\-direction profile explicit before we inspect support structure or geography\.

In [33]:
# ---------------------------------------------------------------------
# Summarize the retained positive-direction finalist packages
# ---------------------------------------------------------------------

positive_surface_candidate_summary_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_CANDIDATE_SUMMARY_PATH
).copy()
positive_surface_profile_breakdown_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_PROFILE_BREAKDOWN_PATH
).copy()

summary_columns = [
    column
    for column in [
        "candidate_surface_id",
        "candidate_surface_label",
        "positive_event_count",
        "positive_event_share_of_full_event_universe",
        "positive_event_share_of_candidate_surface",
        "congestion_positive_event_count",
        "positive_demand_event_count",
        "both_positive_family_event_count",
        "mixed_direction_event_count",
    ]
    if column in positive_surface_candidate_summary_df.columns
]

summary_format_map = {
    column: "{:.3f}"
    for column in [
        "positive_event_share_of_full_event_universe",
        "positive_event_share_of_candidate_surface",
    ]
    if column in positive_surface_candidate_summary_df.columns
}

display(
    positive_surface_candidate_summary_df[summary_columns].style.format(
        summary_format_map
    )
)

print(
    "Positive-direction finalists: 2 | broader package rows: "
    f"{int(positive_surface_candidate_summary_df['positive_event_count'].max()):,} | "
    "stricter package rows: "
    f"{int(positive_surface_candidate_summary_df['positive_event_count'].min()):,}"
)

,candidate_surface_id,candidate_surface_label,positive_event_count,positive_event_share_of_full_event_universe,positive_event_share_of_candidate_surface,congestion_positive_event_count,positive_demand_event_count,both_positive_family_event_count,mixed_direction_event_count
0,framework_consensus_3way,All 3 retained frameworks,155353,0.100,0.689,102904,80954,28505,57135
1,framework_consensus_2plus,2+ retained frameworks,299439,0.192,0.644,194010,148652,43223,92893


Positive-direction finalists: 2 | broader package rows: 299,439 | stricter package rows: 155,353


Findings\. The two finalist packages are clearly differentiated in breadth while remaining comparable in overall structure\. The 2\+ retained frameworks package provides the larger citywide positive\-direction surface, and the All 3 retained frameworks package defines the tighter high\-consensus core\. These two packages now serve as the final candidates for downstream review\.

### Compare Support, Geography, and Drivers Across the Positive\-Direction Finalists

This subsection compares the two retained positive\-direction finalists side by side rather than profiling only the broader working package\. It asks whether the broader \`2\+ retained frameworks\` package remains close enough to the stricter \`All 3 retained frameworks\` package on support composition, geographic spread, and modality\-driver structure to justify carrying both into final evaluation\.

Compare framework support across the retained positive\-direction finalists\. This block measures how the two positive\-direction finalist surfaces distribute across two\-framework versus three\-framework support\. That gives us a clean read on how much broader the \`2\+ retained frameworks\` package really is and whether that broader reach still remains anchored in strong cross\-method agreement\.

In [34]:
# ---------------------------------------------------------------------
# Compare framework support across the retained positive-direction finalists
# ---------------------------------------------------------------------

positive_surface_support_summary_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_SUPPORT_SUMMARY_PATH
).copy()
positive_surface_support_pattern_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_SUPPORT_PATTERN_PATH
).copy()

display(
    positive_surface_support_summary_df.style.format(
        {"event_share_within_positive_package": "{:.3f}"}
    )
)

,candidate_surface_label,candidate_surface_role,framework_anomaly_count,event_count,event_share_within_positive_package
0,2+ retained frameworks,working_broad_surface,2,144086,0.481
1,2+ retained frameworks,working_broad_surface,3,155353,0.519
2,All 3 retained frameworks,strict_reference_surface,3,155353,1.000


Findings\. Framework support remains strong across both finalists\. The All 3 retained frameworks package is fully composed of three\-framework events by definition\. Within the 2\+ retained frameworks package, 52% of positive\-direction events are supported by all three frameworks and 48% are supported by exactly two\. That gives the wider finalist a substantial three\-framework core plus an additional two\-framework layer\.

Compare geography across the retained positive\-direction finalists\. This block tests whether the broader \`2\+ retained frameworks\` package stays geographically close to the stricter all\-three package or whether broader coverage comes with a more concentrated spatial footprint\. It compares citywide coverage, borough mix, and top\-zone concentration side by side\.

In [35]:
# ---------------------------------------------------------------------
# Compare geography across the retained positive-direction finalists
# ---------------------------------------------------------------------

positive_surface_geography_summary_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_GEOGRAPHY_SUMMARY_PATH
).copy()
positive_surface_borough_mix_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_BOROUGH_MIX_PATH
).copy()
positive_surface_zone_summary_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_ZONE_SUMMARY_PATH
).copy()

display(
    positive_surface_geography_summary_df.style.format(
        {
            "taxi_zone_coverage_share": "{:.3f}",
            "borough_coverage_share": "{:.3f}",
            "top_zone_share": "{:.3f}",
            "top10_zone_share": "{:.3f}",
            "top_borough_share": "{:.3f}",
        }
    )
)

display(
    positive_surface_borough_mix_df.style.format(
        {"event_share_within_positive_package": "{:.3f}"}
    )
)

,candidate_surface_label,candidate_surface_role,event_count,anomalous_taxi_zone_count,taxi_zone_coverage_share,anomalous_borough_count,borough_coverage_share,top_zone,top_zone_share,top10_zone_share,top_borough,top_borough_share
0,All 3 retained frameworks,strict_reference_surface,155353,262,0.996,7,1.000,East Chelsea,0.008,0.060,Queens,0.280
1,2+ retained frameworks,working_broad_surface,299439,263,1.000,7,1.000,East Chelsea,0.011,0.064,Queens,0.280


,candidate_surface_label,candidate_surface_role,borough,event_count,event_share_within_surface
0,2+ retained frameworks,working_broad_surface,Bronx,48610,0.162337
1,2+ retained frameworks,working_broad_surface,Brooklyn,69323,0.231510
2,2+ retained frameworks,working_broad_surface,EWR,282,0.000942
3,2+ retained frameworks,working_broad_surface,Manhattan,75235,0.251253
4,2+ retained frameworks,working_broad_surface,Queens,83699,0.279519
5,2+ retained frameworks,working_broad_surface,Staten Island,21199,0.070796
6,2+ retained frameworks,working_broad_surface,Unknown,1091,0.003643
7,All 3 retained frameworks,strict_reference_surface,Bronx,24006,0.154525
8,All 3 retained frameworks,strict_reference_surface,Brooklyn,35908,0.231138
9,All 3 retained frameworks,strict_reference_surface,EWR,158,0.001017


Findings\. Both retained positive\-direction finalists remain citywide in geographic reach\. Each package spans essentially the full Taxi Zone footprint and preserves a similar borough distribution, with Queens holding the largest share, followed by Manhattan and Brooklyn\. Geography therefore remains stable across the finalist comparison\.

Compare driver composition across the retained positive\-direction finalists\. This block measures whether the broader \`2\+ retained frameworks\` package preserves the same overall modality and metric structure as the stricter all\-three package or whether it changes the substantive character of the retained event surface\.

In [36]:
# ---------------------------------------------------------------------
# Compare driver composition across the retained positive-direction finalists
# ---------------------------------------------------------------------

positive_surface_driver_breadth_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_DRIVER_BREADTH_PATH
).copy()
positive_surface_modality_summary_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_MODALITY_SUMMARY_PATH
).copy()
positive_surface_metric_summary_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_METRIC_SUMMARY_PATH
).copy()

display(
    positive_surface_driver_breadth_df.style.format(
        {
            "median_distinct_metric_count": "{:.1f}",
            "p90_distinct_metric_count": "{:.1f}",
            "median_distinct_modality_count": "{:.1f}",
            "p90_distinct_modality_count": "{:.1f}",
        }
    )
)

display(
    positive_surface_modality_summary_df.style.format({"event_share": "{:.3f}"})
)

,candidate_surface_label,candidate_surface_role,positive_event_count,median_distinct_metric_count,p90_distinct_metric_count,median_distinct_modality_count,p90_distinct_modality_count
0,2+ retained frameworks,working_broad_surface,299439,2.0,3.0,1.0,2.0
1,All 3 retained frameworks,strict_reference_surface,155353,2.0,3.0,1.0,2.0


,candidate_surface_label,candidate_surface_role,modality,event_count,event_share
0,2+ retained frameworks,working_broad_surface,Bus,148677,0.338
1,2+ retained frameworks,working_broad_surface,FHVHV,119066,0.271
2,2+ retained frameworks,working_broad_surface,Subway,21805,0.050
3,2+ retained frameworks,working_broad_surface,Taxi,149744,0.341
4,All 3 retained frameworks,strict_reference_surface,Bus,84612,0.343
5,All 3 retained frameworks,strict_reference_surface,FHVHV,65137,0.264
6,All 3 retained frameworks,strict_reference_surface,Subway,12257,0.050
7,All 3 retained frameworks,strict_reference_surface,Taxi,84837,0.344


Findings\. Both finalists preserve a multi\-driver positive\-direction structure\. The All 3 retained frameworks package carries somewhat more mixed\-sign and both\-positive\-family structure, while the 2\+ retained frameworks package carries somewhat larger congestion\-only and positive\-demand\-only shares\. Even with those differences, both packages remain interpretable as coherent positive\-direction event surfaces\.

Visualize the retained positive\-direction finalist comparison\. This block turns the two finalist packages into a visual comparison of support composition and internal positive\-direction profile mix\. The point is not to declare the winner here, but to make the breadth\-versus\-strictness tradeoff easy to read before the next notebook performs the deeper evaluation\.

In [37]:
# ---------------------------------------------------------------------
# Visualize the retained positive-direction finalist comparison
# ---------------------------------------------------------------------

positive_surface_support_summary_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_SUPPORT_SUMMARY_PATH
).copy()
positive_surface_profile_breakdown_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_PROFILE_BREAKDOWN_PATH
).copy()

support_plot_df = positive_surface_support_summary_df.copy()
support_plot_df["support_level"] = support_plot_df["framework_anomaly_count"].map(
    {
        2: "2 retained frameworks",
        3: "3 retained frameworks",
    }
)

support_color_map = {
    "2 retained frameworks": BRAND_COLORS["seafoam"],
    "3 retained frameworks": BRAND_COLORS["dark_teal"],
}

profile_color_map = {
    "congestion_only": BRAND_COLORS["terracotta"],
    "positive_demand_only": BRAND_COLORS["seafoam"],
    "both_positive_families": BRAND_COLORS["dark_teal"],
    "mixed_positive_and_negative": BRAND_COLORS["pale_peach"],
}

support_fig = px.bar(
    support_plot_df,
    x="candidate_surface_label",
    y="event_share_within_positive_package",
    color="support_level",
    barmode="stack",
    text="event_share_within_positive_package",
    color_discrete_map=support_color_map,
)

support_fig.update_traces(texttemplate="%{text:.0%}", textposition="inside")
apply_brand_plotly_layout(
    support_fig,
    title="Framework Support Composition Across Positive-Direction Finalists",
    width=900,
    height=460,
    legend_title="Support level",
    margin={"l": 70, "r": 30, "t": 90, "b": 90},
)
support_fig.update_xaxes(title_text=None)
support_fig.update_yaxes(title_text="Share within positive package", tickformat=".0%")
support_fig.show()

profile_fig = px.bar(
    positive_surface_profile_breakdown_df,
    x="candidate_surface_label",
    y="event_share_within_positive_package",
    color="positive_direction_profile",
    barmode="stack",
    text="event_share_within_positive_package",
    color_discrete_map=profile_color_map,
)

profile_fig.update_traces(texttemplate="%{text:.0%}", textposition="inside")
apply_brand_plotly_layout(
    profile_fig,
    title="Positive-Direction Profile Mix Across Finalist Surfaces",
    width=900,
    height=500,
    legend_title="Positive-direction profile",
    margin={"l": 70, "r": 30, "t": 90, "b": 90},
)
profile_fig.update_xaxes(title_text=None)
profile_fig.update_yaxes(title_text="Share within positive package", tickformat=".0%")
profile_fig.show()

Findings\. The comparison charts make the finalist structure easy to read\. The support chart shows that the 2\+ retained frameworks package contains both a substantial three\-framework core and a sizable two\-framework expansion layer, while the All 3 retained frameworks package is entirely three\-framework by construction\. The profile\-mix chart shows moderate compositional differences across the two finalists while preserving the same core positive\-direction families\.

In [38]:
# ---------------------------------------------------------------------
# Audit the authoritative metric-level positive-direction source contract
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"

directional_row_level_contract_df = pd.read_parquet(
    DIRECTIONAL_ROW_LEVEL_PATH
).copy()
directional_row_level_contract_df[DATE_COLUMN] = pd.to_datetime(
    directional_row_level_contract_df[DATE_COLUMN],
    errors="coerce",
)

if EVENT_ID_COLUMN not in directional_row_level_contract_df.columns:
    directional_row_level_contract_df[EVENT_ID_COLUMN] = (
        directional_row_level_contract_df[ZONE_ID_COLUMN].astype(str)
        + " | "
        + directional_row_level_contract_df[DATE_COLUMN].dt.strftime("%Y-%m-%d")
        + " | "
        + directional_row_level_contract_df[DAYPART_COLUMN].astype(str)
    )

event_metric_key_columns = [EVENT_ID_COLUMN, METRIC_COLUMN]
event_metric_framework_key_columns = [
    EVENT_ID_COLUMN,
    METRIC_COLUMN,
    "framework_name",
]

duplicate_event_metric_count = int(
    directional_row_level_contract_df.duplicated(
        subset=event_metric_key_columns
    ).sum()
)
duplicate_event_metric_framework_count = int(
    directional_row_level_contract_df.duplicated(
        subset=event_metric_framework_key_columns
    ).sum()
)

frameworks_per_event_metric_df = (
    directional_row_level_contract_df.groupby(
        event_metric_key_columns,
        dropna=False,
    )
    .agg(
        framework_row_count=("framework_name", "size"),
        supporting_framework_count=("framework_name", pd.Series.nunique),
        directional_label_count=("directional_label", pd.Series.nunique),
    )
    .reset_index()
)

directional_row_level_contract_summary_df = pd.DataFrame(
    [
        {
            "source_path": str(DIRECTIONAL_ROW_LEVEL_PATH),
            "row_count": int(len(directional_row_level_contract_df)),
            "distinct_event_count": int(
                directional_row_level_contract_df[EVENT_ID_COLUMN].nunique()
            ),
            "distinct_metric_rows": int(
                directional_row_level_contract_df[event_metric_key_columns]
                .drop_duplicates()
                .shape[0]
            ),
            "duplicate_count_at_event_metric": duplicate_event_metric_count,
            "duplicate_count_at_event_metric_framework": duplicate_event_metric_framework_count,
            "event_metric_rows_with_multiple_frameworks": int(
                frameworks_per_event_metric_df["supporting_framework_count"].gt(1).sum()
            ),
            "max_frameworks_for_single_event_metric": int(
                frameworks_per_event_metric_df["supporting_framework_count"].max()
            ),
        }
    ]
)

directional_row_level_schema_df = pd.DataFrame(
    {
        "column_name": directional_row_level_contract_df.columns,
        "dtype": directional_row_level_contract_df.dtypes.astype(str).values,
    }
)

directional_label_inventory_df = pd.DataFrame(
    {
        "directional_label": sorted(
            directional_row_level_contract_df["directional_label"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )
    }
)

directional_row_level_presence_df = pd.DataFrame(
    [
        {
            "framework_name_column_present": "framework_name"
            in directional_row_level_contract_df.columns,
            "feature_set_column_present": "feature_set"
            in directional_row_level_contract_df.columns,
            "residual_zscore_present": RESIDUAL_ZSCORE_COLUMN
            in directional_row_level_contract_df.columns,
            "directional_strength_present": "directional_strength"
            in directional_row_level_contract_df.columns,
            "support_status_present": SUPPORT_STATUS_COLUMN
            in directional_row_level_contract_df.columns,
            "support_review_flag_present": SUPPORT_REVIEW_COLUMN
            in directional_row_level_contract_df.columns,
            "anomaly_flag_present": "anomaly_flag"
            in directional_row_level_contract_df.columns,
        }
    ]
)

display(directional_row_level_contract_summary_df)
display(directional_row_level_presence_df)
display(directional_label_inventory_df)
display(directional_row_level_schema_df)

print(
    "Candidate metric-level reconciliation grain: "
    "comparison_event_id x metric. "
    "Multiple framework rows per event-metric are expected when more than one "
    "framework contributes retained directional evidence."
)


,source_path,row_count,distinct_event_count,distinct_metric_rows,duplicate_count_at_event_metric,duplicate_count_at_event_metric_framework,event_metric_rows_with_multiple_frameworks,max_frameworks_for_single_event_metric
0,/datasets/_deepnote_work/pipeline_data/3.3.5.intermediate_tables/directional_row_level.parquet,1866745,715321,1031344,835401,0,590934,3


,framework_name_column_present,feature_set_column_present,residual_zscore_present,directional_strength_present,support_status_present,support_review_flag_present,anomaly_flag_present
0,True,True,True,True,False,True,True


,directional_label
0,ambiguous
1,congestion_oriented
2,decongestion_oriented
3,negative_demand_shock
4,positive_demand_shock


,column_name,dtype
0,comparison_row_id,object
1,feature_set,object
2,date,datetime64[us]
3,daypart,object
4,temporal_bucket,object
5,taxi_zone_id,int64
6,zone,object
7,borough,object
8,comparison_group_support_review_flag,boolean
9,metric,object


Candidate metric-level reconciliation grain: comparison_event_id x metric. Multiple framework rows per event-metric are expected when more than one framework contributes retained directional evidence.


### Write the Downstream Stress\-Direction Finalist Handoff

This subsection writes the downstream\-ready stress\-direction finalist package in both full\-universe and filtered forms\. It preserves one full Taxi Zone × Date × Daypart event universe with both finalist indicators attached, and it also exports the broader \`2\+ retained frameworks\` and stricter \`All 3 retained frameworks\` stress\-direction packages separately so the next notebook can decide between them without rebuilding the synthesis layer\.

Export the downstream stress\-direction finalist package\. This block writes the full event universe once, then exports both retained stress\-direction finalists: the broader \`2\+ retained frameworks\` working package and the stricter \`All 3 retained frameworks\` reference package\. That gives the next notebook everything it needs to decide between the two finalists without re\-deriving the event\-grain synthesis logic\.

In [39]:
# ---------------------------------------------------------------------
# Export the downstream positive-direction finalist package
# ---------------------------------------------------------------------

EVENT_ID_COLUMN = "comparison_event_id"
EVENT_DIRECTIONALITY_SUMMARY_PATH = (
    INTERMEDIATE_335_DIR / "event_directionality_summary.parquet"
)
STRESS_ALIGNED_DIRECTIONAL_LABELS = [
    "congestion_oriented",
    "positive_demand_shock",
]


def classify_metric_modality(metric_name):
    metric_name = str(metric_name)
    if metric_name.startswith("bus_") or metric_name == "avg_bus_speed":
        return "Bus"
    if metric_name.startswith("subway_"):
        return "Subway"
    if metric_name.startswith("taxi_"):
        return "Taxi"
    if metric_name.startswith("fhvhv_"):
        return "FHVHV"
    return "Other"


selected_surface_decision_df = pd.read_csv(RETAINED_SURFACE_SELECTION_PATH).copy()
selected_surface_id = str(selected_surface_decision_df.loc[0, "selected_surface_id"])
selected_surface_label = str(selected_surface_decision_df.loc[0, "selected_surface_label"])
reference_surface_id = str(selected_surface_decision_df.loc[0, "reference_surface_id"])
reference_surface_label = str(selected_surface_decision_df.loc[0, "reference_surface_label"])

retained_event_spine_df = pd.read_parquet(CONSENSUS_ROW_LEVEL_PATH).copy()
consensus_summary_df = pd.read_parquet(CONSENSUS_SUMMARY_PATH).copy()
event_directionality_summary_df = pd.read_parquet(
    EVENT_DIRECTIONALITY_SUMMARY_PATH
).copy()
event_driver_evidence_df = pd.read_parquet(EVENT_DRIVER_EVIDENCE_PATH).copy()
positive_surface_candidate_summary_df = pd.read_parquet(
    POSITIVE_DIRECTION_SURFACE_CANDIDATE_SUMMARY_PATH
).copy()
selected_positive_event_surface_2plus_df = pd.read_parquet(
    SELECTED_POSITIVE_EVENT_SURFACE_2PLUS_PATH
).copy()
selected_positive_event_surface_3way_df = pd.read_parquet(
    SELECTED_POSITIVE_EVENT_SURFACE_3WAY_PATH
).copy()

directional_row_level_df = pd.read_parquet(DIRECTIONAL_ROW_LEVEL_PATH).copy()
directional_row_level_df[DATE_COLUMN] = pd.to_datetime(
    directional_row_level_df[DATE_COLUMN],
    errors="coerce",
)
if EVENT_ID_COLUMN not in directional_row_level_df.columns:
    directional_row_level_df[EVENT_ID_COLUMN] = (
        directional_row_level_df[ZONE_ID_COLUMN].astype(str)
        + " | "
        + directional_row_level_df[DATE_COLUMN].dt.strftime("%Y-%m-%d")
        + " | "
        + directional_row_level_df[DAYPART_COLUMN].astype(str)
    )

selected_working_event_ids_df = (
    consensus_summary_df.loc[
        consensus_summary_df["candidate_surface_id"].eq(selected_surface_id),
        [EVENT_ID_COLUMN],
    ]
    .drop_duplicates()
    .assign(selected_working_surface_flag=True)
)

reference_surface_event_ids_df = (
    consensus_summary_df.loc[
        consensus_summary_df["candidate_surface_id"].eq(reference_surface_id),
        [EVENT_ID_COLUMN],
    ]
    .drop_duplicates()
    .assign(strict_reference_surface_flag=True)
)

selected_positive_event_ids_2plus_df = (
    selected_positive_event_surface_2plus_df[[EVENT_ID_COLUMN]]
    .drop_duplicates()
    .assign(selected_positive_direction_flag=True)
    .assign(selected_positive_direction_2plus_flag=True)
)

selected_positive_event_ids_3way_df = (
    selected_positive_event_surface_3way_df[[EVENT_ID_COLUMN]]
    .drop_duplicates()
    .assign(reference_positive_direction_flag=True)
    .assign(selected_positive_direction_3way_flag=True)
)

full_event_universe_export_df = (
    retained_event_spine_df.merge(
        selected_working_event_ids_df,
        on=EVENT_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )
    .merge(
        reference_surface_event_ids_df,
        on=EVENT_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )
    .merge(
        selected_positive_event_ids_2plus_df,
        on=EVENT_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )
    .merge(
        selected_positive_event_ids_3way_df,
        on=EVENT_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )
    .merge(
        event_directionality_summary_df,
        on=EVENT_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )
    .merge(
        event_driver_evidence_df[
            [
                EVENT_ID_COLUMN,
                "event_metric_driver_list",
                "event_modality_driver_list",
                "event_supporting_frameworks",
                "event_distinct_metric_count",
                "event_distinct_modality_count",
                "event_supporting_framework_count",
            ]
        ],
        on=EVENT_ID_COLUMN,
        how="left",
        validate="one_to_one",
    )
)

for flag_column in [
    "selected_working_surface_flag",
    "strict_reference_surface_flag",
    "selected_positive_direction_flag",
    "selected_positive_direction_2plus_flag",
    "reference_positive_direction_flag",
    "selected_positive_direction_3way_flag",
    "has_congestion_oriented",
    "has_positive_demand_shock",
    "has_decongestion_oriented",
    "has_negative_demand_shock",
    "has_ambiguous_directional_row",
    "positive_direction_event_flag",
    "negative_direction_event_flag",
    "both_positive_family_event_flag",
    "mixed_direction_event_flag",
    "positive_only_event_flag",
    "negative_only_event_flag",
]:
    if flag_column in full_event_universe_export_df.columns:
        full_event_universe_export_df[flag_column] = (
            full_event_universe_export_df[flag_column].fillna(False).astype(bool)
        )

full_event_universe_export_df["selected_surface_id"] = selected_surface_id
full_event_universe_export_df["selected_surface_label"] = selected_surface_label
full_event_universe_export_df["reference_surface_id"] = reference_surface_id
full_event_universe_export_df["reference_surface_label"] = reference_surface_label

selected_positive_event_export_2plus_df = (
    full_event_universe_export_df.loc[
        full_event_universe_export_df["selected_positive_direction_2plus_flag"]
    ]
    .copy()
    .reset_index(drop=True)
)

selected_positive_event_export_3way_df = (
    full_event_universe_export_df.loc[
        full_event_universe_export_df["selected_positive_direction_3way_flag"]
    ]
    .copy()
    .reset_index(drop=True)
)

event_driver_summary_columns = [
    column
    for column in [
        EVENT_ID_COLUMN,
        ZONE_ID_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
        "framework_anomaly_count",
        "framework_support_pattern",
        "positive_direction_profile",
        "event_metric_driver_list",
        "event_modality_driver_list",
        "event_supporting_frameworks",
        "event_distinct_metric_count",
        "event_distinct_modality_count",
        "event_supporting_framework_count",
        "event_max_directional_strength",
        "event_median_directional_strength",
    ]
    if column in full_event_universe_export_df.columns
]

selected_positive_driver_evidence_2plus_df = (
    selected_positive_event_export_2plus_df[event_driver_summary_columns]
    .drop_duplicates(subset=[EVENT_ID_COLUMN])
    .reset_index(drop=True)
)
selected_positive_driver_evidence_3way_df = (
    selected_positive_event_export_3way_df[event_driver_summary_columns]
    .drop_duplicates(subset=[EVENT_ID_COLUMN])
    .reset_index(drop=True)
)

metric_evidence_source_columns = [
    column
    for column in [
        EVENT_ID_COLUMN,
        "feature_set",
        "framework_name",
        "framework_label",
        ZONE_ID_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
        METRIC_COLUMN,
        "directional_label",
        "directional_strength",
        RESIDUAL_ZSCORE_COLUMN,
        SUPPORT_STATUS_COLUMN,
        SCALE_SOURCE_COLUMN,
        SUPPORT_REVIEW_COLUMN,
        "anomaly_flag",
        "row_has_any_structural_absence_flag",
        "row_has_any_neutral_imputation_flag",
        "row_has_any_low_support_flag",
        "all_base_demand_zero_flag",
    ]
    if column in directional_row_level_df.columns
]

def build_positive_metric_evidence(selected_event_df):
    selected_event_ids = set(
        selected_event_df[EVENT_ID_COLUMN].dropna().astype(str)
    )

    metric_df = directional_row_level_df[
        metric_evidence_source_columns
    ].copy()

    metric_df = metric_df.loc[
        metric_df[EVENT_ID_COLUMN].astype(str).isin(selected_event_ids)
        & metric_df["directional_label"].isin(
            STRESS_ALIGNED_DIRECTIONAL_LABELS
        )
    ].copy()

    if "anomaly_flag" in metric_df.columns:
        metric_df = metric_df.loc[
            metric_df["anomaly_flag"].fillna(False).astype(bool)
        ].copy()

    metric_df["modality"] = metric_df[METRIC_COLUMN].map(
        classify_metric_modality
    )

    metric_df["metric_stress_family"] = (
        metric_df["directional_label"]
        .map(
            {
                "congestion_oriented": "Congestion",
                "positive_demand_shock": "Positive demand",
            }
        )
        .fillna("Unknown")
    )

    metric_df["residual_zscore_abs"] = pd.to_numeric(
        metric_df.get(RESIDUAL_ZSCORE_COLUMN),
        errors="coerce",
    ).abs()

    metric_df["directional_strength"] = pd.to_numeric(
        metric_df["directional_strength"],
        errors="coerce",
    )

    metric_df = metric_df.sort_values(
        [
            EVENT_ID_COLUMN,
            METRIC_COLUMN,
            "directional_strength",
            "residual_zscore_abs",
            "framework_name",
        ],
        ascending=[True, True, False, False, True],
    ).reset_index(drop=True)

    strongest_rows_df = (
        metric_df.groupby(
            [EVENT_ID_COLUMN, METRIC_COLUMN],
            dropna=False,
            as_index=False,
        )
        .head(1)
        .copy()
        .reset_index(drop=True)
    )

    aggregation_spec = {
        "supporting_frameworks": (
            "framework_name",
            lambda values: ", ".join(
                sorted(
                    pd.Series(values)
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),
        ),
        "supporting_framework_labels": (
            "framework_label",
            lambda values: ", ".join(
                sorted(
                    pd.Series(values)
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),
        ),
        "supporting_framework_count": (
            "framework_name",
            pd.Series.nunique,
        ),
        "supporting_framework_row_count": (
            "framework_name",
            "size",
        ),
    }

    if SUPPORT_STATUS_COLUMN in metric_df.columns:
        aggregation_spec["support_status_values"] = (
            SUPPORT_STATUS_COLUMN,
            lambda values: ", ".join(
                sorted(
                    pd.Series(values)
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),
        )

    if SCALE_SOURCE_COLUMN in metric_df.columns:
        aggregation_spec["scale_source_values"] = (
            SCALE_SOURCE_COLUMN,
            lambda values: ", ".join(
                sorted(
                    pd.Series(values)
                    .dropna()
                    .astype(str)
                    .unique()
                )
            ),
        )

    if SUPPORT_REVIEW_COLUMN in metric_df.columns:
        aggregation_spec["any_support_review_flag"] = (
            SUPPORT_REVIEW_COLUMN,
            "max",
        )

    support_summary_df = (
        metric_df.groupby(
            [EVENT_ID_COLUMN, METRIC_COLUMN],
            dropna=False,
        )
        .agg(**aggregation_spec)
        .reset_index()
    )

    framework_presence_df = (
        metric_df.assign(metric_support_flag=True)
        .pivot_table(
            index=[EVENT_ID_COLUMN, METRIC_COLUMN],
            columns="framework_name",
            values="metric_support_flag",
            aggfunc="max",
            fill_value=False,
        )
        .reset_index()
    )

    framework_presence_df.columns.name = None

    framework_presence_df = framework_presence_df.rename(
        columns={
            "dbscan": "dbscan_supporting_metric_flag",
            "isolation_forest": "if_supporting_metric_flag",
            "gmm": "gmm_supporting_metric_flag",
        }
    )

    for support_column in [
        "dbscan_supporting_metric_flag",
        "if_supporting_metric_flag",
        "gmm_supporting_metric_flag",
    ]:
        if support_column not in framework_presence_df.columns:
            framework_presence_df[support_column] = False

    output_df = (
        strongest_rows_df.merge(
            support_summary_df,
            on=[EVENT_ID_COLUMN, METRIC_COLUMN],
            how="left",
            validate="one_to_one",
        )
        .merge(
            framework_presence_df,
            on=[EVENT_ID_COLUMN, METRIC_COLUMN],
            how="left",
            validate="one_to_one",
        )
    )

    output_df[DATE_COLUMN] = pd.to_datetime(
        output_df[DATE_COLUMN],
        errors="coerce",
    )

    output_df = output_df.sort_values(
        [
            DATE_COLUMN,
            ZONE_ID_COLUMN,
            DAYPART_COLUMN,
            EVENT_ID_COLUMN,
            METRIC_COLUMN,
        ]
    ).reset_index(drop=True)

    assert not output_df.duplicated(
        [EVENT_ID_COLUMN, METRIC_COLUMN]
    ).any()

    assert output_df["directional_label"].isin(
        STRESS_ALIGNED_DIRECTIONAL_LABELS
    ).all()

    output_event_ids = set(
        output_df[EVENT_ID_COLUMN].dropna().astype(str)
    )

    assert output_event_ids == selected_event_ids, (
        f"Missing attribution for "
        f"{len(selected_event_ids - output_event_ids):,} events."
    )

    return output_df

selected_positive_metric_evidence_2plus_df = (
    build_positive_metric_evidence(
        selected_positive_event_export_2plus_df
    )
)

selected_positive_metric_evidence_3way_df = (
    build_positive_metric_evidence(
        selected_positive_event_export_3way_df
    )
)

def summarize_metric_evidence_qa(
    evidence_df,
    selected_event_df,
    surface_label,
    output_path,
):
    selected_event_ids = set(
        selected_event_df[EVENT_ID_COLUMN].dropna().astype(str)
    )
    evidence_event_ids = set(
        evidence_df[EVENT_ID_COLUMN].dropna().astype(str)
    )

    qa_row = {
        "surface": surface_label,
        "output_path": str(output_path),
        "row_count": int(len(evidence_df)),
        "distinct_event_count": int(
            evidence_df[EVENT_ID_COLUMN].nunique()
        ),
        "distinct_event_metric_count": int(
            evidence_df[[EVENT_ID_COLUMN, METRIC_COLUMN]]
            .drop_duplicates()
            .shape[0]
        ),
        "duplicate_count_at_event_metric": int(
            evidence_df.duplicated(
                subset=[EVENT_ID_COLUMN, METRIC_COLUMN]
            ).sum()
        ),
        "counter_direction_label_count": int(
            (~evidence_df["directional_label"].isin(
                STRESS_ALIGNED_DIRECTIONAL_LABELS
            )).sum()
        ),
        "events_missing_stress_metric_rows": int(
            len(selected_event_ids - evidence_event_ids)
        ),
        "extra_evidence_events": int(
            len(evidence_event_ids - selected_event_ids)
        ),
    }

    assert qa_row["duplicate_count_at_event_metric"] == 0
    assert qa_row["counter_direction_label_count"] == 0
    assert qa_row["events_missing_stress_metric_rows"] == 0
    assert qa_row["extra_evidence_events"] == 0

    return qa_row


selected_positive_metric_evidence_qa_df = pd.DataFrame(
    [
        summarize_metric_evidence_qa(
            selected_positive_metric_evidence_2plus_df,
            selected_positive_event_export_2plus_df,
            "2+ retained frameworks",
            POSITIVE_DIRECTION_2PLUS_METRIC_EVIDENCE_EXPORT_PATH,
        ),
        summarize_metric_evidence_qa(
            selected_positive_metric_evidence_3way_df,
            selected_positive_event_export_3way_df,
            "All 3 retained frameworks",
            POSITIVE_DIRECTION_3WAY_METRIC_EVIDENCE_EXPORT_PATH,
        ),
    ]
)

positive_surface_summary_export_2plus_df = (
    positive_surface_candidate_summary_df.loc[
        positive_surface_candidate_summary_df["candidate_surface_id"].eq(
            "framework_consensus_2plus"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

positive_surface_summary_export_3way_df = (
    positive_surface_candidate_summary_df.loc[
        positive_surface_candidate_summary_df["candidate_surface_id"].eq(
            "framework_consensus_3way"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

consensus_indicator_export_df = full_event_universe_export_df[
    [
        EVENT_ID_COLUMN,
        ZONE_ID_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
        "any_framework_event_flag",
        "framework_anomaly_count",
        "framework_support_pattern",
        "selected_working_surface_flag",
        "strict_reference_surface_flag",
        "selected_positive_direction_2plus_flag",
        "selected_positive_direction_3way_flag",
    ]
].copy()

framework_flag_export_df = full_event_universe_export_df[
    [
        EVENT_ID_COLUMN,
        ZONE_ID_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
        "dbscan_event_flag",
        "if_event_flag",
        "gmm_event_flag",
    ]
].copy()

directional_label_export_df = full_event_universe_export_df[
    [
        EVENT_ID_COLUMN,
        ZONE_ID_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
        "has_congestion_oriented",
        "has_positive_demand_shock",
        "has_decongestion_oriented",
        "has_negative_demand_shock",
        "positive_direction_event_flag",
        "negative_direction_event_flag",
        "both_positive_family_event_flag",
        "mixed_direction_event_flag",
        "positive_direction_profile",
        "event_max_directional_strength",
        "event_median_directional_strength",
    ]
].copy()

modality_participation_export_df = event_driver_evidence_df.copy()

social_join_focus_export_df = pd.concat(
    [
        selected_positive_event_export_2plus_df.assign(
            candidate_surface_id="framework_consensus_2plus",
            candidate_surface_label="2+ retained frameworks",
        ),
        selected_positive_event_export_3way_df.assign(
            candidate_surface_id="framework_consensus_3way",
            candidate_surface_label="All 3 retained frameworks",
        ),
    ],
    ignore_index=True,
)[
    [
        "candidate_surface_id",
        "candidate_surface_label",
        EVENT_ID_COLUMN,
        ZONE_ID_COLUMN,
        ZONE_COLUMN,
        BOROUGH_COLUMN,
        DATE_COLUMN,
        DAYPART_COLUMN,
        "framework_anomaly_count",
        "framework_support_pattern",
        "positive_direction_profile",
        "event_metric_driver_list",
        "event_modality_driver_list",
        "event_distinct_metric_count",
        "event_distinct_modality_count",
    ]
].copy()

handoff_manifest_df = pd.DataFrame(
    [
        {
            "asset": "full event universe with retained indicators",
            "path": str(FULL_UNIVERSE_EXPORT_PATH),
            "row_count": int(len(full_event_universe_export_df)),
            "status": "ready",
        },
        {
            "asset": "2+ positive-direction finalist",
            "path": str(POSITIVE_DIRECTION_2PLUS_WORKING_EXPORT_PATH),
            "row_count": int(len(selected_positive_event_export_2plus_df)),
            "status": "ready",
        },
        {
            "asset": "all-3 positive-direction finalist",
            "path": str(POSITIVE_DIRECTION_3WAY_WORKING_EXPORT_PATH),
            "row_count": int(len(selected_positive_event_export_3way_df)),
            "status": "ready",
        },
        {
            "asset": "2+ positive-direction summary",
            "path": str(POSITIVE_DIRECTION_2PLUS_SUMMARY_EXPORT_PATH),
            "row_count": int(len(positive_surface_summary_export_2plus_df)),
            "status": "ready",
        },
        {
            "asset": "all-3 positive-direction summary",
            "path": str(POSITIVE_DIRECTION_3WAY_SUMMARY_EXPORT_PATH),
            "row_count": int(len(positive_surface_summary_export_3way_df)),
            "status": "ready",
        },
        {
            "asset": "consensus indicators",
            "path": str(CONSENSUS_INDICATOR_EXPORT_PATH),
            "row_count": int(len(consensus_indicator_export_df)),
            "status": "ready",
        },
        {
            "asset": "directional event labels",
            "path": str(DIRECTIONAL_LABEL_EXPORT_PATH),
            "row_count": int(len(directional_label_export_df)),
            "status": "ready",
        },
        {
            "asset": "framework support flags",
            "path": str(FRAMEWORK_FLAG_EXPORT_PATH),
            "row_count": int(len(framework_flag_export_df)),
            "status": "ready",
        },
        {
            "asset": "event driver summaries",
            "path": str(MODALITY_PARTICIPATION_EXPORT_PATH),
            "row_count": int(len(modality_participation_export_df)),
            "status": "ready",
        },
        {
            "asset": "2+ positive-direction event driver summary",
            "path": str(POSITIVE_DIRECTION_2PLUS_DRIVER_EVIDENCE_EXPORT_PATH),
            "row_count": int(len(selected_positive_driver_evidence_2plus_df)),
            "status": "ready",
        },
        {
            "asset": "all-3 positive-direction event driver summary",
            "path": str(POSITIVE_DIRECTION_3WAY_DRIVER_EVIDENCE_EXPORT_PATH),
            "row_count": int(len(selected_positive_driver_evidence_3way_df)),
            "status": "ready",
        },
        {
            "asset": "2+ positive-direction metric evidence",
            "path": str(POSITIVE_DIRECTION_2PLUS_METRIC_EVIDENCE_EXPORT_PATH),
            "row_count": int(len(selected_positive_metric_evidence_2plus_df)),
            "status": "ready",
        },
        {
            "asset": "all-3 positive-direction metric evidence",
            "path": str(POSITIVE_DIRECTION_3WAY_METRIC_EVIDENCE_EXPORT_PATH),
            "row_count": int(len(selected_positive_metric_evidence_3way_df)),
            "status": "ready",
        },
        {
            "asset": "social join focus table",
            "path": str(SOCIAL_JOIN_FOCUS_EXPORT_PATH),
            "row_count": int(len(social_join_focus_export_df)),
            "status": "ready",
        },
        {
            "asset": "surface selection rationale",
            "path": str(RETAINED_SURFACE_SELECTION_PATH),
            "row_count": int(len(selected_surface_decision_df)),
            "status": "ready",
        },
    ]
)

if WRITE_335_OUTPUTS:
    full_event_universe_export_df.to_parquet(
        FULL_UNIVERSE_EXPORT_PATH,
        index=False,
    )
    selected_positive_event_export_2plus_df.to_parquet(
        POSITIVE_DIRECTION_2PLUS_WORKING_EXPORT_PATH,
        index=False,
    )
    selected_positive_event_export_3way_df.to_parquet(
        POSITIVE_DIRECTION_3WAY_WORKING_EXPORT_PATH,
        index=False,
    )
    positive_surface_summary_export_2plus_df.to_csv(
        POSITIVE_DIRECTION_2PLUS_SUMMARY_EXPORT_PATH,
        index=False,
    )
    positive_surface_summary_export_3way_df.to_csv(
        POSITIVE_DIRECTION_3WAY_SUMMARY_EXPORT_PATH,
        index=False,
    )
    consensus_indicator_export_df.to_parquet(
        CONSENSUS_INDICATOR_EXPORT_PATH,
        index=False,
    )
    directional_label_export_df.to_parquet(
        DIRECTIONAL_LABEL_EXPORT_PATH,
        index=False,
    )
    framework_flag_export_df.to_parquet(
        FRAMEWORK_FLAG_EXPORT_PATH,
        index=False,
    )
    modality_participation_export_df.to_parquet(
        MODALITY_PARTICIPATION_EXPORT_PATH,
        index=False,
    )
    selected_positive_driver_evidence_2plus_df.to_parquet(
        POSITIVE_DIRECTION_2PLUS_DRIVER_EVIDENCE_EXPORT_PATH,
        index=False,
    )
    selected_positive_driver_evidence_3way_df.to_parquet(
        POSITIVE_DIRECTION_3WAY_DRIVER_EVIDENCE_EXPORT_PATH,
        index=False,
    )
    selected_positive_metric_evidence_2plus_df.to_parquet(
        POSITIVE_DIRECTION_2PLUS_METRIC_EVIDENCE_EXPORT_PATH,
        index=False,
    )
    selected_positive_metric_evidence_3way_df.to_parquet(
        SELECTED_POSITIVE_METRIC_EVIDENCE_3WAY_PATH,
        index=False,
    )
    selected_positive_metric_evidence_3way_df.to_parquet(
        POSITIVE_DIRECTION_3WAY_METRIC_EVIDENCE_EXPORT_PATH,
        index=False,
    )
    social_join_focus_export_df.to_parquet(
        SOCIAL_JOIN_FOCUS_EXPORT_PATH,
        index=False,
    )
    handoff_manifest_df.to_csv(
        DOWNSTREAM_HANDOFF_MANIFEST_PATH,
        index=False,
    )

summary_display_columns = [
    column
    for column in [
        "candidate_surface_id",
        "candidate_surface_label",
        "positive_event_count",
        "positive_event_share_of_selected_surface",
        "positive_event_share_of_full_event_universe",
        "positive_event_share_of_candidate_surface",
        "positive_event_share_of_working_surface",
    ]
    if column in positive_surface_candidate_summary_df.columns
]
summary_format_map = {
    column: "{:.3f}"
    for column in [
        "positive_event_share_of_selected_surface",
        "positive_event_share_of_full_event_universe",
        "positive_event_share_of_candidate_surface",
        "positive_event_share_of_working_surface",
    ]
    if column in positive_surface_candidate_summary_df.columns
}

display(selected_positive_metric_evidence_qa_df)
display(handoff_manifest_df)
display(
    positive_surface_candidate_summary_df[summary_display_columns].style.format(
        summary_format_map
    )
)


,surface,output_path,row_count,distinct_event_count,distinct_event_metric_count,duplicate_count_at_event_metric,counter_direction_label_count,events_missing_stress_metric_rows,extra_evidence_events
0,2+ retained frameworks,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/retained_positive_direction_surface_2plus_metric_evidence.parquet,406238,299439,406238,0,0,0,0
1,All 3 retained frameworks,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/retained_positive_direction_surface_3way_metric_evidence.parquet,229842,155353,229842,0,0,0,0


,asset,path,row_count,status
0,full event universe with retained indicators,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/framework_comparison_full_row_universe.parquet,1559590,ready
1,2+ positive-direction finalist,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/retained_positive_direction_surface_2plus.parquet,299439,ready
2,all-3 positive-direction finalist,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/retained_positive_direction_surface_3way.parquet,155353,ready
3,2+ positive-direction summary,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/retained_positive_direction_surface_2plus_summary.csv,1,ready
4,all-3 positive-direction summary,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/retained_positive_direction_surface_3way_summary.csv,1,ready
5,consensus indicators,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/consensus_anomaly_indicators.parquet,1559590,ready
6,directional event labels,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/directional_anomaly_labels.parquet,1559590,ready
7,framework support flags,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/framework_specific_anomaly_flags.parquet,1559590,ready
8,event driver summaries,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/modality_participation_indicators.parquet,715321,ready
9,2+ positive-direction event driver summary,/datasets/_deepnote_work/pipeline_data/3.3.5.final_tables/retained_positive_direction_surface_2plus_driver_evidence.parquet,299439,ready


,candidate_surface_id,candidate_surface_label,positive_event_count,positive_event_share_of_full_event_universe,positive_event_share_of_candidate_surface
0,framework_consensus_3way,All 3 retained frameworks,155353,0.100,0.689
1,framework_consensus_2plus,2+ retained frameworks,299439,0.192,0.644


If present, ignore this superseded findings block\. Driver\-attribution audit work has moved to the current \`3\.3\.5\` export path, and any older reader\-facing claims about event\-level driver evidence should be treated as stale until rebuilt from the authoritative metric\-level positive\-direction evidence\.

### Section Summary

This section carries both retained stress\-direction finalists forward in parallel rather than collapsing the notebook to one winner too early\. It materializes the broader \`2\+ retained frameworks\` stress\-direction surface and the stricter \`All 3 retained frameworks\` stress\-direction surface, then compares their support structure, geography, and driver composition before exporting both in a downstream\-ready package\.

## Close

This notebook completes the synthesis stage of the anomaly pipeline\. It aligned the retained DBSCAN, Isolation Forest, and GMM surfaces on a shared row universe, added stress\-direction interpretation, collapsed those row\-level anomalies to the Taxi Zone × Date × Daypart event grain, and compared broad\-versus\-strict retained event surfaces built from overlap, support, geography, and driver evidence\.

That comparison narrowed the field to two stress\-direction finalists rather than one: the broader \`2\+ retained frameworks\` package and the stricter \`All 3 retained frameworks\` package\. The notebook then profiled both finalists, preserved them against the full event universe, and exported the full\-context tables, directional labels, framework\-support indicators, event\-level driver summaries, and metric\-level positive\-direction evidence needed for final downstream evaluation\.

With that work complete, the pipeline now moves forward with two explicitly retained stress\-direction event surfaces for final review: one broader and one stricter, both citywide, multi\-framework, and fully contextualized for the next notebook\.

### 3\.3\.5 Final Table Inventory

framework\_comparison\_full\_row\_universe\.parquet
Full Taxi Zone × Date × Daypart event universe with framework\-support flags, finalist\-surface indicators, stress\-direction labels, and event\-level driver evidence attached to every event\.

retained\_positive\_direction\_surface\_2plus\.parquet
Broader stress\-direction finalist surface built from the \`2\+ retained frameworks\` event package, preserved at Taxi Zone × Date × Daypart grain with full event context\.

retained\_positive\_direction\_surface\_3way\.parquet
Stricter stress\-direction finalist surface built from the \`All 3 retained frameworks\` event package, preserved at Taxi Zone × Date × Daypart grain with full event context\.

retained\_positive\_direction\_surface\_2plus\_summary\.csv
One\-row summary of the broader \`2\+ retained frameworks\` stress\-direction finalist, including event counts, event shares, and stress\-direction profile totals\.

retained\_positive\_direction\_surface\_3way\_summary\.csv
One\-row summary of the stricter \`All 3 retained frameworks\` stress\-direction finalist, including event counts, event shares, and stress\-direction profile totals\.

consensus\_anomaly\_indicators\.parquet
Event\-level consensus table recording whether each event was retained by any framework, how many frameworks supported it, and whether it belongs to the broader or stricter finalist surface\.

directional\_anomaly\_labels\.parquet
Event\-level directional classification output recording congestion\-oriented, demand\-shock, mixed\-direction, and stress\-direction indicators for every event in the full universe\.

framework\_specific\_anomaly\_flags\.parquet
Event\-level support table showing which of DBSCAN, Isolation Forest, and GMM flagged each event\.

modality\_participation\_indicators\.parquet
Event\-level driver\-summary table recording the consolidated metric and modality lists, distinct\-driver breadth, and supporting\-framework counts attached to each retained event\.

retained\_positive\_direction\_surface\_2plus\_driver\_evidence\.parquet
Event\-level driver\-summary table restricted to the broader \`2\+ retained frameworks\` positive\-direction finalist, preserving one summarized driver record per retained event for downstream profiling\.

retained\_positive\_direction\_surface\_3way\_driver\_evidence\.parquet
Event\-level driver\-summary table restricted to the stricter \`All 3 retained frameworks\` positive\-direction finalist, preserving one summarized driver record per retained event for downstream profiling\.

retained\_positive\_direction\_surface\_3way\_metric\_evidence\.parquet
Metric\-level stress\-aligned attribution table for the selected \`All 3 retained frameworks\` finalist, preserved at \`comparison\_event\_id × metric\` grain with one reconciled row per event\-metric and explicit framework\-support provenance for downstream applications\.

retained\_anomaly\_working\_table\.parquet
Broad retained event\-surface table for the \`2\+ retained frameworks\` finalist before positive\-direction filtering, preserved as the working event package from the overlap review stage\.

retained\_social\_join\_focus\_table\.parquet
Combined finalist review table for downstream joins and interpretive work, carrying both stress\-direction finalists with their support pattern, event keys, and driver summaries\.

retained\_surface\_selection\_rationale\.csv
One\-row decision record documenting the broader working finalist and the stricter reference finalist that advance together into the next notebook’s final evaluation stage\.

framework\_synthesis\_handoff\_manifest\.csv
Manifest of the exported 3\.3\.5 handoff package, listing artifact paths, row counts, and readiness status for the full\-universe table and both positive\-direction finalists\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>